# 05 — Feature Engineering

## Public Music Intelligence Platform (PMIP)

## Purpose

The purpose of this notebook is to transform the integrated PMIP dataset into a machine-learning-ready feature set.

Building on the findings from exploratory data analysis, this stage focuses on creating meaningful variables that better represent music performance, historical chart behaviour, geographic reach, artist audience strength and release characteristics.

The notebook will also address issues identified during EDA, including highly skewed numerical variables, missing values, highly correlated features, repeated artist-level information and potential data leakage.

The resulting feature set will provide the foundation for the machine-learning and music intelligence components of PMIP.

## Objectives

The objectives of this notebook are to:

- load and validate the integrated PMIP dataset;
- review the existing variables available for feature engineering;
- define the modelling role of identifiers, predictors and potential target variables;
- investigate repeated track and artist information;
- create meaningful release-based features;
- create historical chart performance features;
- create geographic performance and market-reach features;
- create artist audience and listener-based features;
- transform highly skewed numerical variables where appropriate;
- handle missing values while preserving their meaning;
- investigate and address highly correlated or redundant features;
- transform chart-ranking variables into more interpretable performance indicators;
- identify and prevent potential target leakage;
- define candidate feature sets for later PMIP machine-learning models;
- validate the engineered features; and
- save a machine-learning-ready dataset for subsequent model development.


## Table of Contents

### [1. Setup and Data Loading](#1-setup-and-data-loading)
- [1.1 Import Libraries](#11-import-libraries)
- [1.2 Load Integrated Dataset](#12-load-integrated-dataset)
- [1.3 Initial Dataset Validation](#13-initial-dataset-validation)

### [2. Feature Engineering Preparation](#2-feature-engineering-preparation)
- [2.1 Review Existing Features](#21-review-existing-features)
- [2.2 Identify Identifier, Predictor and Target Candidates](#22-identify-identifier-predictor-and-target-candidates)
- [2.3 Repeated Track and Artist Analysis](#23-repeated-track-and-artist-analysis)
- [2.4 Missing Feature Coverage](#24-missing-feature-coverage)

### [3. Release Feature Engineering](#3-release-feature-engineering)
- [3.1 Release Year and Month](#31-release-year-and-month)
- [3.2 Release Age](#32-release-age)
- [3.3 Release Feature Validation](#33-release-feature-validation)

### [4. Historical Performance Feature Engineering](#4-historical-performance-feature-engineering)
- [4.1 Historical Streaming Intensity](#41-historical-streaming-intensity)
- [4.2 Chart Longevity](#42-chart-longevity)
- [4.3 Chart Position Strength](#43-chart-position-strength)
- [4.4 Peak and Average Chart Performance](#44-peak-and-average-chart-performance)
- [4.5 Historical Performance Indicators](#45-historical-performance-indicators)

### [5. Geographic Performance Feature Engineering](#5-geographic-performance-feature-engineering)
- [5.1 Geographic Reach](#51-geographic-reach)
- [5.2 Market Coverage Indicators](#52-market-coverage-indicators)
- [5.3 Geographic Performance Intensity](#53-geographic-performance-intensity)

### [6. Artist Audience Feature Engineering](#6-artist-audience-feature-engineering)
- [6.1 Listener Strength](#61-listener-strength)
- [6.2 Peak Listener Relationships](#62-peak-listener-relationships)
- [6.3 Audience Trend Indicators](#63-audience-trend-indicators)
- [6.4 Artist Audience Feature Validation](#64-artist-audience-feature-validation)

### [7. Numerical Feature Transformation](#7-numerical-feature-transformation)
- [7.1 Review Skewed Features](#71-review-skewed-features)
- [7.2 Log Transformations](#72-log-transformations)
- [7.3 Transformation Validation](#73-transformation-validation)


### [8. Missing Value Treatment](#8-missing-value-treatment)
- [8.1 Missingness Review](#81-missingness-review)
- [8.2 Historical Feature Missingness](#82-historical-feature-missingness)
- [8.3 Artist Listener Feature Missingness](#83-artist-listener-feature-missingness)
- [8.4 Missingness Indicator Features](#84-missingness-indicator-features)

### [9. Redundancy and Multicollinearity Review](#9-redundancy-and-multicollinearity-review)
- [9.1 Highly Correlated Features](#91-highly-correlated-features)
- [9.2 Redundant Feature Identification](#92-redundant-feature-identification)
- [9.3 Feature Retention Decisions](#93-feature-retention-decisions)

### [10. Target Leakage and Feature Eligibility](#10-target-leakage-and-feature-eligibility)
- [10.1 Potential Target Variables](#101-potential-target-variables)
- [10.2 Leakage Risk Analysis](#102-leakage-risk-analysis)
- [10.3 Identifier and Non-Predictive Columns](#103-identifier-and-non-predictive-columns)
- [10.4 Final Eligible Predictor Features](#104-final-eligible-predictor-features)

### [11. Model-Specific Feature Sets](#11-model-specific-feature-sets)
- [11.1 Artist Performance Forecasting Features](#111-artist-performance-forecasting-features)
- [11.2 Artist Momentum Features](#112-artist-momentum-features)
- [11.3 Streaming Anomaly Detection Features](#113-streaming-anomaly-detection-features)
- [11.4 Geographic and Market Intelligence Features](#114-geographic-and-market-intelligence-features)
- [11.5 Release Performance Features](#115-release-performance-features)

### [12. Final Feature Dataset Validation](#12-final-feature-dataset-validation)
- [12.1 Final Dataset Shape](#121-final-dataset-shape)
- [12.2 Data Type Validation](#122-data-type-validation)
- [12.3 Missing Value Validation](#123-missing-value-validation)
- [12.4 Duplicate Validation](#124-duplicate-validation)
- [12.5 Engineered Feature Summary](#125-engineered-feature-summary)

### [13. Save Machine-Learning-Ready Dataset](#13-save-machine-learning-ready-dataset)
- [13.1 Save Feature Dataset](#131-save-feature-dataset)
- [13.2 Reload and Verify](#132-reload-and-verify)

### [14. Feature Engineering Summary](#14-feature-engineering-summary)
- [14.1 Key Engineered Features](#141-key-engineered-features)
- [14.2 Features Removed or Excluded](#142-features-removed-or-excluded)
- [14.3 Data Limitations](#143-data-limitations)
- [14.4 Recommendations for Model Development](#144-recommendations-for-model-development)

## 1. Setup and Data Loading

This section prepares the notebook for feature engineering by importing the required Python libraries, loading the integrated PMIP dataset and performing an initial validation of the data.

The purpose of this stage is to confirm that the dataset produced during the data integration process can be loaded successfully and is ready for feature engineering.

### 1.1 Import Libraries

The required Python libraries are imported to support data manipulation, numerical operations and later feature engineering tasks.

In [1]:
# Import libraries
from pathlib import Path

import numpy as np
import pandas as pd

# Configure pandas display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Libraries imported successfully.")

Libraries imported successfully.


### 1.2 Load Integrated Dataset

The integrated PMIP dataset produced during the data integration stage is loaded into a Pandas DataFrame.

This dataset combines Spotify performance information, artist listener statistics and historical chart performance features. It will act as the starting dataset for the feature engineering process.

In [2]:
# Define path to the integrated PMIP dataset
data_path = Path("../data/integrated/spotify_integrated.csv")

# Check that the dataset exists
if not data_path.exists():
    raise FileNotFoundError(
        f"Integrated dataset could not be found at: {data_path}"
    )

# Load the integrated dataset
df = pd.read_csv(data_path)

# Confirm successful loading
print("Integrated PMIP dataset loaded successfully.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

# Preview the dataset
df.head()

Integrated PMIP dataset loaded successfully.
Rows: 4,593
Columns: 42


,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Spotify Popularity,YouTube Views,YouTube Likes,TikTok Posts,TikTok Likes,TikTok Views,YouTube Playlist Reach,Apple Music Playlist Count,AirPlay Spins,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Shazam Counts,Explicit Track,track_normalised,artist_normalised,best_chart_position,average_chart_position,chart_observations,countries_charted,total_historical_streams,average_historical_streams,max_historical_streams,first_chart_date,last_chart_date,Listeners,Daily Trend,Peak,PkListeners
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,2024-04-26,QM24S2402528,1,725.40,"390,470,936.00","30,716.00","196,631,588.00",92.00,"84,274,754.00","1,713,126.00","5,767,700.00","651,565,900.00","5,332,281,936.00","150,597,040.00",210.00,"40,975.00",684.00,62.00,"17,598,718.00",114.00,"18,004,655.00","22,931.00","2,669,262.00",0,million dollar baby,tommy richman,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Not Like Us,Not Like Us,Kendrick Lamar,2024-05-04,USUG12400910,2,545.90,"323,703,884.00","28,113.00","174,597,137.00",92.00,"116,347,040.00","3,486,739.00","674,700.00","35,223,547.00","208,339,025.00","156,380,351.00",188.00,"40,778.00",3.00,67.00,"10,422,430.00",111.00,"7,780,028.00","28,444.00","1,118,279.00",1,not like us,kendrick lamar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"47,391,930.00","-8,503.00",33.00,"54,045,549.00"
2,i like the way you kiss me,I like the way you kiss me,Artemas,2024-03-19,QZJ842400387,3,538.40,"601,309,283.00","54,331.00","211,607,669.00",92.00,"122,599,116.00","2,228,730.00","3,025,400.00","275,154,237.00","3,369,120,610.00","373,784,955.00",190.00,"74,333.00",536.00,136.00,"36,321,847.00",172.00,"5,022,621.00","5,639.00","5,285,340.00",0,i like the way you kiss me,artemas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Flowers,Flowers - Single,Miley Cyrus,2023-01-12,USSM12209777,4,444.90,"2,031,280,633.00","269,802.00","136,569,078.00",85.00,"1,096,100,899.00","10,629,796.00","7,189,811.00","1,078,757,968.00","14,603,725,994.00","3,351,188,582.00",394.00,"1,474,799.00","2,182.00",264.00,"24,684,248.00",210.00,"190,260,277.00","203,384.00","11,822,942.00",0,flowers,miley cyrus,1.00,19.70,873.00,74.00,"1,658,244,203.00","1,899,477.90","115,156,896.00",2023-01-19,2023-04-06,"71,277,599.00","71,089.00",2.00,"84,140,935.00"
4,Houdini,Houdini,Eminem,2024-05-31,USUG12403398,5,423.30,"107,034,922.00","7,223.00","151,469,874.00",88.00,"77,373,957.00","3,670,188.00","16,400.00",NaN,NaN,"112,763,851.00",182.00,"12,185.00",1.00,82.00,"17,660,624.00",105.00,"4,493,884.00","7,006.00","457,017.00",1,houdini,eminem,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"64,022,485.00","-52,362.00",11.00,"68,591,390.00"


#### Interpretation

The integrated PMIP dataset was loaded successfully and contains **4,593 records and 42 features**. Each record represents a track together with available information from the Spotify 2024 dataset and the additional artist listener and historical chart features introduced during data integration.

The successful loading of all 42 columns confirms that the integrated dataset is available as the starting point for feature engineering. The preview also shows that the dataset contains a mixture of track identifiers, release information, platform performance metrics and other music-performance variables.

At this stage, the original integrated dataset has not been modified. Further validation is required before engineering new features to ensure that the dataset structure, data types, missing values and duplicate records remain suitable for the next stages of the PMIP machine-learning pipeline.

### 1.3 Initial Dataset Validation

Before creating new features, the integrated dataset is validated to confirm its structure and current data quality.

This validation checks the dataset dimensions, data types, missing values, duplicate records and release-date field. Establishing this baseline helps ensure that feature engineering is performed on a consistent dataset and prevents existing data-quality issues from being mistaken for problems introduced during this notebook.

In [3]:
# Display dataset dimensions
print("Dataset dimensions")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

# Check exact duplicate rows
duplicate_rows = df.duplicated().sum()

print("\nDuplicate records")
print(f"Exact duplicate rows: {duplicate_rows:,}")

# Display data types
print("\nData types")
display(df.dtypes.to_frame(name="Data Type"))

# Check missing values
missing_summary = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Missing Percentage": (df.isna().mean() * 100).round(2)
})

missing_summary = (
    missing_summary[missing_summary["Missing Values"] > 0]
    .sort_values("Missing Percentage", ascending=False)
)

print(f"\nColumns containing missing values: {len(missing_summary):,}")
display(missing_summary)

# Validate the release date without modifying the original column
release_dates = pd.to_datetime(df["Release Date"], errors="coerce")

print("\nRelease date validation")
print(f"Valid release dates: {release_dates.notna().sum():,}")
print(f"Invalid or missing release dates: {release_dates.isna().sum():,}")

if release_dates.notna().any():
    print(f"Earliest release date: {release_dates.min().date()}")
    print(f"Latest release date: {release_dates.max().date()}")

Dataset dimensions
Rows: 4,593
Columns: 42

Duplicate records
Exact duplicate rows: 0

Data types


,Data Type
Track,str
Album Name,str
Artist,str
Release Date,str
ISRC,str
All Time Rank,int64
Track Score,float64
Spotify Streams,float64
Spotify Playlist Count,float64
Spotify Playlist Reach,float64



Columns containing missing values: 32


,Missing Values,Missing Percentage
last_chart_date,2358,51.34
first_chart_date,2358,51.34
max_historical_streams,2358,51.34
average_historical_streams,2358,51.34
total_historical_streams,2358,51.34
countries_charted,2358,51.34
chart_observations,2358,51.34
average_chart_position,2358,51.34
best_chart_position,2358,51.34
SiriusXM Spins,2118,46.11



Release date validation
Valid release dates: 4,593
Invalid or missing release dates: 0
Earliest release date: 1987-07-21
Latest release date: 2024-06-14


#### Interpretation

The initial validation confirms that the integrated PMIP dataset contains **4,593 records and 42 variables**, with **no exact duplicate rows**. This indicates that the integration stage produced a structurally consistent dataset without introducing complete duplicate records.

Most performance-related variables have already been stored as numerical data types, making them suitable for later feature engineering and machine-learning preparation. Identifier and descriptive variables such as track, artist, album, ISRC and normalised names remain as text fields.

Missing data is present across **32 columns**, although the amount varies considerably between features. The historical chart variables have the largest common missingness, with **2,358 records (51.34%)** missing historical chart information. This is expected because only tracks successfully matched with the historical chart dataset contain these features. Artist listener variables such as `Listeners`, `Daily Trend`, `Peak` and `PkListeners` are missing for **1,204 records (26.21%)**, reflecting incomplete artist-level matching.

Several platform-specific variables also contain missing values, including SiriusXM, Pandora, TikTok, Amazon Music, YouTube, Deezer and Spotify metrics. These values should not automatically be removed or replaced because missingness may represent unavailable platform coverage rather than invalid observations. Appropriate treatment will therefore depend on how each feature is used during modelling.

The release-date validation produced **4,593 valid dates with no invalid or missing values**. Releases range from **21 July 1987 to 14 June 2024**, providing a valid foundation for constructing release year, month and release-age features.

Overall, the dataset is suitable for feature engineering. However, missing historical, listener and platform-specific information will need to be considered carefully when selecting predictors and preparing individual machine-learning datasets.

## 2. Feature Engineering Preparation

### 2.1 Review Existing Features

Before creating new features, the existing variables within the integrated PMIP dataset are reviewed and grouped according to their purpose.

This step helps distinguish between identifiers, release information, platform performance metrics, historical chart information and artist-level listener statistics. Understanding the existing feature groups reduces the risk of creating redundant variables and provides a clearer foundation for later machine-learning preparation.

In [4]:
# Group the existing PMIP variables by their general purpose

feature_groups = {
    "Identifiers and Descriptive Features": [
        "Track",
        "Album Name",
        "Artist",
        "ISRC",
        "track_normalised",
        "artist_normalised"
    ],

    "Release Features": [
        "Release Date"
    ],

    "Ranking and Track Performance Features": [
        "All Time Rank",
        "Track Score"
    ],

    "Spotify Features": [
        "Spotify Streams",
        "Spotify Playlist Count",
        "Spotify Playlist Reach",
        "Spotify Popularity"
    ],

    "Other Platform Features": [
        "YouTube Views",
        "YouTube Likes",
        "TikTok Posts",
        "TikTok Likes",
        "TikTok Views",
        "YouTube Playlist Reach",
        "Apple Music Playlist Count",
        "AirPlay Spins",
        "SiriusXM Spins",
        "Deezer Playlist Count",
        "Deezer Playlist Reach",
        "Amazon Playlist Count",
        "Pandora Streams",
        "Pandora Track Stations",
        "Shazam Counts"
    ],

    "Track Characteristics": [
        "Explicit Track"
    ],

    "Historical Chart Features": [
        "best_chart_position",
        "average_chart_position",
        "chart_observations",
        "countries_charted",
        "total_historical_streams",
        "average_historical_streams",
        "max_historical_streams",
        "first_chart_date",
        "last_chart_date"
    ],

    "Artist Listener Features": [
        "Listeners",
        "Daily Trend",
        "Peak",
        "PkListeners"
    ]
}

# Check that every listed feature actually exists in the dataset
grouped_features = [
    feature
    for features in feature_groups.values()
    for feature in features
]

missing_from_dataset = [
    feature
    for feature in grouped_features
    if feature not in df.columns
]

unclassified_features = [
    column
    for column in df.columns
    if column not in grouped_features
]

# Display feature groups
for group_name, features in feature_groups.items():
    available_features = [
        feature for feature in features
        if feature in df.columns
    ]

    print(f"\n{group_name} ({len(available_features)} features)")
    print("-" * len(group_name))

    for feature in available_features:
        print(f"• {feature}")

print("\nFeature review summary")
print(f"Dataset columns: {df.shape[1]}")
print(f"Features classified: {len(grouped_features)}")
print(f"Listed features missing from dataset: {len(missing_from_dataset)}")
print(f"Dataset features not yet classified: {len(unclassified_features)}")

if missing_from_dataset:
    print("\nListed features not found:")
    for feature in missing_from_dataset:
        print(f"• {feature}")

if unclassified_features:
    print("\nUnclassified dataset features:")
    for feature in unclassified_features:
        print(f"• {feature}")


Identifiers and Descriptive Features (6 features)
------------------------------------
• Track
• Album Name
• Artist
• ISRC
• track_normalised
• artist_normalised

Release Features (1 features)
----------------
• Release Date

Ranking and Track Performance Features (2 features)
--------------------------------------
• All Time Rank
• Track Score

Spotify Features (4 features)
----------------
• Spotify Streams
• Spotify Playlist Count
• Spotify Playlist Reach
• Spotify Popularity

Other Platform Features (15 features)
-----------------------
• YouTube Views
• YouTube Likes
• TikTok Posts
• TikTok Likes
• TikTok Views
• YouTube Playlist Reach
• Apple Music Playlist Count
• AirPlay Spins
• SiriusXM Spins
• Deezer Playlist Count
• Deezer Playlist Reach
• Amazon Playlist Count
• Pandora Streams
• Pandora Track Stations
• Shazam Counts

Track Characteristics (1 features)
---------------------
• Explicit Track

Historical Chart Features (9 features)
-------------------------
• best_chart_po

#### Interpretation

The feature review successfully accounts for all **42 variables** in the integrated PMIP dataset. Every dataset column has been assigned to a meaningful feature group, with no missing or unclassified variables.

The dataset contains several different types of information that can support later PMIP intelligence models. These include descriptive identifiers, release information, ranking and track-performance measures, Spotify metrics, cross-platform performance measures, historical chart statistics and artist-level listener information.

The review also shows that some variables serve primarily as identifiers rather than direct numerical predictors. For example, `Track`, `Artist`, `Album Name`, `ISRC`, `track_normalised` and `artist_normalised` are useful for identifying and grouping records but should not automatically be treated as ordinary numerical model inputs.

Other variables represent measurable performance signals and may provide useful predictors or target variables. These include Spotify streams and popularity, playlist exposure, artist listeners, historical streaming performance, chart positions, chart observations and geographic chart coverage.

This complete feature inventory provides the foundation for distinguishing identifiers from potential predictors and prediction targets in the next stage.

### 2.2 Identify Identifier, Predictor and Target Candidates

The existing PMIP variables are now separated according to their potential roles in later machine-learning tasks.

Identifier variables describe or uniquely identify tracks, artists and releases but are not automatically suitable as direct numerical model inputs. Predictor candidates represent measurable information that may help explain or predict music performance. Target candidates represent performance outcomes that PMIP may attempt to estimate, classify or analyse through its different intelligence models.

These classifications are preliminary. A variable may act as a predictor in one PMIP model and as a target in another. Final feature and target selection will therefore be performed separately for each machine-learning task to reduce data leakage and ensure that only appropriate information is available to each model.

In [5]:
# Define preliminary feature roles for later PMIP modelling

identifier_features = [
    "Track",
    "Album Name",
    "Artist",
    "ISRC",
    "track_normalised",
    "artist_normalised"
]

predictor_candidates = [
    "Release Date",
    "All Time Rank",
    "Track Score",
    "Spotify Playlist Count",
    "Spotify Playlist Reach",
    "Spotify Popularity",
    "YouTube Views",
    "YouTube Likes",
    "TikTok Posts",
    "TikTok Likes",
    "TikTok Views",
    "YouTube Playlist Reach",
    "Apple Music Playlist Count",
    "AirPlay Spins",
    "SiriusXM Spins",
    "Deezer Playlist Count",
    "Deezer Playlist Reach",
    "Amazon Playlist Count",
    "Pandora Streams",
    "Pandora Track Stations",
    "Shazam Counts",
    "Explicit Track",
    "best_chart_position",
    "average_chart_position",
    "chart_observations",
    "countries_charted",
    "total_historical_streams",
    "average_historical_streams",
    "max_historical_streams",
    "first_chart_date",
    "last_chart_date",
    "Listeners",
    "Daily Trend",
    "Peak",
    "PkListeners"
]

target_candidates = [
    "Spotify Streams",
    "Spotify Popularity",
    "Track Score",
    "Listeners",
    "Daily Trend",
    "best_chart_position",
    "average_chart_position",
    "chart_observations",
    "countries_charted",
    "total_historical_streams",
    "max_historical_streams"
]

# Validate that all selected variables exist
role_features = {
    "Identifier Features": identifier_features,
    "Predictor Candidates": predictor_candidates,
    "Target Candidates": target_candidates
}

for role, features in role_features.items():

    available = [
        feature for feature in features
        if feature in df.columns
    ]

    missing = [
        feature for feature in features
        if feature not in df.columns
    ]

    print(f"\n{role}")
    print("-" * len(role))
    print(f"Available: {len(available)}")

    for feature in available:
        print(f"• {feature}")

    if missing:
        print("\nNot found in dataset:")
        for feature in missing:
            print(f"• {feature}")

# Summary
print("\nFeature role summary")
print(f"Identifier features: {len(identifier_features)}")
print(f"Predictor candidates: {len(predictor_candidates)}")
print(f"Target candidates: {len(target_candidates)}")


Identifier Features
-------------------
Available: 6
• Track
• Album Name
• Artist
• ISRC
• track_normalised
• artist_normalised

Predictor Candidates
--------------------
Available: 35
• Release Date
• All Time Rank
• Track Score
• Spotify Playlist Count
• Spotify Playlist Reach
• Spotify Popularity
• YouTube Views
• YouTube Likes
• TikTok Posts
• TikTok Likes
• TikTok Views
• YouTube Playlist Reach
• Apple Music Playlist Count
• AirPlay Spins
• SiriusXM Spins
• Deezer Playlist Count
• Deezer Playlist Reach
• Amazon Playlist Count
• Pandora Streams
• Pandora Track Stations
• Shazam Counts
• Explicit Track
• best_chart_position
• average_chart_position
• chart_observations
• countries_charted
• total_historical_streams
• average_historical_streams
• max_historical_streams
• first_chart_date
• last_chart_date
• Listeners
• Daily Trend
• Peak
• PkListeners

Target Candidates
-----------------
Available: 11
• Spotify Streams
• Spotify Popularity
• Track Score
• Listeners
• Daily Trend
• 

#### Interpretation

The preliminary feature-role analysis identified 6 identifier features, 35 potential predictor variables and 11 potential target variables within the integrated PMIP dataset.

The identifier variables, including track name, artist name, album name, ISRC and the normalised track and artist fields, are primarily useful for identifying and linking records rather than being used directly as numerical predictors.

The 35 predictor candidates provide a broad collection of information describing release characteristics, rankings, streaming activity, playlist exposure, social-platform engagement, historical chart performance and artist listener statistics. These variables provide the initial feature pool from which more appropriate model-specific features can later be selected.

The 11 target candidates represent measurable performance outcomes that could support different PMIP intelligence tasks. For example, Spotify Streams could support performance prediction, Daily Trend could contribute to momentum analysis, and chart-related variables could support historical or geographic performance modelling.

These roles are not fixed. A variable may be used as a predictor in one model and as a target in another. Final predictor and target selection must therefore be performed separately for each PMIP model to avoid data leakage and ensure that each model answers a clearly defined prediction problem.

### 2.3 Repeated Track and Artist Analysis

The integrated PMIP dataset may contain multiple records associated with the same track or artist. These repetitions do not necessarily represent exact duplicate rows, as different records may contain different identifiers, release information or platform performance values.

This section investigates repeated tracks and artists before feature engineering. Understanding these repetitions is important because repeated observations may influence model training, feature aggregation and evaluation if they are not handled appropriately.

The analysis will examine:

- the number of unique tracks and artists;
- tracks appearing multiple times;
- repeated track-and-artist combinations;
- whether repeated records have different ISRC identifiers;
- the artists represented by the largest number of records; and
- the implications of repeated observations for later feature engineering and machine-learning development.

In [6]:
# Count total and unique tracks and artists
total_records = len(df)
unique_tracks = df["track_normalised"].nunique()
unique_artists = df["artist_normalised"].nunique()

print("Dataset record summary")
print("----------------------")
print(f"Total records: {total_records:,}")
print(f"Unique normalised tracks: {unique_tracks:,}")
print(f"Unique normalised artists: {unique_artists:,}")


# Count how frequently each track appears
track_counts = (
    df.groupby("track_normalised")
    .size()
    .sort_values(ascending=False)
)

repeated_tracks = track_counts[track_counts > 1]

print("\nRepeated track summary")
print("----------------------")
print(f"Tracks appearing more than once: {len(repeated_tracks):,}")
print(f"Maximum records for a single track: {track_counts.max():,}")


# Count repeated track-and-artist combinations
track_artist_counts = (
    df.groupby(["track_normalised", "artist_normalised"])
    .size()
    .sort_values(ascending=False)
)

repeated_track_artist = track_artist_counts[
    track_artist_counts > 1
]

print("\nRepeated track-and-artist summary")
print("---------------------------------")
print(
    f"Track-and-artist combinations appearing more than once: "
    f"{len(repeated_track_artist):,}"
)
print(
    f"Maximum records for one track-and-artist combination: "
    f"{track_artist_counts.max():,}"
)


# Display the most frequently repeated track-and-artist combinations
repeated_track_artist.head(15).to_frame("Record Count")

Dataset record summary
----------------------
Total records: 4,593
Unique normalised tracks: 4,314
Unique normalised artists: 1,997

Repeated track summary
----------------------
Tracks appearing more than once: 224
Maximum records for a single track: 13

Repeated track-and-artist summary
---------------------------------
Track-and-artist combinations appearing more than once: 98
Maximum records for one track-and-artist combination: 8


,,Record Count
track_normalised,artist_normalised,
danza kuduro - cover,music lab jpn,8
cake by the ocean - cover,music lab jpn,6
greedy,tate mcrae,3
espresso,sabrina carpenter,3
paint the town red,doja cat,3
ýýýýýý,yoasobi,3
flowers,miley cyrus,3
rich baby daddy (feat. sexyy red & sza),drake,2
rich flex,drake,2


#### Interpretation

The repeated-record analysis shows that the integrated dataset contains 4,593 records representing 4,314 unique normalised track names and 1,997 unique normalised artists.

A total of 224 track names appear more than once. However, track names alone are not sufficient for identifying repeated releases because different artists may release tracks with identical or similar titles.

Using the stricter combination of normalised track and artist names reduces the number of repeated groups to 98. The largest repeated track-and-artist combination contains eight records.

Examples include repeated records for tracks such as *Greedy*, *Espresso*, *Flowers* and *Paint the Town Red*. Some cover recordings also occur several times within the dataset.

These observations do not necessarily represent erroneous duplicates. Different records may correspond to different releases, versions, identifiers or platform measurements. Therefore, the repeated records should not be removed at this stage.

Further investigation of ISRC identifiers and other release-level attributes is required before deciding how repeated track-and-artist observations should be treated during feature engineering and machine-learning development.

In [7]:
# Extract records belonging to repeated track-and-artist combinations
repeated_pairs = repeated_track_artist.reset_index()[
    ["track_normalised", "artist_normalised"]
]

repeated_records = df.merge(
    repeated_pairs,
    on=["track_normalised", "artist_normalised"],
    how="inner"
)

# Count unique ISRCs within each repeated track-and-artist combination
repeated_isrc_summary = (
    repeated_records
    .groupby(["track_normalised", "artist_normalised"])
    .agg(
        record_count=("ISRC", "size"),
        unique_isrc_count=("ISRC", "nunique")
    )
    .sort_values(
        ["record_count", "unique_isrc_count"],
        ascending=False
    )
)

print("Repeated track-and-artist ISRC analysis")
print("---------------------------------------")
print(f"Repeated combinations analysed: {len(repeated_isrc_summary):,}")

same_isrc_groups = (repeated_isrc_summary["unique_isrc_count"] == 1).sum()
multiple_isrc_groups = (repeated_isrc_summary["unique_isrc_count"] > 1).sum()

print(f"Groups containing one unique ISRC: {same_isrc_groups:,}")
print(f"Groups containing multiple ISRCs: {multiple_isrc_groups:,}")

# Display the most frequently repeated combinations
repeated_isrc_summary.head(20)

Repeated track-and-artist ISRC analysis
---------------------------------------
Repeated combinations analysed: 98
Groups containing one unique ISRC: 0
Groups containing multiple ISRCs: 98


,,record_count,unique_isrc_count
track_normalised,artist_normalised,,
danza kuduro - cover,music lab jpn,8,8
cake by the ocean - cover,music lab jpn,6,6
espresso,sabrina carpenter,3,3
flowers,miley cyrus,3,3
greedy,tate mcrae,3,3
paint the town red,doja cat,3,3
ýýýýýý,yoasobi,3,3
2002,anne-marie,2,2
24k magic,bruno mars,2,2


#### ISRC Analysis Interpretation

The ISRC investigation provides further evidence that the repeated track-and-artist combinations should not be treated as simple duplicate records.

All 98 repeated track-and-artist groups contain multiple unique ISRC identifiers, while no repeated group contains only one unique ISRC. In several cases, the number of unique ISRCs is equal to the number of records within the group. For example, *Espresso* by Sabrina Carpenter appears three times with three different ISRCs, while *Danza Kuduro - Cover* by Music Lab JPN appears eight times with eight different ISRCs.

Since an ISRC identifies a specific sound recording, different ISRC values indicate that these observations represent distinct recording or release identities within the source data, despite sharing the same normalised track and artist names.

Therefore, these records will not be removed as duplicates during feature engineering. The ISRC should be retained as an important release-level identifier, while the normalised track and artist fields can continue to support grouping and artist-level feature creation.

This finding is particularly important for later machine-learning development because records should not be collapsed solely on the basis of matching track and artist names. Where model evaluation requires separation between related releases, grouping strategies may also be considered to reduce the risk of information leakage between training and testing data.

### 2.4 Missing Feature Coverage

Before engineering new variables, the availability of the existing features must be assessed. Machine-learning models depend on sufficiently complete input data, and variables with substantial missing values may require additional preprocessing or may only be suitable for particular subsets of the dataset.

This section measures the coverage of each feature and groups variables according to their level of data availability.

The analysis will:

- calculate the number and percentage of available values for each feature;
- calculate the percentage of missing values;
- identify features with high, moderate and low coverage;
- highlight variables that may require missing-value treatment; and
- support decisions about which variables should be used during later feature engineering and model development.

Missing values will not be automatically removed or filled during this analysis. Appropriate treatment will depend on the meaning of each feature and the requirements of the individual PMIP models.

In [8]:
# Calculate feature coverage
feature_coverage = pd.DataFrame({
    "Available Values": df.notna().sum(),
    "Missing Values": df.isna().sum()
})

# Calculate percentages
feature_coverage["Coverage (%)"] = (
    feature_coverage["Available Values"] / len(df) * 100
)

feature_coverage["Missing (%)"] = (
    feature_coverage["Missing Values"] / len(df) * 100
)

# Classify feature coverage
def classify_coverage(coverage):
    if coverage >= 90:
        return "High"
    elif coverage >= 70:
        return "Moderate"
    else:
        return "Low"

feature_coverage["Coverage Level"] = (
    feature_coverage["Coverage (%)"]
    .apply(classify_coverage)
)

# Sort from lowest to highest coverage
feature_coverage = feature_coverage.sort_values(
    "Coverage (%)",
    ascending=True
)

# Display summary
print("Feature coverage summary")
print("------------------------")
print(f"Total features analysed: {len(feature_coverage)}")
print(
    f"High coverage (>= 90%): "
    f"{(feature_coverage['Coverage Level'] == 'High').sum()}"
)
print(
    f"Moderate coverage (70% - <90%): "
    f"{(feature_coverage['Coverage Level'] == 'Moderate').sum()}"
)
print(
    f"Low coverage (<70%): "
    f"{(feature_coverage['Coverage Level'] == 'Low').sum()}"
)

# Display complete coverage table
feature_coverage.style.format({
    "Coverage (%)": "{:.2f}",
    "Missing (%)": "{:.2f}"
})

Feature coverage summary
------------------------
Total features analysed: 42
High coverage (>= 90%): 15
Moderate coverage (70% - <90%): 17
Low coverage (<70%): 10


,Available Values,Missing Values,Coverage (%),Missing (%),Coverage Level
average_chart_position,2235,2358,48.66,51.34,Low
last_chart_date,2235,2358,48.66,51.34,Low
first_chart_date,2235,2358,48.66,51.34,Low
max_historical_streams,2235,2358,48.66,51.34,Low
average_historical_streams,2235,2358,48.66,51.34,Low
total_historical_streams,2235,2358,48.66,51.34,Low
countries_charted,2235,2358,48.66,51.34,Low
chart_observations,2235,2358,48.66,51.34,Low
best_chart_position,2235,2358,48.66,51.34,Low
SiriusXM Spins,2475,2118,53.89,46.11,Low


#### Interpretation

The feature coverage analysis shows that the integrated PMIP dataset contains a mixture of highly complete, moderately complete and relatively sparse variables. Of the 42 available features, 15 have high coverage, 17 have moderate coverage and 10 have low coverage.

Several important variables have particularly strong availability. Core descriptive fields such as Track, Artist, Album Name, Release Date and ISRC contain complete records, while Track Score and All Time Rank also have 100% coverage. Spotify-based variables are similarly well represented, with Spotify Streams available for 97.65% of records and Spotify Playlist Count and Spotify Playlist Reach available for more than 98% of the dataset. These variables therefore provide a strong foundation for later feature engineering and machine-learning development.

The artist listener variables, including Listeners, Peak, Daily Trend and PkListeners, have 73.79% coverage. Although these variables contain missing observations, their availability remains sufficient for further investigation and they may provide useful artist-level indicators when developing PMIP's performance and momentum intelligence features.

The historical chart variables have considerably lower coverage. Features including chart_observations, countries_charted, best_chart_position, average_chart_position and historical streaming statistics are available for 2,235 records, representing 48.66% of the integrated dataset. This reflects the partial matching between the Spotify dataset and the historical chart dataset rather than necessarily indicating poor data quality. These variables remain potentially valuable because they describe historical performance, but their missing values will require careful treatment during model preparation.

The analysis therefore suggests that PMIP should not apply one universal missing-value strategy across all variables. Feature treatment should depend on the meaning and source of each variable. High-coverage features can generally be retained with limited preprocessing, while moderate- and low-coverage variables should be evaluated individually before imputation, transformation or model-specific selection.

# 3. Release Feature Engineering

The integrated PMIP dataset contains the original release date for each track. While the complete date is useful for reference, machine-learning models can benefit from additional numerical features derived from this information.

This section transforms the existing `Release Date` variable into features that describe when a track was released and how old the release is relative to the dataset reference period.

The engineered release features will support later analysis of whether release timing and release age are associated with streaming, chart and artist performance.

## 3.1 Release Year and Month

This section converts `Release Date` into a datetime variable and extracts the year and month in which each track was released.

The resulting `release_year` and `release_month` features provide numerical representations of release timing that can be used during later analysis and machine-learning development.

In [9]:
# Create a working copy before feature engineering
df_features = df.copy()

# Convert Release Date to datetime
df_features["Release Date"] = pd.to_datetime(
    df_features["Release Date"],
    errors="coerce"
)

# Extract release year and month
df_features["release_year"] = df_features["Release Date"].dt.year
df_features["release_month"] = df_features["Release Date"].dt.month

# Validate the engineered features
print("Release feature engineering")
print("---------------------------")
print(f"Total records: {len(df_features):,}")
print(f"Missing release years: {df_features['release_year'].isna().sum():,}")
print(f"Missing release months: {df_features['release_month'].isna().sum():,}")

print(
    f"Release year range: "
    f"{df_features['release_year'].min():.0f} - "
    f"{df_features['release_year'].max():.0f}"
)

print("\nRelease month distribution")
print("--------------------------")

release_month_distribution = (
    df_features["release_month"]
    .value_counts()
    .sort_index()
    .rename_axis("Release Month")
    .to_frame("Track Count")
)

display(release_month_distribution)

print("\nSample engineered release features")
print("-----------------------------------")

display(
    df_features[
        [
            "Track",
            "Artist",
            "Release Date",
            "release_year",
            "release_month"
        ]
    ].head(10)
)

Release feature engineering
---------------------------
Total records: 4,593
Missing release years: 0
Missing release months: 0
Release year range: 1987 - 2024

Release month distribution
--------------------------


,Track Count
Release Month,
1,503
2,371
3,430
4,420
5,508
6,393
7,321
8,315
9,327



Sample engineered release features
-----------------------------------


,Track,Artist,Release Date,release_year,release_month
0,MILLION DOLLAR BABY,Tommy Richman,2024-04-26,2024,4
1,Not Like Us,Kendrick Lamar,2024-05-04,2024,5
2,i like the way you kiss me,Artemas,2024-03-19,2024,3
3,Flowers,Miley Cyrus,2023-01-12,2023,1
4,Houdini,Eminem,2024-05-31,2024,5
5,Lovin On Me,Jack Harlow,2023-11-10,2023,11
6,Beautiful Things,Benson Boone,2024-01-18,2024,1
7,Gata Only,FloyyMenor,2024-02-02,2024,2
8,Danza Kuduro - Cover,MUSIC LAB JPN,2024-06-09,2024,6
9,BAND4BAND (feat. Lil Baby),Central Cee,2024-05-23,2024,5


#### Interpretation

The release feature engineering process successfully converted the original `Release Date` variable into two additional numerical features: `release_year` and `release_month`.

All 4,593 records produced valid release year and month values, with no missing values introduced during the transformation. This confirms that the original release dates are sufficiently complete and correctly formatted for feature engineering.

The releases represented in the dataset range from 1987 to 2024, showing that the dataset contains both relatively recent releases and older tracks that continue to have measurable music-platform performance.

The monthly distribution also shows that releases are represented across all twelve months. May contains the largest number of records with 508 tracks, followed closely by January with 503 tracks, while December contains the fewest with 271 tracks. Although the distribution varies between months, no month is absent from the dataset.

The engineered `release_year` and `release_month` variables provide more machine-learning-friendly representations of release timing than the original date alone. They can later help investigate whether release period is associated with streaming, chart, listener or overall track performance.

The original `Release Date` variable has also been retained, allowing additional time-based features to be derived without losing the original temporal information.

## 3.2 Release Age

The amount of time that a track has been available can influence its accumulated streaming and performance statistics. Older releases have generally had more time to accumulate streams, playlist exposure and other engagement measures than recently released tracks.

To represent this effect, release age is calculated relative to the latest release date observed within the PMIP dataset rather than the current calendar date. This avoids introducing additional time information that was not available when the source dataset was collected.

Two features are created:

- `release_age_days` — number of days between the track's release and the dataset reference date.
- `release_age_years` — the same release age expressed approximately in years.

These variables provide a numerical representation of track age that can later be investigated as a predictor of music performance.

In [10]:
# Define the dataset reference date
reference_date = feature_df["Release Date"].max()

# Calculate release age
feature_df["release_age_days"] = (
    reference_date - feature_df["Release Date"]
).dt.days

feature_df["release_age_years"] = (
    feature_df["release_age_days"] / 365.25
)

# Validate engineered release-age features
print("Release age feature engineering")
print("-------------------------------")
print(f"Reference date: {reference_date.date()}")
print(f"Total records: {len(feature_df):,}")

print(
    f"Missing release_age_days: "
    f"{feature_df['release_age_days'].isna().sum():,}"
)

print(
    f"Missing release_age_years: "
    f"{feature_df['release_age_years'].isna().sum():,}"
)

print(
    f"Release age range (days): "
    f"{feature_df['release_age_days'].min():,.0f} - "
    f"{feature_df['release_age_days'].max():,.0f}"
)

print(
    f"Release age range (years): "
    f"{feature_df['release_age_years'].min():.2f} - "
    f"{feature_df['release_age_years'].max():.2f}"
)

# Display descriptive statistics
print("\nRelease age statistics")
print("----------------------")

display(
    feature_df[
        ["release_age_days", "release_age_years"]
    ].describe().round(2)
)

# Display sample results
print("\nSample engineered release-age features")
print("---------------------------------------")

display(
    feature_df[
        [
            "Track",
            "Artist",
            "Release Date",
            "release_year",
            "release_month",
            "release_age_days",
            "release_age_years"
        ]
    ].head(10)
)

NameError: name 'feature_df' is not defined

#### Interpretation

Release age was successfully calculated for all 4,593 records using 14 June 2024, the latest release date represented in the dataset, as the reference point. No missing values were introduced during the transformation.

The engineered tracks range from newly released records with an age of 0 days to older releases approximately 36.90 years old. The median release age is approximately 2.04 years, meaning that half of the records are around two years old or younger relative to the dataset reference date. The mean release age is higher at approximately 3.38 years, indicating that a smaller number of substantially older tracks increase the overall average.

Release age is potentially important for later modelling because cumulative performance measures such as streaming counts and playlist exposure may naturally increase as a track remains available for longer. Including release age therefore provides the models with additional temporal context when comparing the performance of newer and older releases.

Both `release_age_days` and `release_age_years` represent the same underlying information at different scales. Their usefulness and potential redundancy will therefore be considered during later feature selection.

## 3.3 Release Feature Validation

The engineered release features are validated before proceeding to further feature engineering. This step checks their completeness, numerical ranges and internal consistency with the original release date.

The validation focuses on:

- confirming that no missing values were introduced;
- checking that release years and months remain within valid ranges;
- ensuring that release-age values are non-negative;
- confirming that the engineered year and month correspond to the original release date; and
- identifying any unexpected values produced during the transformation.

In [ ]:
# Define engineered release features
release_features = [
    "release_year",
    "release_month",
    "release_age_days",
    "release_age_years"
]

# Check missing values
release_missing = feature_df[release_features].isna().sum()

# Validate ranges
invalid_years = ~feature_df["release_year"].between(
    feature_df["Release Date"].dt.year.min(),
    feature_df["Release Date"].dt.year.max()
)

invalid_months = ~feature_df["release_month"].between(1, 12)

negative_age_days = feature_df["release_age_days"] < 0
negative_age_years = feature_df["release_age_years"] < 0

# Validate consistency with original Release Date
year_mismatches = (
    feature_df["release_year"] != feature_df["Release Date"].dt.year
)

month_mismatches = (
    feature_df["release_month"] != feature_df["Release Date"].dt.month
)

# Display validation results
print("Release feature validation")
print("--------------------------")
print(f"Records validated: {len(feature_df):,}")
print(f"Missing engineered values: {release_missing.sum():,}")
print(f"Invalid release years: {invalid_years.sum():,}")
print(f"Invalid release months: {invalid_months.sum():,}")
print(f"Negative release ages (days): {negative_age_days.sum():,}")
print(f"Negative release ages (years): {negative_age_years.sum():,}")
print(f"Release year mismatches: {year_mismatches.sum():,}")
print(f"Release month mismatches: {month_mismatches.sum():,}")

# Display summary of engineered release features
release_validation_summary = pd.DataFrame({
    "Data Type": feature_df[release_features].dtypes.astype(str),
    "Missing Values": feature_df[release_features].isna().sum(),
    "Minimum": feature_df[release_features].min(),
    "Maximum": feature_df[release_features].max()
})

print("\nEngineered release feature summary")
print("----------------------------------")

display(release_validation_summary)

Release feature validation
--------------------------
Records validated: 4,593
Missing engineered values: 0
Invalid release years: 0
Invalid release months: 0
Negative release ages (days): 0
Negative release ages (years): 0
Release year mismatches: 0
Release month mismatches: 0

Engineered release feature summary
----------------------------------


,Data Type,Missing Values,Minimum,Maximum
release_year,int32,0,"1,987.00","2,024.00"
release_month,int32,0,1.00,12.00
release_age_days,int64,0,0.00,"13,478.00"
release_age_years,float64,0,0.00,36.90


# 4. Historical Performance Feature Engineering

This section engineers features that represent the historical performance of tracks within the Spotify chart data. The existing historical variables are transformed into more interpretable indicators of streaming intensity, chart longevity and chart strength.

These features are intended to provide the later PMIP machine-learning models with information about how strongly and consistently a track has performed historically.

## 4.1 Historical Streaming Intensity

Historical streaming intensity measures the average amount of historical streaming activity associated with each chart observation.

This helps distinguish tracks that accumulated large historical stream totals because they appeared on the charts frequently from tracks that generated particularly strong streaming activity during each chart observation.

In [11]:
# Create historical streaming intensity feature

import numpy as np

# Avoid division by zero
valid_chart_observations = (
    df_features["chart_observations"].notna()
    & df_features["total_historical_streams"].notna()
    & (df_features["chart_observations"] > 0)
)

# Initialise the feature as missing
df_features["historical_streaming_intensity"] = np.nan

# Historical streams generated per chart observation
df_features.loc[
    valid_chart_observations,
    "historical_streaming_intensity"
] = (
    df_features.loc[
        valid_chart_observations,
        "total_historical_streams"
    ]
    /
    df_features.loc[
        valid_chart_observations,
        "chart_observations"
    ]
)

# Summary
print("Historical streaming intensity feature engineering")
print("-" * 48)

print(f"Total records: {len(df_features):,}")
print(
    f"Records with historical chart data: "
    f"{valid_chart_observations.sum():,}"
)
print(
    f"Records without historical chart data: "
    f"{(~valid_chart_observations).sum():,}"
)
print(
    f"Historical streaming intensity coverage: "
    f"{df_features['historical_streaming_intensity'].notna().mean() * 100:.2f}%"
)

# Descriptive statistics
print("\nHistorical streaming intensity statistics")
print("-" * 48)

display(
    df_features["historical_streaming_intensity"]
    .describe()
    .to_frame(name="Historical Streaming Intensity")
)

# Display sample records with historical data
print("\nSample engineered historical streaming intensity")
print("-" * 48)

display(
    df_features.loc[
        valid_chart_observations,
        [
            "Track",
            "Artist",
            "total_historical_streams",
            "chart_observations",
            "historical_streaming_intensity"
        ]
    ].head(10)
)

Historical streaming intensity feature engineering
------------------------------------------------
Total records: 4,593
Records with historical chart data: 2,235
Records without historical chart data: 2,358
Historical streaming intensity coverage: 48.66%

Historical streaming intensity statistics
------------------------------------------------


,Historical Streaming Intensity
count,"2,235.00"
mean,"585,288.93"
std,"578,920.57"
min,"1,560.00"
25%,"237,524.19"
50%,"440,868.67"
75%,"718,015.15"
max,"6,320,841.50"



Sample engineered historical streaming intensity
------------------------------------------------


,Track,Artist,total_historical_streams,chart_observations,historical_streaming_intensity
3,Flowers,Miley Cyrus,"1,658,244,203.00",873.00,"1,899,477.90"
15,LALA,Myke Towers,"428,761.00",1.00,"428,761.00"
19,As It Was,Harry Styles,"2,876,810,895.00","2,125.00","1,353,793.36"
26,STAY (with Justin Bieber),The Kid LAROI,"4,132,903,676.00","4,486.00","921,289.27"
27,Baby Shark,Pinkfong,"4,331,073.00",41.00,"105,635.93"
29,Numb / Encore,JAY-Z,"16,601,255.00",167.00,"99,408.71"
39,Dance Monkey,Tones And I,"4,787,862,859.00","7,013.00","682,712.51"
42,I'm Good (Blue),David Guetta,"637,152,716.00",638.00,"998,671.97"
48,If We Ever Broke Up,Mae Stephens,"83,619,615.00",128.00,"653,278.24"
49,Despacito,Luis Fonsi,"692,692.00",6.00,"115,448.67"


#### Interpretation

The `historical_streaming_intensity` feature was successfully created for the 2,235 records that contain valid historical chart information, giving the feature a coverage rate of 48.66%.

The feature measures the amount of historical streaming activity generated per chart observation by dividing total historical streams by the number of chart observations. This provides additional context beyond total historical streams alone because it distinguishes between tracks that accumulated large totals through long chart exposure and tracks that generated particularly strong streaming activity during each chart appearance.

The mean historical streaming intensity is approximately 585,289 streams per chart observation, while the median is lower at approximately 440,869. This difference suggests that the feature is positively skewed, with a smaller number of tracks achieving substantially higher streaming intensity than the majority.

The values range from approximately 1,560 to more than 6.32 million streams per chart observation, demonstrating substantial differences in historical performance strength across tracks.

Records without historical chart information remain missing rather than being assigned a value of zero. This preserves the distinction between unavailable historical data and genuine low historical performance.

## 4.2 Chart Longevity

Chart longevity measures the length of time between a track's first and last recorded historical chart appearance.

A longer chart lifespan may indicate sustained audience interest, stronger long-term performance or repeated chart activity over time. This feature therefore provides temporal information that is not captured by chart observations alone.

The engineered feature will be expressed in both days and years to support later analysis and model development.

In [12]:
# Convert historical chart dates to datetime
df_features["first_chart_date"] = pd.to_datetime(
    df_features["first_chart_date"],
    errors="coerce"
)

df_features["last_chart_date"] = pd.to_datetime(
    df_features["last_chart_date"],
    errors="coerce"
)

# Identify records with valid chart dates
valid_chart_dates = (
    df_features["first_chart_date"].notna()
    & df_features["last_chart_date"].notna()
)

# Calculate chart longevity
df_features["chart_longevity_days"] = np.nan
df_features["chart_longevity_years"] = np.nan

df_features.loc[
    valid_chart_dates,
    "chart_longevity_days"
] = (
    df_features.loc[
        valid_chart_dates,
        "last_chart_date"
    ]
    -
    df_features.loc[
        valid_chart_dates,
        "first_chart_date"
    ]
).dt.days

df_features.loc[
    valid_chart_dates,
    "chart_longevity_years"
] = (
    df_features.loc[
        valid_chart_dates,
        "chart_longevity_days"
    ] / 365.25
)

# Validate results
negative_longevity = (
    df_features["chart_longevity_days"] < 0
).sum()

print("Chart longevity feature engineering")
print("-----------------------------------")
print(f"Total records: {len(df_features):,}")
print(f"Records with valid chart dates: {valid_chart_dates.sum():,}")
print(f"Records without chart dates: {(~valid_chart_dates).sum():,}")
print(f"Negative chart longevity values: {negative_longevity:,}")

print(
    f"Chart longevity coverage: "
    f"{df_features['chart_longevity_days'].notna().mean() * 100:.2f}%"
)

# Descriptive statistics
print("\nChart longevity statistics")
print("--------------------------")

display(
    df_features[
        [
            "chart_longevity_days",
            "chart_longevity_years"
        ]
    ].describe().round(2)
)

# Display sample records
print("\nSample engineered chart longevity features")
print("------------------------------------------")

display(
    df_features.loc[
        valid_chart_dates,
        [
            "Track",
            "Artist",
            "first_chart_date",
            "last_chart_date",
            "chart_observations",
            "chart_longevity_days",
            "chart_longevity_years"
        ]
    ].head(10)
)

Chart longevity feature engineering
-----------------------------------
Total records: 4,593
Records with valid chart dates: 2,235
Records without chart dates: 2,358
Negative chart longevity values: 0
Chart longevity coverage: 48.66%

Chart longevity statistics
--------------------------


,chart_longevity_days,chart_longevity_years
count,"2,235.00","2,235.00"
mean,718.47,1.97
std,838.80,2.30
min,0.00,0.00
25%,133.00,0.36
50%,413.00,1.13
75%,945.00,2.59
max,"3,630.00",9.94



Sample engineered chart longevity features
------------------------------------------


,Track,Artist,first_chart_date,last_chart_date,chart_observations,chart_longevity_days,chart_longevity_years
3,Flowers,Miley Cyrus,2023-01-19,2023-04-06,873.00,77.00,0.21
15,LALA,Myke Towers,2023-03-30,2023-03-30,1.00,0.00,0.00
19,As It Was,Harry Styles,2022-04-07,2022-11-10,"2,125.00",217.00,0.59
26,STAY (with Justin Bieber),The Kid LAROI,2021-07-15,2022-11-10,"4,486.00",483.00,1.32
27,Baby Shark,Pinkfong,2017-09-28,2019-02-21,41.00,511.00,1.40
29,Numb / Encore,JAY-Z,2014-10-26,2018-02-08,167.00,"1,201.00",3.29
39,Dance Monkey,Tones And I,2019-05-16,2022-11-03,"7,013.00","1,267.00",3.47
42,I'm Good (Blue),David Guetta,2022-09-01,2022-11-10,638.00,70.00,0.19
48,If We Ever Broke Up,Mae Stephens,2023-02-16,2023-04-06,128.00,49.00,0.13
49,Despacito,Luis Fonsi,2019-02-07,2022-10-27,6.00,"1,358.00",3.72


#### Interpretation

The `chart_longevity_days` and `chart_longevity_years` features were successfully created for 2,235 records with valid historical chart dates, representing 48.66% of the integrated dataset. No negative longevity values were identified, indicating that the first and last chart dates are temporally consistent.

The median chart longevity is 413 days, or approximately 1.13 years, meaning that half of the tracks with historical chart data have chart observations spanning just over one year or less. The mean longevity is higher at approximately 718 days (1.97 years), suggesting that the distribution is positively skewed by tracks with much longer historical chart lifespans.

The middle 50% of tracks have chart longevity values ranging from approximately 133 days to 945 days. The maximum observed longevity is 3,630 days, equivalent to approximately 9.94 years, showing that some tracks maintained or regained chart presence across a substantially longer period.

A chart longevity value of zero represents tracks whose first and last recorded chart observations occurred on the same date. This does not necessarily indicate poor performance, but rather that the available historical dataset only recorded the track on a single date.

Chart longevity should not be interpreted as continuous chart presence. It measures the time span between the first and last recorded observations, while `chart_observations` measures the frequency of chart appearances within that period. Using both features together may therefore provide a stronger representation of sustained historical performance.

### 4.3 Chart Position Strength

Historical chart position is an important indicator of track performance. However, chart rankings are inverse measures, where lower numerical positions represent stronger performance.

For example, chart position 1 represents stronger performance than chart position 100. To make these variables easier to interpret alongside other performance features, transformed chart-position strength features are created.

The original `best_chart_position` and `average_chart_position` variables are preserved. Two additional features are engineered so that higher values represent stronger historical chart performance:

- `best_chart_strength`
- `average_chart_strength`

This transformation provides a more intuitive representation for later statistical analysis and machine-learning development.

In [13]:
# Identify valid historical chart positions
valid_best_position = (
    df_features["best_chart_position"].notna()
    & (df_features["best_chart_position"] > 0)
)

valid_average_position = (
    df_features["average_chart_position"].notna()
    & (df_features["average_chart_position"] > 0)
)

# Initialise engineered features
df_features["best_chart_strength"] = np.nan
df_features["average_chart_strength"] = np.nan

# Convert inverse chart rankings into strength measures
df_features.loc[
    valid_best_position,
    "best_chart_strength"
] = (
    1 / df_features.loc[
        valid_best_position,
        "best_chart_position"
    ]
)

df_features.loc[
    valid_average_position,
    "average_chart_strength"
] = (
    1 / df_features.loc[
        valid_average_position,
        "average_chart_position"
    ]
)

# Validation
print("Chart position strength feature engineering")
print("-------------------------------------------")

print(f"Total records: {len(df_features):,}")
print(
    f"Valid best chart positions: "
    f"{valid_best_position.sum():,}"
)
print(
    f"Valid average chart positions: "
    f"{valid_average_position.sum():,}"
)

print(
    f"Best chart strength coverage: "
    f"{df_features['best_chart_strength'].notna().mean() * 100:.2f}%"
)

print(
    f"Average chart strength coverage: "
    f"{df_features['average_chart_strength'].notna().mean() * 100:.2f}%"
)

print("\nChart strength statistics")
print("-------------------------")

display(
    df_features[
        [
            "best_chart_strength",
            "average_chart_strength"
        ]
    ].describe().round(4)
)

print("\nSample chart position strength features")
print("---------------------------------------")

display(
    df_features.loc[
        valid_best_position & valid_average_position,
        [
            "Track",
            "Artist",
            "best_chart_position",
            "best_chart_strength",
            "average_chart_position",
            "average_chart_strength"
        ]
    ].head(15)
)

Chart position strength feature engineering
-------------------------------------------
Total records: 4,593
Valid best chart positions: 2,235
Valid average chart positions: 2,235
Best chart strength coverage: 48.66%
Average chart strength coverage: 48.66%

Chart strength statistics
-------------------------


,best_chart_strength,average_chart_strength
count,"2,235.00","2,235.00"
mean,0.38,0.01
std,0.40,0.01
min,0.00,0.00
25%,0.04,0.01
50%,0.17,0.01
75%,1.00,0.01
max,1.00,0.16



Sample chart position strength features
---------------------------------------


,Track,Artist,best_chart_position,best_chart_strength,average_chart_position,average_chart_strength
3,Flowers,Miley Cyrus,1.00,1.00,19.70,0.05
15,LALA,Myke Towers,168.00,0.01,168.00,0.01
19,As It Was,Harry Styles,1.00,1.00,21.59,0.05
26,STAY (with Justin Bieber),The Kid LAROI,1.00,1.00,45.06,0.02
27,Baby Shark,Pinkfong,88.00,0.01,162.90,0.01
29,Numb / Encore,JAY-Z,53.00,0.02,155.74,0.01
39,Dance Monkey,Tones And I,1.00,1.00,66.36,0.02
42,I'm Good (Blue),David Guetta,1.00,1.00,48.37,0.02
48,If We Ever Broke Up,Mae Stephens,15.00,0.07,106.33,0.01
49,Despacito,Luis Fonsi,154.00,0.01,176.33,0.01


#### Interpretation

The chart-position strength features were successfully created for 2,235 records, representing 48.66% of the integrated dataset. This coverage is consistent with the historical chart data available from the earlier integration stage, while the remaining records do not have historical chart information.

The transformation successfully reverses the original interpretation of chart position so that higher values now represent stronger chart performance. For example, tracks that achieved a best chart position of 1 receive a `best_chart_strength` value of 1.00, while tracks with weaker peak positions receive substantially lower strength values.

The results also show an important difference between peak and sustained chart performance. Several tracks achieved a best chart position of 1 but have much lower `average_chart_strength` values. This indicates that reaching a very high chart position does not necessarily mean that a track consistently maintained that level of performance throughout its chart history.

The engineered `best_chart_strength` therefore represents peak historical chart strength, while `average_chart_strength` provides an indication of more sustained historical chart performance. Both features may provide useful predictive information for later PMIP machine-learning models.

The original chart-position variables are retained alongside these transformed features so that their usefulness and potential redundancy can be evaluated during model development.

### 4.4 Peak and Average Chart Performance

This section engineers additional historical performance features by combining
chart-position strength with the amount of historical chart activity recorded
for each track.

Two features are created:

- **Peak Chart Performance** — combines the track's strongest historical chart
  position with its chart observation frequency.
- **Average Chart Performance** — combines average chart-position strength with
  chart observation frequency.

A logarithmic transformation of chart observations is used so that tracks with
extremely large numbers of observations do not dominate the engineered
features. These variables provide measures of historical performance that
consider both chart strength and the amount of evidence available for that
performance.

In [14]:
# Create chart activity factor using a logarithmic transformation
df_features["chart_activity_factor"] = np.log1p(
    df_features["chart_observations"]
)

# Combine chart strength with chart activity
df_features["peak_chart_performance"] = (
    df_features["best_chart_strength"]
    * df_features["chart_activity_factor"]
)

df_features["average_chart_performance"] = (
    df_features["average_chart_strength"]
    * df_features["chart_activity_factor"]
)

# Count valid engineered values
valid_peak = df_features["peak_chart_performance"].notna().sum()
valid_average = df_features["average_chart_performance"].notna().sum()

print("Peak and average chart performance feature engineering")
print("------------------------------------------------------")
print(f"Total records: {len(df_features):,}")
print(f"Valid peak performance values: {valid_peak:,}")
print(f"Valid average performance values: {valid_average:,}")

print(
    f"Peak performance coverage: "
    f"{valid_peak / len(df_features) * 100:.2f}%"
)

print(
    f"Average performance coverage: "
    f"{valid_average / len(df_features) * 100:.2f}%"
)

print("\nEngineered performance statistics")
print("---------------------------------")

display(
    df_features[
        [
            "chart_activity_factor",
            "peak_chart_performance",
            "average_chart_performance"
        ]
    ].describe().round(4)
)

print("\nSample engineered chart performance features")
print("--------------------------------------------")

display(
    df_features.loc[
        df_features["chart_observations"].notna(),
        [
            "Track",
            "Artist",
            "chart_observations",
            "best_chart_position",
            "best_chart_strength",
            "average_chart_position",
            "average_chart_strength",
            "chart_activity_factor",
            "peak_chart_performance",
            "average_chart_performance"
        ]
    ].head(15).round(4)
)

Peak and average chart performance feature engineering
------------------------------------------------------
Total records: 4,593
Valid peak performance values: 2,235
Valid average performance values: 2,235
Peak performance coverage: 48.66%
Average performance coverage: 48.66%

Engineered performance statistics
---------------------------------


,chart_activity_factor,peak_chart_performance,average_chart_performance
count,"2,235.00","2,235.00","2,235.00"
mean,5.64,2.60,0.07
std,1.98,2.96,0.04
min,0.69,0.00,0.00
25%,4.26,0.18,0.04
50%,6.02,1.05,0.06
75%,7.19,4.55,0.09
max,9.54,9.54,0.51



Sample engineered chart performance features
--------------------------------------------


,Track,Artist,chart_observations,best_chart_position,best_chart_strength,average_chart_position,average_chart_strength,chart_activity_factor,peak_chart_performance,average_chart_performance
3,Flowers,Miley Cyrus,873.00,1.00,1.00,19.70,0.05,6.77,6.77,0.34
15,LALA,Myke Towers,1.00,168.00,0.01,168.00,0.01,0.69,0.00,0.00
19,As It Was,Harry Styles,"2,125.00",1.00,1.00,21.59,0.05,7.66,7.66,0.35
26,STAY (with Justin Bieber),The Kid LAROI,"4,486.00",1.00,1.00,45.06,0.02,8.41,8.41,0.19
27,Baby Shark,Pinkfong,41.00,88.00,0.01,162.90,0.01,3.74,0.04,0.02
29,Numb / Encore,JAY-Z,167.00,53.00,0.02,155.74,0.01,5.12,0.10,0.03
39,Dance Monkey,Tones And I,"7,013.00",1.00,1.00,66.36,0.02,8.86,8.86,0.13
42,I'm Good (Blue),David Guetta,638.00,1.00,1.00,48.37,0.02,6.46,6.46,0.13
48,If We Ever Broke Up,Mae Stephens,128.00,15.00,0.07,106.33,0.01,4.86,0.32,0.05
49,Despacito,Luis Fonsi,6.00,154.00,0.01,176.33,0.01,1.95,0.01,0.01


#### Interpretation

The peak and average chart performance features were successfully generated
for 2,235 records, representing 48.66% of the integrated dataset. This coverage
is consistent with the availability of historical chart information identified
during the earlier feature-engineering stages.

The logarithmic chart activity factor ranges from 0.69 to 9.54. This confirms
that the transformation reduces the effect of extremely large chart observation
counts while still allowing tracks with greater historical chart activity to
receive higher values.

The `peak_chart_performance` feature shows substantial variation, with a median
of 1.05 and a maximum of 9.54. Tracks that achieved strong peak chart positions
and accumulated many chart observations receive higher values. For example,
tracks that reached number one can still receive different peak-performance
values depending on the amount of historical chart activity recorded for them.

The `average_chart_performance` feature is considerably smaller, with a median
of 0.06 and a maximum of 0.51. This reflects the fact that maintaining a strong
average chart position across many observations is more difficult than
temporarily achieving a strong peak position.

Together, these features distinguish between peak chart success and more
sustained historical performance. They may therefore provide more informative
predictive signals than using chart position or chart observation frequency
independently.

### 4.5 Historical Performance Indicators

This section creates interpretable historical performance indicators from the
engineered chart features.

The indicators summarise whether a track demonstrates evidence of historical
chart success, strong peak performance, broad international reach and sustained
chart activity.

These features are designed to provide simpler representations of historical
performance that may later support machine-learning models and PMIP analytics.
The original continuous features are retained so that the usefulness of the
indicator variables can be evaluated during model development.

In [15]:
# Historical chart availability indicator
df_features["has_historical_chart_data"] = (
    df_features["chart_observations"].notna()
).astype(int)

# Top-10 historical chart achievement
df_features["reached_top_10"] = (
    df_features["best_chart_position"].le(10)
    & df_features["best_chart_position"].notna()
).astype(int)

# Number-one historical chart achievement
df_features["reached_number_one"] = (
    df_features["best_chart_position"].eq(1)
).astype(int)

# Broad geographic chart reach
# Five or more countries is used as an interpretable threshold
df_features["broad_geographic_reach"] = (
    df_features["countries_charted"].ge(5)
).astype(int)

# Sustained chart activity
# Median is calculated only from tracks with historical chart data
chart_observation_median = (
    df_features.loc[
        df_features["chart_observations"].notna(),
        "chart_observations"
    ].median()
)

df_features["sustained_chart_activity"] = (
    df_features["chart_observations"].ge(chart_observation_median)
).astype(int)

indicator_columns = [
    "has_historical_chart_data",
    "reached_top_10",
    "reached_number_one",
    "broad_geographic_reach",
    "sustained_chart_activity"
]

print("Historical performance indicator engineering")
print("--------------------------------------------")
print(f"Total records: {len(df_features):,}")
print(
    f"Median historical chart observations: "
    f"{chart_observation_median:,.0f}"
)

print("\nIndicator distribution")
print("----------------------")

indicator_summary = pd.DataFrame({
    "Records": df_features[indicator_columns].sum(),
    "Percentage (%)": (
        df_features[indicator_columns].mean() * 100
    ).round(2)
})

display(indicator_summary)

print("\nSample historical performance indicators")
print("----------------------------------------")

display(
    df_features[
        [
            "Track",
            "Artist",
            "best_chart_position",
            "chart_observations",
            "countries_charted",
            *indicator_columns
        ]
    ].head(20)
)

Historical performance indicator engineering
--------------------------------------------
Total records: 4,593
Median historical chart observations: 409

Indicator distribution
----------------------


,Records,Percentage (%)
has_historical_chart_data,2235,48.66
reached_top_10,1369,29.81
reached_number_one,609,13.26
broad_geographic_reach,1819,39.60
sustained_chart_activity,1118,24.34



Sample historical performance indicators
----------------------------------------


,Track,Artist,best_chart_position,chart_observations,countries_charted,has_historical_chart_data,reached_top_10,reached_number_one,broad_geographic_reach,sustained_chart_activity
0,MILLION DOLLAR BABY,Tommy Richman,NaN,NaN,NaN,0,0,0,0,0
1,Not Like Us,Kendrick Lamar,NaN,NaN,NaN,0,0,0,0,0
2,i like the way you kiss me,Artemas,NaN,NaN,NaN,0,0,0,0,0
3,Flowers,Miley Cyrus,1.00,873.00,74.00,1,1,1,1,1
4,Houdini,Eminem,NaN,NaN,NaN,0,0,0,0,0
5,Lovin On Me,Jack Harlow,NaN,NaN,NaN,0,0,0,0,0
6,Beautiful Things,Benson Boone,NaN,NaN,NaN,0,0,0,0,0
7,Gata Only,FloyyMenor,NaN,NaN,NaN,0,0,0,0,0
8,Danza Kuduro - Cover,MUSIC LAB JPN,NaN,NaN,NaN,0,0,0,0,0
9,BAND4BAND (feat. Lil Baby),Central Cee,NaN,NaN,NaN,0,0,0,0,0


#### Interpretation

Historical performance indicators were successfully generated for the integrated
dataset. Historical chart information is available for 2,235 records, representing
48.66% of the 4,593 tracks.

Among all records, 1,369 tracks (29.81%) achieved a historical Top 10 chart
position, while 609 tracks (13.26%) reached number one. This provides two
different indicators of historical chart success rather than relying only on
the original numerical chart-position values.

Broad geographic reach was identified for 1,819 records (39.60%), indicating
that a substantial proportion of the dataset contains tracks that appeared in
at least five geographic markets.

The median number of historical chart observations among tracks with available
chart data was 409. Using this threshold, 1,118 records (24.34% of the complete
dataset) were classified as having sustained chart activity.

The `has_historical_chart_data` indicator is particularly important because
tracks without historical observations receive zero values for the derived
achievement indicators. Retaining a separate availability indicator allows
later models to distinguish between the absence of historical evidence and
observed historical performance.

These binary indicators complement the continuous historical features created
earlier in this section and provide simpler, interpretable representations of
chart success, geographic reach and sustained activity for later modelling.

## 5. Geographic Performance Feature Engineering

This section investigates and engineers features describing the geographic
performance and international reach of tracks.

Geographic performance is an important component of PMIP because the platform
aims to identify differences in music performance across markets and provide
information about the international reach of artists and releases.

Before engineering geographic features, the available geographic information
in the integrated dataset is reviewed to determine which market-level
characteristics can be represented reliably.

### 5.1 Geographic Feature Availability

The integrated dataset contains aggregated historical chart information,
including the number of countries in which each matched track appeared.

This subsection examines the geographic variables currently available before
additional geographic performance features are created.

In [16]:
# Identify potentially geographic columns
geographic_keywords = [
    "country",
    "countries",
    "market",
    "region",
    "geographic"
]

geographic_columns = [
    column for column in df_features.columns
    if any(
        keyword in column.lower()
        for keyword in geographic_keywords
    )
]

print("Geographic feature availability")
print("-------------------------------")
print(f"Total dataset columns: {len(df_features.columns):,}")
print(f"Geographic-related columns found: {len(geographic_columns):,}")

for column in geographic_columns:
    print(f"• {column}")

print("\nGeographic feature coverage")
print("---------------------------")

if geographic_columns:
    geographic_coverage = pd.DataFrame({
        "Data Type": df_features[geographic_columns].dtypes.astype(str),
        "Available Values": df_features[geographic_columns].notna().sum(),
        "Missing Values": df_features[geographic_columns].isna().sum(),
        "Coverage (%)": (
            df_features[geographic_columns].notna().mean() * 100
        ).round(2)
    })

    display(geographic_coverage)

# Inspect countries_charted if available
if "countries_charted" in df_features.columns:
    print("\nCountries charted statistics")
    print("----------------------------")

    display(
        df_features["countries_charted"]
        .describe()
        .to_frame("countries_charted")
    )

    print("\nSample tracks with geographic chart information")
    print("-----------------------------------------------")

    display(
        df_features.loc[
            df_features["countries_charted"].notna(),
            [
                "Track",
                "Artist",
                "countries_charted",
                "chart_observations",
                "best_chart_position"
            ]
        ].head(15)
    )

Geographic feature availability
-------------------------------
Total dataset columns: 57
Geographic-related columns found: 2
• countries_charted
• broad_geographic_reach

Geographic feature coverage
---------------------------


,Data Type,Available Values,Missing Values,Coverage (%)
countries_charted,float64,2235,2358,48.66
broad_geographic_reach,int64,4593,0,100.00



Countries charted statistics
----------------------------


,countries_charted
count,"2,235.00"
mean,30.27
std,23.40
min,1.00
25%,8.00
50%,24.00
75%,54.00
max,74.00



Sample tracks with geographic chart information
-----------------------------------------------


,Track,Artist,countries_charted,chart_observations,best_chart_position
3,Flowers,Miley Cyrus,74.00,873.00,1.00
15,LALA,Myke Towers,1.00,1.00,168.00
19,As It Was,Harry Styles,69.00,"2,125.00",1.00
26,STAY (with Justin Bieber),The Kid LAROI,70.00,"4,486.00",1.00
27,Baby Shark,Pinkfong,6.00,41.00,88.00
29,Numb / Encore,JAY-Z,42.00,167.00,53.00
39,Dance Monkey,Tones And I,72.00,"7,013.00",1.00
42,I'm Good (Blue),David Guetta,65.00,638.00,1.00
48,If We Ever Broke Up,Mae Stephens,23.00,128.00,15.00
49,Despacito,Luis Fonsi,5.00,6.00,154.00


#### Interpretation

The geographic feature review identified two geographic-related variables:
`countries_charted` and the previously engineered `broad_geographic_reach`
indicator.

The original `countries_charted` feature is available for 2,235 records,
representing 48.66% of the complete dataset. This coverage corresponds to the
records successfully matched with historical chart information.

Among tracks with available historical geographic information, the average
number of countries charted is approximately 30, while the median is 24.
The values range from 1 to 74 countries, demonstrating substantial variation
in the international reach of tracks.

The upper quartile begins at 54 countries, indicating that approximately
one quarter of tracks with historical chart information achieved particularly
broad international chart coverage.

However, `countries_charted` represents the number of countries in which a
track appeared rather than identifying the individual countries or describing
performance within each market. Therefore, the integrated dataset supports
the engineering of international-reach features, but it cannot independently
support detailed country-level market analysis.

Detailed geographic intelligence within PMIP will require the underlying
historical chart dataset, which retains individual country-level observations.

### 5.2 Market Coverage Indicators

This section engineers features that describe the extent of a track's geographic market coverage using the available historical chart information.

The raw `countries_charted` variable shows how many markets a track reached, while the previously created `broad_geographic_reach` indicator provides a simple binary measure of whether a track achieved wider international coverage.

To make geographic coverage easier to compare across tracks, a normalised market coverage score is also created. This score expresses the number of countries charted relative to the maximum number of countries observed in the historical dataset.

The engineered indicators therefore provide both continuous and categorical representations of market coverage for later PMIP modelling and analytics.

In [18]:
# Determine the maximum observed market coverage
max_countries_charted = df_features["countries_charted"].max()

# Create normalised market coverage score
df_features["market_coverage_score"] = (
    df_features["countries_charted"] / max_countries_charted
)

# Validate the engineered feature
valid_market_coverage = df_features["market_coverage_score"].notna()

print("Market coverage indicator feature engineering")
print("--------------------------------------------")
print(f"Maximum countries observed: {max_countries_charted:.0f}")
print(f"Total records: {len(df_features):,}")
print(f"Records with geographic data: {valid_market_coverage.sum():,}")
print(f"Records without geographic data: {(~valid_market_coverage).sum():,}")

print(
    f"Market coverage score coverage: "
    f"{valid_market_coverage.mean() * 100:.2f}%"
)

print("\nMarket coverage score statistics")
print("--------------------------------")

display(
    df_features.loc[
        valid_market_coverage,
        [
            "countries_charted",
            "market_coverage_score"
        ]
    ].describe()
)

print("\nSample market coverage indicators")
print("---------------------------------")

display(
    df_features.loc[
        valid_market_coverage,
        [
            "Track",
            "Artist",
            "countries_charted",
            "broad_geographic_reach",
            "market_coverage_score"
        ]
    ]
    .sort_values(
        "market_coverage_score",
        ascending=False
    )
    .head(15)
)

Market coverage indicator feature engineering
--------------------------------------------
Maximum countries observed: 74
Total records: 4,593
Records with geographic data: 2,235
Records without geographic data: 2,358
Market coverage score coverage: 48.66%

Market coverage score statistics
--------------------------------


,countries_charted,market_coverage_score
count,"2,235.00","2,235.00"
mean,30.27,0.41
std,23.40,0.32
min,1.00,0.01
25%,8.00,0.11
50%,24.00,0.32
75%,54.00,0.73
max,74.00,1.00



Sample market coverage indicators
---------------------------------


,Track,Artist,countries_charted,broad_geographic_reach,market_coverage_score
3,Flowers,Miley Cyrus,74.00,1,1.00
3751,Flowers,Miley Cyrus,74.00,1,1.00
67,Kill Bill,SZA,74.00,1,1.00
1626,Flowers,Miley Cyrus,74.00,1,1.00
4324,Kill Bill,SZA,74.00,1,1.00
262,Starboy,The Weeknd,73.00,1,0.99
39,Dance Monkey,Tones And I,72.00,1,0.97
55,Blinding Lights,The Weeknd,72.00,1,0.97
1828,Save Your Tears,The Weeknd,72.00,1,0.97
166,ROCKSTAR (feat. Roddy Ricch),DaBaby,72.00,1,0.97


#### Interpretation

The market coverage indicators were successfully generated for the 2,235
records with historical geographic information, representing 48.66% of the
integrated dataset.

The normalised `market_coverage_score` ranges from approximately 0.01 to 1.00,
where higher values indicate that a track charted across a larger proportion
of the maximum number of countries observed in the historical dataset.

The average score is approximately 0.41, while the median is approximately
0.32. This suggests that a typical historically matched track reached around
one-third of the maximum observed geographic market coverage.

The upper quartile begins at approximately 0.73, indicating that the strongest
quarter of historically matched tracks achieved particularly broad market
coverage.

The continuous `market_coverage_score` complements the existing
`broad_geographic_reach` indicator. The score preserves differences in the
extent of geographic coverage, while the binary indicator provides a simpler
interpretation of whether a track reached a broad range of markets.

Records without historical geographic information remain missing rather than
being assigned a score of zero, preserving the distinction between unavailable
market data and genuinely limited geographic performance.

### 5.3 Geographic Performance Intensity

Market coverage measures how widely a track has appeared across geographic
markets, but it does not indicate how strongly the track performed within
those markets.

To capture this distinction, a geographic performance intensity feature is
created by dividing the track's total historical chart streams by the number
of countries in which it charted.

This produces an approximate measure of historical streaming activity per
charted market.

A higher value represents stronger streaming intensity relative to geographic
reach, while a lower value indicates that historical streaming activity was
distributed across markets at a lower average intensity.

Records without historical geographic information remain missing rather than
being assigned a value of zero.

In [19]:
# Create geographic performance intensity feature

valid_geographic_performance = (
    df_features["countries_charted"].notna()
    & df_features["total_historical_streams"].notna()
    & (df_features["countries_charted"] > 0)
)

df_features["geographic_performance_intensity"] = np.nan

df_features.loc[
    valid_geographic_performance,
    "geographic_performance_intensity"
] = (
    df_features.loc[
        valid_geographic_performance,
        "total_historical_streams"
    ]
    /
    df_features.loc[
        valid_geographic_performance,
        "countries_charted"
    ]
)

# Validation
records_with_intensity = (
    df_features["geographic_performance_intensity"]
    .notna()
    .sum()
)

records_without_intensity = (
    df_features["geographic_performance_intensity"]
    .isna()
    .sum()
)

coverage_percentage = (
    records_with_intensity / len(df_features)
) * 100

print("Geographic performance intensity feature engineering")
print("-----------------------------------------------")
print(f"Total records: {len(df_features):,}")
print(
    f"Records with geographic performance data: "
    f"{records_with_intensity:,}"
)
print(
    f"Records without geographic performance data: "
    f"{records_without_intensity:,}"
)
print(
    f"Geographic performance intensity coverage: "
    f"{coverage_percentage:.2f}%"
)

print("\nGeographic performance intensity statistics")
print("-------------------------------------------")

display(
    df_features[
        ["geographic_performance_intensity"]
    ].describe().round(2)
)

print("\nSample geographic performance intensity features")
print("-----------------------------------------------")

display(
    df_features.loc[
        valid_geographic_performance,
        [
            "Track",
            "Artist",
            "countries_charted",
            "total_historical_streams",
            "market_coverage_score",
            "geographic_performance_intensity"
        ]
    ]
    .sort_values(
        "geographic_performance_intensity",
        ascending=False
    )
    .head(15)
)

Geographic performance intensity feature engineering
-----------------------------------------------
Total records: 4,593
Records with geographic performance data: 2,235
Records without geographic performance data: 2,358
Geographic performance intensity coverage: 48.66%

Geographic performance intensity statistics
-------------------------------------------


,geographic_performance_intensity
count,"2,235.00"
mean,"12,548,404.17"
std,"14,146,991.48"
min,"1,904.00"
25%,"2,787,071.45"
50%,"8,384,070.09"
75%,"17,401,698.08"
max,"202,417,971.50"



Sample geographic performance intensity features
-----------------------------------------------


,Track,Artist,countries_charted,total_historical_streams,market_coverage_score,geographic_performance_intensity
2715,Whiskey Glasses,Morgan Wallen,2.00,"404,835,943.00",0.03,"202,417,971.50"
1590,Khairiyat,Arijit Singh,1.00,"110,114,415.00",0.01,"110,114,415.00"
2565,Rumah Singgah,Fabio Asher,2.00,"190,966,107.00",0.03,"95,483,053.50"
2979,Baby Me Atende,Matheus Fernandes,3.00,"272,446,923.00",0.04,"90,815,641.00"
3252,We Rollin,Shubh,1.00,"90,610,967.00",0.01,"90,610,967.00"
3480,Chasin' You,Morgan Wallen,2.00,"176,992,467.00",0.03,"88,496,233.50"
544,Wasted On You,Morgan Wallen,3.00,"264,841,347.00",0.04,"88,280,449.00"
55,Blinding Lights,The Weeknd,72.00,"6,226,855,629.00",0.97,"86,484,105.96"
753,Pasoori,Shae Gill,3.00,"253,265,841.00",0.04,"84,421,947.00"
1512,Beautiful Crazy,Luke Combs,4.00,"337,677,442.00",0.05,"84,419,360.50"


#### Interpretation

The `geographic_performance_intensity` feature was successfully created for the 2,235 records with valid historical geographic information, giving the feature a coverage rate of 48.66%.

The feature measures historical streaming activity relative to the number of countries in which a track charted. This provides a different perspective from market coverage alone because it distinguishes between tracks that reached many markets and tracks that generated particularly strong streaming performance within a smaller number of markets.

The mean geographic performance intensity is approximately 12.55 million historical streams per charted country, while the median is lower at approximately 8.38 million. This difference suggests a positively skewed distribution in which a smaller number of tracks achieve substantially higher geographic streaming intensity than the majority.

The values range from approximately 1,904 streams per market to more than 202 million streams per market, demonstrating considerable variation in how strongly tracks perform relative to their geographic reach.

The results also illustrate the difference between breadth and intensity. Some tracks achieve very high market coverage scores, while others achieve lower geographic reach but much stronger streaming intensity within the markets where they charted. These features may therefore provide complementary signals for later PMIP geographic and market-performance models.

# 6. Artist Audience Feature Engineering

This section engineers features that describe the size, strength and behaviour of an artist's audience using the available listener statistics.

The integrated PMIP dataset contains artist-level variables including current listeners, peak listeners, ranking peak and daily audience trend. These variables can be transformed into additional indicators that may help represent artist audience strength, growth and stability.

The engineered audience features are intended to support later PMIP components such as artist performance forecasting, momentum scoring and audience intelligence.

## 6.1 Listener Strength

Listener strength represents the size of an artist's current audience relative to the largest listener count observed within the available artist data.

A normalised `listener_strength_score` is created so that artist audience size can be compared on a consistent scale from 0 to 1.

Higher values indicate artists with larger current listener audiences relative to the maximum observed listener count, while records without artist listener information remain missing.

In [20]:
# Determine maximum observed listener count
max_listeners = df_features["Listeners"].max()

# Create normalised listener strength score
df_features["listener_strength_score"] = (
    df_features["Listeners"] / max_listeners
)

# Validate engineered feature
valid_listener_strength = (
    df_features["listener_strength_score"].notna()
)

print("Listener strength feature engineering")
print("------------------------------------")
print(f"Maximum listeners observed: {max_listeners:,.0f}")
print(f"Total records: {len(df_features):,}")
print(
    f"Records with listener data: "
    f"{valid_listener_strength.sum():,}"
)
print(
    f"Records without listener data: "
    f"{(~valid_listener_strength).sum():,}"
)

print(
    f"Listener strength coverage: "
    f"{valid_listener_strength.mean() * 100:.2f}%"
)

print("\nListener strength statistics")
print("----------------------------")

display(
    df_features.loc[
        valid_listener_strength,
        [
            "Listeners",
            "listener_strength_score"
        ]
    ].describe()
)

print("\nSample listener strength features")
print("---------------------------------")

display(
    df_features.loc[
        valid_listener_strength,
        [
            "Artist",
            "Listeners",
            "PkListeners",
            "Peak",
            "listener_strength_score"
        ]
    ]
    .sort_values(
        "listener_strength_score",
        ascending=False
    )
    .head(15)
)

Listener strength feature engineering
------------------------------------
Maximum listeners observed: 107,592,328
Total records: 4,593
Records with listener data: 3,389
Records without listener data: 1,204
Listener strength coverage: 73.79%

Listener strength statistics
----------------------------


,Listeners,listener_strength_score
count,"3,389.00","3,389.00"
mean,"31,168,246.74",0.29
std,"24,312,165.49",0.23
min,"4,281,434.00",0.04
25%,"11,178,406.00",0.10
50%,"23,976,058.00",0.22
75%,"46,559,302.00",0.43
max,"107,592,328.00",1.00



Sample listener strength features
---------------------------------


,Artist,Listeners,PkListeners,Peak,listener_strength_score
4240,The Weeknd,"107,592,328.00","113,034,886.00",1.00,1.00
2609,The Weeknd,"107,592,328.00","113,034,886.00",1.00,1.00
605,The Weeknd,"107,592,328.00","113,034,886.00",1.00,1.00
370,The Weeknd,"107,592,328.00","113,034,886.00",1.00,1.00
387,The Weeknd,"107,592,328.00","113,034,886.00",1.00,1.00
1428,The Weeknd,"107,592,328.00","113,034,886.00",1.00,1.00
1462,The Weeknd,"107,592,328.00","113,034,886.00",1.00,1.00
1487,The Weeknd,"107,592,328.00","113,034,886.00",1.00,1.00
3245,The Weeknd,"107,592,328.00","113,034,886.00",1.00,1.00
1828,The Weeknd,"107,592,328.00","113,034,886.00",1.00,1.00


#### Interpretation

The `listener_strength_score` was successfully generated for 3,389 of the 4,593 records, providing 73.79% coverage. The remaining 1,204 records do not contain matched artist listener information and therefore retain missing listener-strength values.

The maximum observed audience size is approximately 107.59 million listeners, which is used as the reference point for normalising the feature. The resulting score ranges from approximately 0.04 to 1.00, where higher values represent larger artist audiences relative to the largest audience observed in the dataset.

The median listener strength score is 0.22, while the mean is 0.29. This indicates that most matched artists have substantially smaller audiences than the largest artists in the dataset, while a smaller group of highly popular artists increases the overall average.

The repeated appearances of artists such as The Weeknd occur because the integrated dataset is track-level while listener statistics are artist-level. Therefore, the same artist audience information is associated with multiple tracks by that artist. This is expected and should be considered during later model development to avoid treating repeated artist-level measurements as independent audience observations.

Overall, `listener_strength_score` provides PMIP with a standardised measure of current artist audience size that can be compared across artists and combined with other performance indicators during later modelling.

## 6.2 Peak Listener Relationship

This section compares an artist's current listener count with their recorded peak listener count.

The engineered `listener_peak_ratio` measures the proportion of an artist's peak audience that is represented by their current listener count. A value close to 1 indicates that the artist is currently performing near their observed audience peak, while lower values indicate a larger difference between current and peak listener levels.

This feature complements `listener_strength_score` because audience size and proximity to peak performance represent different aspects of artist performance. An artist may have a relatively small audience but currently be close to their historical peak, while a globally established artist may have a much larger audience but be further below their peak.

In [21]:
# Identify records with valid current and peak listener information
valid_peak_listener_data = (
    df_features["Listeners"].notna()
    & df_features["PkListeners"].notna()
    & (df_features["PkListeners"] > 0)
)

# Create listener-to-peak ratio
df_features["listener_peak_ratio"] = np.nan

df_features.loc[
    valid_peak_listener_data,
    "listener_peak_ratio"
] = (
    df_features.loc[
        valid_peak_listener_data,
        "Listeners"
    ]
    / df_features.loc[
        valid_peak_listener_data,
        "PkListeners"
    ]
)

# Check whether any ratios exceed 1
ratios_above_one = (
    df_features["listener_peak_ratio"] > 1
).sum()

print("Peak listener relationship feature engineering")
print("----------------------------------------------")
print(f"Total records: {len(df_features):,}")
print(
    f"Records with valid listener and peak data: "
    f"{valid_peak_listener_data.sum():,}"
)
print(
    f"Records without valid listener and peak data: "
    f"{(~valid_peak_listener_data).sum():,}"
)
print(
    f"Listener peak ratio coverage: "
    f"{valid_peak_listener_data.mean() * 100:.2f}%"
)
print(f"Ratios above 1: {ratios_above_one:,}")

print("\nListener peak ratio statistics")
print("------------------------------")

display(
    df_features.loc[
        valid_peak_listener_data,
        [
            "Listeners",
            "PkListeners",
            "listener_peak_ratio"
        ]
    ].describe()
)

print("\nSample listener peak relationships")
print("----------------------------------")

display(
    df_features.loc[
        valid_peak_listener_data,
        [
            "Artist",
            "Listeners",
            "PkListeners",
            "listener_strength_score",
            "listener_peak_ratio"
        ]
    ]
    .drop_duplicates(subset=["Artist"])
    .sort_values(
        "listener_peak_ratio",
        ascending=False
    )
    .head(15)
)

Peak listener relationship feature engineering
----------------------------------------------
Total records: 4,593
Records with valid listener and peak data: 3,389
Records without valid listener and peak data: 1,204
Listener peak ratio coverage: 73.79%
Ratios above 1: 0

Listener peak ratio statistics
------------------------------


,Listeners,PkListeners,listener_peak_ratio
count,"3,389.00","3,389.00","3,389.00"
mean,"31,168,246.74","33,671,018.95",0.92
std,"24,312,165.49","25,721,963.76",0.08
min,"4,281,434.00","4,393,932.00",0.40
25%,"11,178,406.00","12,231,275.00",0.89
50%,"23,976,058.00","25,280,553.00",0.94
75%,"46,559,302.00","52,131,934.00",0.99
max,"107,592,328.00","113,034,886.00",1.00



Sample listener peak relationships
----------------------------------


,Artist,Listeners,PkListeners,listener_strength_score,listener_peak_ratio
3980,Evanescence,"12,427,349.00","12,427,349.00",0.12,1.00
2859,MK,"9,741,098.00","9,741,098.00",0.09,1.00
4063,Dhee,"6,369,496.00","6,369,496.00",0.06,1.00
1406,Russ,"16,701,888.00","16,701,888.00",0.16,1.00
2763,Luan Santana,"11,716,351.00","11,716,351.00",0.11,1.00
1388,Chefin,"5,920,459.00","5,920,459.00",0.06,1.00
1381,Lorde,"21,135,977.00","21,135,977.00",0.20,1.00
447,Anirudh Ravichander,"23,976,058.00","23,976,058.00",0.22,1.00
3999,Men I Trust,"8,422,936.00","8,422,936.00",0.08,1.00
934,Warren Zeiders,"4,567,972.00","4,567,972.00",0.04,1.00


#### Interpretation

The `listener_peak_ratio` was successfully calculated for 3,389 records, representing 73.79% of the integrated dataset. The remaining 1,204 records do not contain matched listener information and therefore cannot receive this feature.

The ratio ranges from 0.40 to 1.00, with no values exceeding 1.00, confirming that current listener counts do not exceed the recorded peak listener values within the available data.

The median ratio of 0.94 indicates that half of the matched records currently retain at least 94% of their recorded peak listener audience. The mean value of 0.92 similarly suggests that artists in the dataset generally remain relatively close to their observed peak audience levels.

The upper quartile reaches 0.99, showing that at least 25% of the matched records are extremely close to their recorded peak. Several artists have a ratio of exactly 1.00, meaning their current listener count is equal to their recorded peak listener count.

This feature provides information that differs from `listener_strength_score`. The listener strength score measures the relative size of an artist's audience, whereas `listener_peak_ratio` measures how close the artist's current audience is to their own observed peak. Therefore, an artist can have a relatively small audience but still have a high peak ratio.

Overall, `listener_peak_ratio` provides PMIP with a useful relative audience-performance indicator that can complement absolute audience size during later modelling and artist intelligence analysis.

## 6.3 Audience Trend Indicators

This section engineers features that describe the current direction and recent behaviour of an artist's audience.

The available listener dataset contains `Listeners`, `PkListeners`, `Peak` and `Daily Trend`. The `Daily Trend` variable provides information about recent changes in listener numbers and can therefore be used to distinguish artists whose audiences are currently growing, declining or remaining stable.

Three features are engineered:

- `listener_daily_growth_rate` — expresses the daily listener change relative to the artist's current audience size;
- `audience_trend_direction` — categorises the current audience direction as declining, stable or growing;
- `audience_below_peak` — measures the proportional distance between the artist's current listener audience and their recorded peak audience.

These features complement the audience size and peak relationship features created previously by describing the direction and relative movement of artist audiences rather than only their absolute size.

In [22]:
# Identify records with valid listener and daily-trend data
valid_trend_data = (
    df_features["Listeners"].notna()
    & df_features["Daily Trend"].notna()
    & (df_features["Listeners"] > 0)
)

# ---------------------------------------------------------
# 1. Daily listener growth rate
# ---------------------------------------------------------

df_features["listener_daily_growth_rate"] = np.nan

df_features.loc[
    valid_trend_data,
    "listener_daily_growth_rate"
] = (
    df_features.loc[valid_trend_data, "Daily Trend"]
    / df_features.loc[valid_trend_data, "Listeners"]
)

# ---------------------------------------------------------
# 2. Audience trend direction
# ---------------------------------------------------------

df_features["audience_trend_direction"] = pd.NA

df_features.loc[
    valid_trend_data
    & (df_features["Daily Trend"] > 0),
    "audience_trend_direction"
] = "Growing"

df_features.loc[
    valid_trend_data
    & (df_features["Daily Trend"] < 0),
    "audience_trend_direction"
] = "Declining"

df_features.loc[
    valid_trend_data
    & (df_features["Daily Trend"] == 0),
    "audience_trend_direction"
] = "Stable"

# ---------------------------------------------------------
# 3. Distance below recorded listener peak
# ---------------------------------------------------------

valid_peak_data = df_features["listener_peak_ratio"].notna()

df_features["audience_below_peak"] = np.nan

df_features.loc[
    valid_peak_data,
    "audience_below_peak"
] = (
    1 - df_features.loc[
        valid_peak_data,
        "listener_peak_ratio"
    ]
)

# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------

print("Audience trend indicator feature engineering")
print("--------------------------------------------")
print(f"Total records: {len(df_features):,}")
print(
    f"Records with valid trend data: "
    f"{valid_trend_data.sum():,}"
)
print(
    f"Records without valid trend data: "
    f"{(~valid_trend_data).sum():,}"
)
print(
    f"Audience trend coverage: "
    f"{valid_trend_data.mean() * 100:.2f}%"
)

print("\nAudience trend direction distribution")
print("-------------------------------------")

trend_distribution = (
    df_features.loc[
        valid_trend_data,
        "audience_trend_direction"
    ]
    .value_counts()
    .rename_axis("Audience Trend")
    .to_frame("Records")
)

trend_distribution["Percentage (%)"] = (
    trend_distribution["Records"]
    / valid_trend_data.sum()
    * 100
).round(2)

display(trend_distribution)

print("\nAudience trend feature statistics")
print("---------------------------------")

display(
    df_features.loc[
        valid_trend_data,
        [
            "Daily Trend",
            "listener_daily_growth_rate",
            "audience_below_peak"
        ]
    ].describe()
)

print("\nSample artist audience trend indicators")
print("---------------------------------------")

display(
    df_features.loc[
        valid_trend_data,
        [
            "Artist",
            "Listeners",
            "PkListeners",
            "Daily Trend",
            "listener_strength_score",
            "listener_peak_ratio",
            "listener_daily_growth_rate",
            "audience_below_peak",
            "audience_trend_direction"
        ]
    ]
    .drop_duplicates(subset=["Artist"])
    .sort_values(
        "listener_daily_growth_rate",
        ascending=False
    )
    .head(15)
)

Audience trend indicator feature engineering
--------------------------------------------
Total records: 4,593
Records with valid trend data: 3,389
Records without valid trend data: 1,204
Audience trend coverage: 73.79%

Audience trend direction distribution
-------------------------------------


,Records,Percentage (%)
Audience Trend,,
Declining,1844,54.41
Growing,1545,45.59



Audience trend feature statistics
---------------------------------


,Daily Trend,listener_daily_growth_rate,audience_below_peak
count,"3,389.00","3,389.00","3,389.00"
mean,"3,045.85",0.00,0.08
std,"127,065.17",0.00,0.08
min,"-368,853.00",-0.03,0.00
25%,"-42,758.00",-0.00,0.01
50%,"-4,254.00",-0.00,0.06
75%,"29,283.00",0.00,0.11
max,"743,072.00",0.06,0.60



Sample artist audience trend indicators
---------------------------------------


,Artist,Listeners,PkListeners,Daily Trend,listener_strength_score,listener_peak_ratio,listener_daily_growth_rate,audience_below_peak,audience_trend_direction
2428,Bryan Martin,"5,786,225.00","5,786,225.00","356,099.00",0.05,1.00,0.06,0.00,Growing
392,Calle 24,"6,923,319.00","6,923,319.00","322,507.00",0.06,1.00,0.05,0.00,Growing
775,Dove Cameron,"11,178,406.00","15,363,797.00","504,512.00",0.10,0.73,0.05,0.27,Growing
1127,Tee Grizzley,"4,890,843.00","5,192,358.00","174,802.00",0.05,0.94,0.04,0.06,Growing
705,Lefty Sm,"5,409,932.00","5,409,932.00","191,077.00",0.05,1.00,0.04,0.00,Growing
528,Megan Thee Stallion,"22,276,169.00","24,128,927.00","743,072.00",0.21,0.92,0.03,0.08,Growing
4249,Duke Dumont,"6,714,435.00","6,830,967.00","223,078.00",0.06,0.98,0.03,0.02,Growing
2084,D-Block Europe,"14,019,098.00","14,019,098.00","458,049.00",0.13,1.00,0.03,0.00,Growing
1206,V,"11,606,153.00","11,606,153.00","302,681.00",0.11,1.00,0.03,0.00,Growing
168,Zach Bryan,"21,027,724.00","21,027,724.00","418,482.00",0.20,1.00,0.02,0.00,Growing


#### Interpretation

The audience trend indicators were successfully generated for 3,389 records, representing 73.79% of the integrated dataset. The remaining 1,204 records do not contain matched listener trend information and therefore retain missing values for these engineered features.

The audience direction results show that 1,844 records (54.41%) are associated with declining listener audiences, while 1,545 records (45.59%) are associated with growing audiences. No records were classified as stable because none of the available `Daily Trend` values were exactly zero.

The `Daily Trend` values range from a decrease of 368,853 listeners to an increase of 743,072 listeners. However, absolute listener changes can be difficult to compare between artists with substantially different audience sizes. The engineered `listener_daily_growth_rate` addresses this by expressing the daily change relative to the artist's current listener audience.

The listener daily growth rate ranges from approximately -0.03 to 0.06. This indicates that the strongest observed daily decline is approximately 3% of the artist's current audience, while the strongest observed daily increase is approximately 6%.

The `audience_below_peak` feature ranges from 0.00 to 0.60. A value of 0 indicates that an artist's current listener count is equal to their recorded peak, while larger values represent a greater distance below that peak. The median value of 0.06 indicates that half of the matched records are within approximately 6% of their recorded peak audience.

Together, these engineered variables provide complementary measures of audience behaviour. `listener_strength_score` represents relative audience size, `listener_peak_ratio` represents proximity to the artist's recorded peak, `listener_daily_growth_rate` measures recent proportional audience movement, and `audience_trend_direction` provides an interpretable classification of whether the audience is currently growing or declining.

These features may be particularly useful for PMIP's later artist momentum analysis because they distinguish large established audiences from artists experiencing stronger current audience growth.

### 6.4 Artist Audience Feature Validation

The final stage of artist audience feature engineering validates the engineered
audience variables before they are used in later modelling and analysis.

The validation checks the availability, data types and expected ranges of the
listener strength, peak relationship and audience trend features. It also
verifies that the engineered values remain logically consistent with the
original listener statistics.

The features validated in this section are:

- `listener_strength_score`
- `listener_peak_ratio`
- `listener_daily_growth_rate`
- `audience_below_peak`
- `audience_trend_direction`

This validation helps ensure that the artist audience features are reliable and
suitable for later machine-learning development.

In [23]:
# 6.4 Artist Audience Feature Validation

import numpy as np
import pandas as pd

audience_features = [
    "listener_strength_score",
    "listener_peak_ratio",
    "listener_daily_growth_rate",
    "audience_below_peak",
    "audience_trend_direction"
]

print("Artist audience feature validation")
print("----------------------------------")
print(f"Total records: {len(df_features):,}")

# ---------------------------------------------------------
# 1. Check that all expected engineered features exist
# ---------------------------------------------------------

missing_feature_columns = [
    column
    for column in audience_features
    if column not in df_features.columns
]

print(
    f"Missing engineered feature columns: "
    f"{len(missing_feature_columns)}"
)

if missing_feature_columns:
    print("Missing columns:")
    for column in missing_feature_columns:
        print(f"• {column}")

# ---------------------------------------------------------
# 2. Validate numerical feature ranges
# ---------------------------------------------------------

numerical_audience_features = [
    "listener_strength_score",
    "listener_peak_ratio",
    "listener_daily_growth_rate",
    "audience_below_peak"
]

validation_results = []

for column in numerical_audience_features:

    available_values = df_features[column].notna().sum()
    missing_values = df_features[column].isna().sum()

    validation_results.append(
        {
            "Feature": column,
            "Data Type": str(df_features[column].dtype),
            "Available Values": available_values,
            "Missing Values": missing_values,
            "Coverage (%)": round(
                available_values / len(df_features) * 100,
                2
            ),
            "Minimum": df_features[column].min(),
            "Maximum": df_features[column].max()
        }
    )

validation_summary = pd.DataFrame(validation_results)

print("\nNumerical audience feature summary")
print("----------------------------------")

display(validation_summary)

# ---------------------------------------------------------
# 3. Logical validation checks
# ---------------------------------------------------------

invalid_listener_strength = (
    (
        (df_features["listener_strength_score"] < 0)
        | (df_features["listener_strength_score"] > 1)
    )
    & df_features["listener_strength_score"].notna()
).sum()

invalid_peak_ratio = (
    (
        (df_features["listener_peak_ratio"] < 0)
        | (df_features["listener_peak_ratio"] > 1)
    )
    & df_features["listener_peak_ratio"].notna()
).sum()

invalid_audience_below_peak = (
    (
        (df_features["audience_below_peak"] < 0)
        | (df_features["audience_below_peak"] > 1)
    )
    & df_features["audience_below_peak"].notna()
).sum()

# Current listeners should not exceed recorded peak listeners
listeners_above_peak = (
    df_features["Listeners"].notna()
    & df_features["PkListeners"].notna()
    & (df_features["Listeners"] > df_features["PkListeners"])
).sum()

# Check audience trend direction against Daily Trend
valid_trend_rows = (
    df_features["Daily Trend"].notna()
    & df_features["audience_trend_direction"].notna()
)

trend_direction_mismatches = (
    valid_trend_rows
    & (
        (
            (df_features["Daily Trend"] > 0)
            & (df_features["audience_trend_direction"] != "Growing")
        )
        |
        (
            (df_features["Daily Trend"] < 0)
            & (df_features["audience_trend_direction"] != "Declining")
        )
        |
        (
            (df_features["Daily Trend"] == 0)
            & (df_features["audience_trend_direction"] != "Stable")
        )
    )
).sum()

print("\nLogical validation checks")
print("-------------------------")
print(
    f"Invalid listener strength scores: "
    f"{invalid_listener_strength:,}"
)
print(
    f"Invalid listener peak ratios: "
    f"{invalid_peak_ratio:,}"
)
print(
    f"Invalid audience-below-peak values: "
    f"{invalid_audience_below_peak:,}"
)
print(
    f"Listener values above recorded peak: "
    f"{listeners_above_peak:,}"
)
print(
    f"Audience trend direction mismatches: "
    f"{trend_direction_mismatches:,}"
)

# ---------------------------------------------------------
# 4. Audience trend category validation
# ---------------------------------------------------------

print("\nAudience trend category validation")
print("----------------------------------")

trend_validation = (
    df_features["audience_trend_direction"]
    .value_counts(dropna=False)
    .rename_axis("Audience Trend")
    .to_frame("Records")
)

trend_validation["Percentage (%)"] = (
    trend_validation["Records"]
    / len(df_features)
    * 100
).round(2)

display(trend_validation)

# ---------------------------------------------------------
# 5. Final sample
# ---------------------------------------------------------

print("\nValidated artist audience feature sample")
print("----------------------------------------")

display(
    df_features[
        [
            "Artist",
            "Listeners",
            "PkListeners",
            "Daily Trend",
            "listener_strength_score",
            "listener_peak_ratio",
            "listener_daily_growth_rate",
            "audience_below_peak",
            "audience_trend_direction"
        ]
    ]
    .dropna(subset=["Listeners"])
    .head(15)
)

Artist audience feature validation
----------------------------------
Total records: 4,593
Missing engineered feature columns: 0

Numerical audience feature summary
----------------------------------


,Feature,Data Type,Available Values,Missing Values,Coverage (%),Minimum,Maximum
0,listener_strength_score,float64,3389,1204,73.79,0.04,1.00
1,listener_peak_ratio,float64,3389,1204,73.79,0.40,1.00
2,listener_daily_growth_rate,float64,3389,1204,73.79,-0.03,0.06
3,audience_below_peak,float64,3389,1204,73.79,0.00,0.60



Logical validation checks
-------------------------
Invalid listener strength scores: 0
Invalid listener peak ratios: 0
Invalid audience-below-peak values: 0
Listener values above recorded peak: 0
Audience trend direction mismatches: 0

Audience trend category validation
----------------------------------


,Records,Percentage (%)
Audience Trend,,
Declining,1844,40.15
Growing,1545,33.64
<NA>,1204,26.21



Validated artist audience feature sample
----------------------------------------


,Artist,Listeners,PkListeners,Daily Trend,listener_strength_score,listener_peak_ratio,listener_daily_growth_rate,audience_below_peak,audience_trend_direction
1,Kendrick Lamar,"47,391,930.00","54,045,549.00","-8,503.00",0.44,0.88,-0.00,0.12,Declining
3,Miley Cyrus,"71,277,599.00","84,140,935.00","71,089.00",0.66,0.85,0.00,0.15,Growing
4,Eminem,"64,022,485.00","68,591,390.00","-52,362.00",0.60,0.93,-0.00,0.07,Declining
5,Jack Harlow,"26,620,407.00","32,198,313.00","-14,428.00",0.25,0.83,-0.00,0.17,Declining
6,Benson Boone,"14,274,669.00","14,307,536.00","-10,909.00",0.13,1.00,-0.00,0.00,Declining
9,Central Cee,"32,987,690.00","35,154,343.00","-33,056.00",0.31,0.94,-0.00,0.06,Declining
10,Post Malone,"63,914,311.00","67,066,636.00","-55,144.00",0.59,0.95,-0.00,0.05,Declining
11,Teddy Swims,"5,706,645.00","5,706,645.00","45,131.00",0.05,1.00,0.01,0.00,Growing
12,Billie Eilish,"71,793,820.00","72,368,528.00","-72,877.00",0.67,0.99,-0.00,0.01,Declining
13,Future,"52,074,976.00","58,360,653.00","-148,160.00",0.48,0.89,-0.00,0.11,Declining


#### Interpretation

The validation results confirm that the engineered artist audience features are
internally consistent and suitable for later analysis and machine-learning
development. All expected feature columns were successfully created, and no
invalid values or logical inconsistencies were detected.

Audience information is available for 3,389 of the 4,593 records, representing
73.79% coverage. The remaining 1,204 records (26.21%) do not contain the
listener information required to calculate the audience-derived features. This
missingness should therefore be considered during later model preparation
rather than interpreted as zero audience performance.

The `listener_strength_score` ranges from 0.04 to 1.00 and provides a
normalised representation of current listener size relative to the largest
audience observed in the dataset. The `listener_peak_ratio` ranges from 0.40
to 1.00 and represents how close an artist's current listener count is to
their recorded peak. Similarly, `audience_below_peak` ranges from 0.00 to
0.60, representing the proportional distance between current and peak
listener levels.

The engineered audience trend indicators show that 1,844 records are
classified as declining and 1,545 as growing. Among the 3,389 records with
available audience data, this corresponds to approximately 54.41% declining
and 45.59% growing. The remaining 1,204 records cannot be assigned an audience
trend because the required listener data is unavailable.

All logical validation checks returned zero errors. No listener-strength
scores, peak ratios or audience-below-peak values were outside their expected
ranges. No current listener values exceeded their recorded peak values, and
all audience trend classifications were consistent with the direction of the
original `Daily Trend` variable.

Overall, the artist audience feature engineering stage successfully transforms
the original listener statistics into interpretable measures of audience size,
distance from peak performance and audience growth direction. These features
can provide useful signals for later PMIP tasks such as artist momentum
analysis and predictive modelling.

## 7. Numerical Feature Transformation

### Purpose

The purpose of this section is to prepare highly skewed numerical features for
later machine-learning development.

Music-performance data often contains large differences between observations.
For example, a relatively small number of highly successful tracks or artists
may accumulate substantially more streams, listeners, playlist reach or chart
activity than the majority of the dataset. These extreme values can produce
strongly skewed numerical distributions.

This section first reviews the distributions of relevant continuous numerical
features and identifies variables with substantial skewness. Appropriate
logarithmic transformations are then applied where required to reduce the
influence of extreme values while preserving the underlying ordering and
performance information.

The transformed features are retained alongside their original variables so
that their usefulness can later be evaluated during machine-learning model
development.

The section consists of:

- **7.1 Review Skewed Features** — identify numerical variables with substantial
  distributional skewness.
- **7.2 Log Transformations** — apply appropriate logarithmic transformations to
  selected features.
- **7.3 Transformation Validation** — compare the original and transformed
  variables and verify that the transformations were performed correctly.

### 7.1 Review Skewed Features

Many music-performance variables can have highly uneven distributions because a
small number of artists or tracks may achieve substantially higher streaming,
listener or chart-performance values than the majority of records.

This section evaluates the skewness of relevant continuous numerical features
before applying any transformations. Features with strong positive or negative
skewness will be considered for transformation in the next stage.

Binary indicators, categorical variables, dates and already normalised score
features are excluded because log transformation would not provide meaningful
benefits for these variables.

In [24]:
# Review skewness of continuous numerical features

import pandas as pd
import numpy as np

# Candidate continuous features currently available in df_features
candidate_skew_features = [
    # Original performance metrics
    "Spotify Streams",
    "Spotify Playlist Count",
    "Spotify Playlist Reach",
    "YouTube Views",
    "YouTube Likes",
    "TikTok Posts",
    "TikTok Likes",
    "TikTok Views",
    "YouTube Playlist Reach",
    "Apple Music Playlist Count",
    "AirPlay Spins",
    "SiriusXM Spins",
    "Deezer Playlist Count",
    "Deezer Playlist Reach",
    "Amazon Playlist Count",
    "Pandora Streams",
    "Pandora Track Stations",
    "Soundcloud Streams",
    "Shazam Counts",

    # Historical chart features
    "best_chart_position",
    "average_chart_position",
    "chart_observations",
    "countries_charted",
    "total_historical_streams",
    "average_historical_streams",
    "maximum_historical_streams",

    # Engineered historical features
    "historical_streaming_intensity",
    "chart_longevity_days",
    "chart_longevity_years",
    "chart_activity_factor",
    "peak_chart_performance",
    "average_chart_performance",
    "geographic_performance_intensity",

    # Artist audience features
    "Listeners",
    "PkListeners",
    "Daily Trend",
    "listener_daily_growth_rate"
]

# Keep only columns that actually exist in the current dataframe
skew_features = [
    column
    for column in candidate_skew_features
    if column in df_features.columns
]

# Calculate skewness
skewness = (
    df_features[skew_features]
    .skew(numeric_only=True)
    .sort_values(ascending=False)
)

# Build review table
skewness_review = pd.DataFrame({
    "Skewness": skewness
})

# Absolute skewness helps us judge severity regardless of direction
skewness_review["Absolute Skewness"] = (
    skewness_review["Skewness"].abs()
)

# Classify the degree of skewness
def classify_skewness(value):
    absolute_value = abs(value)

    if absolute_value < 0.5:
        return "Low"
    elif absolute_value < 1.0:
        return "Moderate"
    else:
        return "High"

skewness_review["Skew Level"] = (
    skewness_review["Skewness"]
    .apply(classify_skewness)
)

# Sort by strongest skewness
skewness_review = (
    skewness_review
    .sort_values("Absolute Skewness", ascending=False)
)

print("Numerical feature skewness review")
print("---------------------------------")
print(f"Features reviewed: {len(skewness_review)}")

print(
    f"Highly skewed features: "
    f"{(skewness_review['Absolute Skewness'] >= 1).sum()}"
)

print(
    f"Moderately skewed features: "
    f"{((skewness_review['Absolute Skewness'] >= 0.5) & (skewness_review['Absolute Skewness'] < 1)).sum()}"
)

print(
    f"Low-skew features: "
    f"{(skewness_review['Absolute Skewness'] < 0.5).sum()}"
)

print("\nSkewness analysis")
print("-----------------")

display(
    skewness_review.style.format({
        "Skewness": "{:.2f}",
        "Absolute Skewness": "{:.2f}"
    })
)

print("\nHighly skewed features requiring review")
print("---------------------------------------")

highly_skewed_features = skewness_review[
    skewness_review["Absolute Skewness"] >= 1
]

display(
    highly_skewed_features.style.format({
        "Skewness": "{:.2f}",
        "Absolute Skewness": "{:.2f}"
    })
)

Numerical feature skewness review
---------------------------------
Features reviewed: 35
Highly skewed features: 30
Moderately skewed features: 4
Low-skew features: 1

Skewness analysis
-----------------


,Skewness,Absolute Skewness,Skew Level
TikTok Likes,32.05,32.05,High
TikTok Views,31.01,31.01,High
Shazam Counts,18.47,18.47,High
TikTok Posts,7.63,7.63,High
Pandora Track Stations,6.72,6.72,High
Deezer Playlist Reach,6.12,6.12,High
SiriusXM Spins,6.07,6.07,High
YouTube Views,6.02,6.02,High
AirPlay Spins,5.17,5.17,High
Deezer Playlist Count,4.79,4.79,High



Highly skewed features requiring review
---------------------------------------


,Skewness,Absolute Skewness,Skew Level
TikTok Likes,32.05,32.05,High
TikTok Views,31.01,31.01,High
Shazam Counts,18.47,18.47,High
TikTok Posts,7.63,7.63,High
Pandora Track Stations,6.72,6.72,High
Deezer Playlist Reach,6.12,6.12,High
SiriusXM Spins,6.07,6.07,High
YouTube Views,6.02,6.02,High
AirPlay Spins,5.17,5.17,High
Deezer Playlist Count,4.79,4.79,High


#### Interpretation

The skewness review identified substantial non-normality across the numerical
features. Of the 35 continuous variables reviewed, 30 were classified as highly
skewed, four as moderately skewed and only one as having low skewness.

The strongest positive skewness occurred in social-media and engagement
variables, particularly TikTok Likes, TikTok Views and Shazam Counts. This
indicates that most tracks have considerably lower engagement values while a
small number of highly successful tracks produce extremely large observations.

Strong positive skewness was also observed in several important streaming and
historical-performance variables, including Spotify Streams, historical
streams, chart observations and geographic performance intensity. These
distributions may cause extreme observations to have a disproportionate
influence on some machine-learning algorithms.

However, not every highly skewed variable should automatically be transformed.
Ranking variables such as best chart position have a specific interpretation,
while Daily Trend and listener daily growth rate can contain negative values.
The transformation stage will therefore focus primarily on non-negative
continuous count and performance variables where logarithmic transformation is
appropriate, while preserving the original variables for later comparison.

### 7.2 Log Transformations

The skewness review identified several continuous count and performance
variables with strongly positively skewed distributions. These variables
contain large differences between typical observations and a smaller number
of extremely high-performing tracks or artists.

To reduce this skewness, selected non-negative numerical features are
transformed using the natural logarithm through `log1p`. This transformation
compresses large values while preserving their relative ordering and can
reduce the influence of extreme observations during machine-learning model
training.

The `log1p` transformation is used because it can safely transform legitimate
zero values by calculating `log(1 + x)`.

Only variables where a logarithmic transformation is meaningful are included.
Ranking variables, signed trend variables, normalised scores and binary
indicators are excluded. The original variables are retained alongside the
new transformed features so that both representations remain available for
later model development and evaluation.

In [25]:
# 7.2 Log Transformations

import numpy as np
import pandas as pd

# Selected non-negative magnitude/count features based on the 7.1 review
log_transform_candidates = [
    # Cross-platform performance
    "Spotify Streams",
    "Spotify Playlist Count",
    "Spotify Playlist Reach",
    "YouTube Views",
    "YouTube Likes",
    "TikTok Posts",
    "TikTok Likes",
    "TikTok Views",
    "YouTube Playlist Reach",
    "Apple Music Playlist Count",
    "AirPlay Spins",
    "SiriusXM Spins",
    "Deezer Playlist Count",
    "Deezer Playlist Reach",
    "Amazon Playlist Count",
    "Pandora Streams",
    "Pandora Track Stations",
    "Soundcloud Streams",
    "Shazam Counts",

    # Historical performance
    "chart_observations",
    "total_historical_streams",
    "average_historical_streams",
    "maximum_historical_streams",
    "historical_streaming_intensity",
    "chart_longevity_days",
    "chart_longevity_years",
    "geographic_performance_intensity",

    # Audience size
    "Listeners",
    "PkListeners"
]

# Keep only features that exist in df_features
available_log_features = [
    column
    for column in log_transform_candidates
    if column in df_features.columns
]

print("Log transformation")
print("------------------")
print(f"Candidate features: {len(log_transform_candidates)}")
print(f"Available features: {len(available_log_features)}")

transformation_results = []

for column in available_log_features:

    numeric_values = pd.to_numeric(
        df_features[column],
        errors="coerce"
    )

    # Check whether the feature contains negative values
    negative_count = (numeric_values < 0).sum()

    if negative_count > 0:
        transformation_results.append({
            "Original Feature": column,
            "Transformed Feature": "Not transformed",
            "Negative Values": negative_count,
            "Status": "Skipped"
        })

        continue

    # Create a new log-transformed feature
    transformed_column = f"{column}_log"

    df_features[transformed_column] = np.log1p(
        numeric_values
    )

    transformation_results.append({
        "Original Feature": column,
        "Transformed Feature": transformed_column,
        "Negative Values": negative_count,
        "Status": "Transformed"
    })


transformation_summary = pd.DataFrame(
    transformation_results
)

print(
    f"Features successfully transformed: "
    f"{(transformation_summary['Status'] == 'Transformed').sum()}"
)

print(
    f"Features skipped: "
    f"{(transformation_summary['Status'] == 'Skipped').sum()}"
)

print("\nTransformation summary")
print("----------------------")

display(transformation_summary)


# Compare skewness before and after transformation
comparison_results = []

for column in available_log_features:

    transformed_column = f"{column}_log"

    if transformed_column not in df_features.columns:
        continue

    original_skewness = (
        pd.to_numeric(
            df_features[column],
            errors="coerce"
        ).skew()
    )

    transformed_skewness = (
        df_features[transformed_column].skew()
    )

    comparison_results.append({
        "Feature": column,
        "Original Skewness": original_skewness,
        "Log Skewness": transformed_skewness,
        "Absolute Skewness Reduction":
            abs(original_skewness) - abs(transformed_skewness)
    })


log_skewness_comparison = pd.DataFrame(
    comparison_results
)

log_skewness_comparison = (
    log_skewness_comparison
    .sort_values(
        "Absolute Skewness Reduction",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nSkewness before and after log transformation")
print("--------------------------------------------")

display(
    log_skewness_comparison.style.format({
        "Original Skewness": "{:.2f}",
        "Log Skewness": "{:.2f}",
        "Absolute Skewness Reduction": "{:.2f}"
    })
)

Log transformation
------------------
Candidate features: 29
Available features: 27
Features successfully transformed: 27
Features skipped: 0

Transformation summary
----------------------


,Original Feature,Transformed Feature,Negative Values,Status
0,Spotify Streams,Spotify Streams_log,0,Transformed
1,Spotify Playlist Count,Spotify Playlist Count_log,0,Transformed
2,Spotify Playlist Reach,Spotify Playlist Reach_log,0,Transformed
3,YouTube Views,YouTube Views_log,0,Transformed
4,YouTube Likes,YouTube Likes_log,0,Transformed
5,TikTok Posts,TikTok Posts_log,0,Transformed
6,TikTok Likes,TikTok Likes_log,0,Transformed
7,TikTok Views,TikTok Views_log,0,Transformed
8,YouTube Playlist Reach,YouTube Playlist Reach_log,0,Transformed
9,Apple Music Playlist Count,Apple Music Playlist Count_log,0,Transformed



Skewness before and after log transformation
--------------------------------------------


,Feature,Original Skewness,Log Skewness,Absolute Skewness Reduction
0,TikTok Likes,32.05,-1.64,30.41
1,TikTok Views,31.01,-1.59,29.42
2,Shazam Counts,18.47,-1.69,16.78
3,TikTok Posts,7.63,-0.95,6.67
4,Pandora Track Stations,6.72,-0.23,6.48
5,SiriusXM Spins,6.07,-0.17,5.91
6,Deezer Playlist Reach,6.12,-0.73,5.39
7,YouTube Views,6.02,-0.95,5.07
8,Deezer Playlist Count,4.79,0.15,4.64
9,AirPlay Spins,5.17,-0.54,4.62


#### Interpretation

The logarithmic transformation was successfully applied to 27 available
non-negative numerical features. No selected feature contained negative values,
meaning that all available transformation candidates could be processed using
`log1p` without introducing domain-related errors.

The transformation substantially reduced skewness for most of the selected
features. Some of the strongest improvements occurred in TikTok Likes,
TikTok Views and Shazam Counts, which originally displayed extremely large
positive skewness values. Similar improvements were observed across several
playlist, streaming, chart-activity and audience variables.

The artist audience variables responded particularly well to the
transformation. Listener and peak-listener distributions changed from
positively skewed distributions to values close to zero skewness, indicating
considerably more balanced transformed distributions.

The results also demonstrate that logarithmic transformation does not
automatically improve every numerical feature. Spotify Playlist Reach changed
from a skewness of 2.60 to -2.69, while Spotify Streams changed from 2.02 to
-2.17. In both cases, the absolute skewness increased slightly after the
transformation. These results suggest that the logarithmic versions of these
variables should not automatically replace their original representations.

Several other transformed features remained moderately or highly negatively
skewed despite showing substantial reductions in their original positive
skewness. This is acceptable at the feature-engineering stage because the
objective is not to force every variable into a perfectly normal distribution,
but to provide alternative representations that may reduce the influence of
extreme observations.

The original numerical features have therefore been retained alongside their
log-transformed versions. Later model development and evaluation can determine
which representation provides the most useful predictive information for each
PMIP machine-learning task.

### 7.3 Transformation Validation

The log-transformed numerical features are validated to confirm that the
transformation process was completed correctly and did not introduce invalid
values.

This validation checks that the expected transformed features were created,
that no infinite values were introduced, and that missing values remain
consistent with their corresponding original variables. The transformed
values are also compared with independently calculated `log1p` values to
verify mathematical correctness.

Finally, the absolute skewness of each original and transformed feature is
compared. This identifies features whose distributions improved following
transformation and features for which the original representation may remain
more appropriate for later machine-learning development.

In [26]:
# 7.3 Transformation Validation

import numpy as np
import pandas as pd

# Identify successfully created log-transformed features
transformed_features = [
    column
    for column in available_log_features
    if f"{column}_log" in df_features.columns
]

print("Log transformation validation")
print("-----------------------------")
print(f"Total dataset records: {len(df_features):,}")
print(f"Expected transformed features: {len(available_log_features)}")
print(f"Transformed features found: {len(transformed_features)}")
print(
    f"Missing transformed features: "
    f"{len(available_log_features) - len(transformed_features)}"
)


# ---------------------------------------------------------
# 1. Validate missing and infinite values
# ---------------------------------------------------------

validation_results = []

for original_column in transformed_features:

    transformed_column = f"{original_column}_log"

    original_values = pd.to_numeric(
        df_features[original_column],
        errors="coerce"
    )

    transformed_values = pd.to_numeric(
        df_features[transformed_column],
        errors="coerce"
    )

    original_missing = original_values.isna().sum()
    transformed_missing = transformed_values.isna().sum()

    infinite_values = np.isinf(
        transformed_values.dropna()
    ).sum()

    # Independently recalculate transformation
    expected_values = np.log1p(original_values)

    valid_mask = (
        expected_values.notna()
        & transformed_values.notna()
    )

    if valid_mask.any():
        maximum_difference = np.max(
            np.abs(
                transformed_values[valid_mask]
                - expected_values[valid_mask]
            )
        )
    else:
        maximum_difference = np.nan

    original_skew = original_values.skew()
    transformed_skew = transformed_values.skew()

    absolute_original_skew = abs(original_skew)
    absolute_transformed_skew = abs(transformed_skew)

    skewness_reduction = (
        absolute_original_skew
        - absolute_transformed_skew
    )

    if skewness_reduction > 0:
        transformation_effect = "Improved"
    elif np.isclose(skewness_reduction, 0):
        transformation_effect = "No change"
    else:
        transformation_effect = "Worsened"

    validation_results.append({
        "Feature": original_column,
        "Original Missing": original_missing,
        "Log Missing": transformed_missing,
        "Infinite Values": infinite_values,
        "Maximum Calculation Difference": maximum_difference,
        "Original Skewness": original_skew,
        "Log Skewness": transformed_skew,
        "Skewness Reduction": skewness_reduction,
        "Transformation Effect": transformation_effect
    })


transformation_validation = pd.DataFrame(
    validation_results
)


# ---------------------------------------------------------
# 2. Overall integrity checks
# ---------------------------------------------------------

missing_mismatches = (
    transformation_validation["Original Missing"]
    != transformation_validation["Log Missing"]
).sum()

infinite_total = (
    transformation_validation["Infinite Values"].sum()
)

calculation_errors = (
    transformation_validation[
        "Maximum Calculation Difference"
    ].fillna(0) > 1e-10
).sum()

improved_features = (
    transformation_validation[
        "Transformation Effect"
    ] == "Improved"
).sum()

worsened_features = (
    transformation_validation[
        "Transformation Effect"
    ] == "Worsened"
).sum()

unchanged_features = (
    transformation_validation[
        "Transformation Effect"
    ] == "No change"
).sum()


print("\nIntegrity checks")
print("----------------")

print(
    f"Missing-value mismatches: {missing_mismatches}"
)

print(
    f"Infinite transformed values: {infinite_total}"
)

print(
    f"Transformation calculation errors: {calculation_errors}"
)

print("\nDistributional effect")
print("---------------------")

print(
    f"Features with reduced absolute skewness: "
    f"{improved_features}"
)

print(
    f"Features with increased absolute skewness: "
    f"{worsened_features}"
)

print(
    f"Features with no meaningful change: "
    f"{unchanged_features}"
)


# ---------------------------------------------------------
# 3. Detailed validation table
# ---------------------------------------------------------

print("\nDetailed transformation validation")
print("----------------------------------")

display(
    transformation_validation[
        [
            "Feature",
            "Original Missing",
            "Log Missing",
            "Infinite Values",
            "Maximum Calculation Difference",
            "Original Skewness",
            "Log Skewness",
            "Skewness Reduction",
            "Transformation Effect"
        ]
    ].style.format({
        "Maximum Calculation Difference": "{:.12f}",
        "Original Skewness": "{:.2f}",
        "Log Skewness": "{:.2f}",
        "Skewness Reduction": "{:.2f}"
    })
)


# ---------------------------------------------------------
# 4. Features where log transformation worsened skewness
# ---------------------------------------------------------

worsened_transformation_features = (
    transformation_validation[
        transformation_validation[
            "Transformation Effect"
        ] == "Worsened"
    ]
    .sort_values(
        "Skewness Reduction"
    )
    .reset_index(drop=True)
)

print("\nFeatures requiring further consideration")
print("----------------------------------------")

if len(worsened_transformation_features) == 0:
    print(
        "No transformed features increased absolute skewness."
    )
else:
    display(
        worsened_transformation_features[
            [
                "Feature",
                "Original Skewness",
                "Log Skewness",
                "Skewness Reduction"
            ]
        ].style.format({
            "Original Skewness": "{:.2f}",
            "Log Skewness": "{:.2f}",
            "Skewness Reduction": "{:.2f}"
        })
    )

Log transformation validation
-----------------------------
Total dataset records: 4,593
Expected transformed features: 27
Transformed features found: 27
Missing transformed features: 0

Integrity checks
----------------
Missing-value mismatches: 0
Infinite transformed values: 0
Transformation calculation errors: 0

Distributional effect
---------------------
Features with reduced absolute skewness: 25
Features with increased absolute skewness: 2
Features with no meaningful change: 0

Detailed transformation validation
----------------------------------


,Feature,Original Missing,Log Missing,Infinite Values,Maximum Calculation Difference,Original Skewness,Log Skewness,Skewness Reduction,Transformation Effect
0,Spotify Streams,108,108,0,0.000000000000,2.02,-2.17,-0.15,Worsened
1,Spotify Playlist Count,65,65,0,0.000000000000,1.84,-1.67,0.16,Improved
2,Spotify Playlist Reach,67,67,0,0.000000000000,2.60,-2.69,-0.09,Worsened
3,YouTube Views,303,303,0,0.000000000000,6.02,-0.95,5.07,Improved
4,YouTube Likes,310,310,0,0.000000000000,4.22,-0.96,3.26,Improved
5,TikTok Posts,1168,1168,0,0.000000000000,7.63,-0.95,6.67,Improved
6,TikTok Likes,975,975,0,0.000000000000,32.05,-1.64,30.41,Improved
7,TikTok Views,976,976,0,0.000000000000,31.01,-1.59,29.42,Improved
8,YouTube Playlist Reach,1004,1004,0,0.000000000000,3.68,-1.54,2.14,Improved
9,Apple Music Playlist Count,556,556,0,0.000000000000,2.89,-0.14,2.75,Improved



Features requiring further consideration
----------------------------------------


,Feature,Original Skewness,Log Skewness,Skewness Reduction
0,Spotify Streams,2.02,-2.17,-0.15
1,Spotify Playlist Reach,2.60,-2.69,-0.09


#### Interpretation

The transformation validation confirms that all 27 expected log-transformed
features were successfully created. No transformed features were missing, and
the transformation process introduced no infinite values or calculation
errors.

Missing values were also preserved correctly. Every transformed feature
contained the same number of missing observations as its corresponding
original variable. This is important because the transformation has not
artificially created or removed data during feature engineering.

The distributional comparison shows that logarithmic transformation reduced
absolute skewness for 25 of the 27 transformed features. This demonstrates that
the transformation was effective for the majority of the selected streaming,
playlist, historical-performance and audience variables.

Two exceptions were identified. Spotify Streams changed from an original
skewness of 2.02 to a log-transformed skewness of -2.17, while Spotify Playlist
Reach changed from 2.60 to -2.69. Their absolute skewness therefore increased
slightly following transformation. The log-transformed versions of these two
features should not automatically replace their original representations and
can instead be evaluated during later model development.

Overall, the numerical transformation stage was completed successfully. The
original numerical variables have been retained alongside their transformed
versions, allowing later PMIP machine-learning experiments to determine which
representation provides better predictive performance.

## 8. Missing Value Treatment

### Purpose

The purpose of this section is to examine and appropriately represent missing
values within the engineered PMIP dataset before machine-learning development.

Missing values in this dataset do not always represent data errors. Some
features are unavailable because a track has no matched historical chart
record, while artist audience features may be unavailable because the artist
could not be matched with the listener dataset. Treating these values as zero
without considering their meaning could therefore introduce misleading
information into later models.

This section reviews the overall pattern of missingness and separately examines
historical-performance and artist-listener features. Appropriate treatment
strategies are then considered based on the meaning of the missing data.

Missingness indicator features are also created where useful so that later
machine-learning models can distinguish between an observed value and a value
that is unavailable because the corresponding data source was not matched.

The section consists of:

- **8.1 Missingness Review** — examine the amount and distribution of missing
  data across the engineered dataset.
- **8.2 Historical Feature Missingness** — investigate missing historical chart
  and performance features.
- **8.3 Artist Listener Feature Missingness** — investigate missing audience and
  listener-derived features.
- **8.4 Missingness Indicator Features** — create explicit indicators that
  preserve information about unavailable historical and audience data.

### 8.1 Missingness Review

Before applying any missing-value treatment, the current feature-engineered
dataset is reviewed to determine which variables contain missing observations
and how substantial the missingness is.

The review calculates both the number and percentage of missing values for each
feature. This helps distinguish variables with complete coverage from features
whose availability depends on historical chart matching, artist-listener
matching or the availability of platform-specific performance data.

No missing values are modified at this stage. The objective is to understand
the missingness structure before selecting appropriate treatment strategies.

In [27]:
# 8.1 Missingness Review

import pandas as pd
import numpy as np

total_records = len(df_features)
total_features = len(df_features.columns)

# Calculate missing-value counts and percentages
missing_count = df_features.isna().sum()

missing_percentage = (
    missing_count / total_records
) * 100

# Build missingness review table
missingness_review = pd.DataFrame({
    "Missing Values": missing_count,
    "Missing Percentage": missing_percentage
})

# Keep only features containing missing values
features_with_missing = (
    missingness_review[
        missingness_review["Missing Values"] > 0
    ]
    .sort_values(
        "Missing Percentage",
        ascending=False
    )
)

features_without_missing = (
    missingness_review[
        missingness_review["Missing Values"] == 0
    ]
)

print("Dataset missingness review")
print("--------------------------")
print(f"Total records: {total_records:,}")
print(f"Total features: {total_features}")
print(
    f"Features with missing values: "
    f"{len(features_with_missing)}"
)
print(
    f"Features without missing values: "
    f"{len(features_without_missing)}"
)

print(
    f"Total missing cells: "
    f"{df_features.isna().sum().sum():,}"
)

total_cells = total_records * total_features

overall_missing_percentage = (
    df_features.isna().sum().sum()
    / total_cells
) * 100

print(
    f"Overall dataset missingness: "
    f"{overall_missing_percentage:.2f}%"
)


print("\nFeatures containing missing values")
print("----------------------------------")

display(
    features_with_missing.style.format({
        "Missing Values": "{:,.0f}",
        "Missing Percentage": "{:.2f}%"
    })
)


# Group features into broad missingness levels
def classify_missingness(percentage):

    if percentage == 0:
        return "Complete"

    elif percentage < 10:
        return "Low"

    elif percentage < 25:
        return "Moderate"

    elif percentage < 50:
        return "High"

    else:
        return "Very High"


missingness_review["Missingness Level"] = (
    missingness_review["Missing Percentage"]
    .apply(classify_missingness)
)

print("\nMissingness level distribution")
print("------------------------------")

display(
    missingness_review[
        "Missingness Level"
    ]
    .value_counts()
    .rename_axis("Missingness Level")
    .to_frame("Features")
)


# Show the most incomplete features
print("\nFeatures with the highest missingness")
print("-------------------------------------")

display(
    features_with_missing
    .head(20)
    .style.format({
        "Missing Values": "{:,.0f}",
        "Missing Percentage": "{:.2f}%"
    })
)

Dataset missingness review
--------------------------
Total records: 4,593
Total features: 92
Features with missing values: 75
Features without missing values: 17
Total missing cells: 105,653
Overall dataset missingness: 25.00%

Features containing missing values
----------------------------------


,Missing Values,Missing Percentage
chart_activity_factor,"2,358",51.34%
best_chart_position,"2,358",51.34%
chart_observations,"2,358",51.34%
countries_charted,"2,358",51.34%
total_historical_streams,"2,358",51.34%
average_historical_streams,"2,358",51.34%
max_historical_streams,"2,358",51.34%
first_chart_date,"2,358",51.34%
last_chart_date,"2,358",51.34%
geographic_performance_intensity,"2,358",51.34%



Missingness level distribution
------------------------------


,Features
Missingness Level,
Very High,27
Moderate,21
Complete,17
High,17
Low,10



Features with the highest missingness
-------------------------------------


,Missing Values,Missing Percentage
chart_activity_factor,"2,358",51.34%
best_chart_position,"2,358",51.34%
chart_observations,"2,358",51.34%
countries_charted,"2,358",51.34%
total_historical_streams,"2,358",51.34%
average_historical_streams,"2,358",51.34%
max_historical_streams,"2,358",51.34%
first_chart_date,"2,358",51.34%
last_chart_date,"2,358",51.34%
geographic_performance_intensity,"2,358",51.34%


#### Interpretation

The missingness review shows that the feature-engineered dataset contains
4,593 records and 92 features. Of these features, 75 contain at least one
missing value, while 17 are complete. Across the entire dataset, approximately
25.00% of all feature values are missing.

The results indicate that a substantial proportion of the missingness is
structured rather than being independently distributed across individual
features.

Historical chart and historical-performance variables show a particularly
clear pattern. Many of these features contain exactly 2,358 missing values,
representing 51.34% of the dataset. This consistent pattern suggests that the
missing values originate primarily from records that could not be associated
with historical chart information. Consequently, these missing values should
not automatically be interpreted as zero historical performance.

A second structured pattern occurs within the artist-listener variables.
Listener-related features such as Listeners, PkListeners, Daily Trend,
listener strength, listener peak ratio and audience trend measures contain
1,204 missing values, representing 26.21% of records. This corresponds to
artists for which listener information is unavailable.

Platform-specific variables display more varied levels of missingness. For
example, some platforms contain relatively limited missing data, while others
such as SiriusXM and Pandora have substantially lower coverage. This suggests
that platform-specific missingness may require different treatment from the
structured historical and artist-listener feature groups.

The review therefore demonstrates that a single missing-value treatment would
not be appropriate for all features. Historical, listener and platform-specific
missingness should be considered separately according to the meaning and
availability of their underlying data sources.

### 8.2 Historical Feature Missingness

Historical chart and performance features contain a substantial and consistent
amount of missing data. The previous missingness review showed that many of
these variables are unavailable for 2,358 records, representing 51.34% of the
dataset.

This subsection investigates whether these missing values belong to the same
records and whether they correspond to tracks without matched historical chart
data.

This distinction is important because the absence of historical information
does not necessarily mean that a track achieved zero streams, zero chart
activity or zero geographic reach. Instead, it may indicate that no matching
historical chart record was available in the integrated dataset.

The missingness pattern is therefore examined before deciding how these
features should be represented during later machine-learning preparation.

In [28]:
# 8.2 Historical Feature Missingness

import pandas as pd
import numpy as np

# Core historical features used to determine historical-data availability
historical_core_features = [
    "best_chart_position",
    "average_chart_position",
    "chart_observations",
    "countries_charted",
    "total_historical_streams",
    "average_historical_streams",
    "max_historical_streams",
    "first_chart_date",
    "last_chart_date"
]

# Keep only columns that exist
available_historical_features = [
    column
    for column in historical_core_features
    if column in df_features.columns
]

print("Historical feature missingness review")
print("-------------------------------------")
print(f"Total records: {len(df_features):,}")
print(
    f"Historical core features requested: "
    f"{len(historical_core_features)}"
)
print(
    f"Historical core features available: "
    f"{len(available_historical_features)}"
)


# ---------------------------------------------------------
# 1. Missingness for each historical core feature
# ---------------------------------------------------------

historical_missingness = pd.DataFrame({
    "Missing Values":
        df_features[
            available_historical_features
        ].isna().sum(),

    "Missing Percentage":
        df_features[
            available_historical_features
        ].isna().mean() * 100
})

print("\nHistorical core feature missingness")
print("-----------------------------------")

display(
    historical_missingness.style.format({
        "Missing Values": "{:,.0f}",
        "Missing Percentage": "{:.2f}%"
    })
)


# ---------------------------------------------------------
# 2. Determine whether missingness occurs on the same rows
# ---------------------------------------------------------

historical_missing_matrix = (
    df_features[
        available_historical_features
    ].isna()
)

all_historical_missing = (
    historical_missing_matrix.all(axis=1)
)

all_historical_available = (
    ~historical_missing_matrix.any(axis=1)
)

partial_historical_missing = (
    ~all_historical_missing
    & ~all_historical_available
)

print("\nHistorical missingness pattern")
print("------------------------------")

print(
    f"Records with all historical features missing: "
    f"{all_historical_missing.sum():,} "
    f"({all_historical_missing.mean() * 100:.2f}%)"
)

print(
    f"Records with all historical features available: "
    f"{all_historical_available.sum():,} "
    f"({all_historical_available.mean() * 100:.2f}%)"
)

print(
    f"Records with partial historical missingness: "
    f"{partial_historical_missing.sum():,} "
    f"({partial_historical_missing.mean() * 100:.2f}%)"
)


# ---------------------------------------------------------
# 3. Compare against existing historical-data indicator
# ---------------------------------------------------------

if "has_historical_chart_data" in df_features.columns:

    expected_historical_available = (
        df_features[
            "has_historical_chart_data"
        ] == 1
    )

    indicator_mismatches = (
        expected_historical_available
        != all_historical_available
    ).sum()

    print("\nHistorical indicator consistency")
    print("--------------------------------")

    print(
        "Records marked as having historical data: "
        f"{expected_historical_available.sum():,}"
    )

    print(
        "Records with complete historical core data: "
        f"{all_historical_available.sum():,}"
    )

    print(
        f"Indicator mismatches: "
        f"{indicator_mismatches:,}"
    )


# ---------------------------------------------------------
# 4. Create temporary review classification
# ---------------------------------------------------------

historical_missingness_status = pd.Series(
    np.select(
        [
            all_historical_available,
            all_historical_missing,
            partial_historical_missing
        ],
        [
            "Historical data available",
            "Historical data unavailable",
            "Partial historical data"
        ],
        default="Unknown"
    ),
    index=df_features.index,
    name="Historical Data Status"
)

print("\nHistorical data availability")
print("----------------------------")

historical_status_summary = (
    historical_missingness_status
    .value_counts()
    .to_frame("Records")
)

historical_status_summary["Percentage"] = (
    historical_status_summary["Records"]
    / len(df_features)
) * 100

display(
    historical_status_summary.style.format({
        "Records": "{:,.0f}",
        "Percentage": "{:.2f}%"
    })
)

Historical feature missingness review
-------------------------------------
Total records: 4,593
Historical core features requested: 9
Historical core features available: 9

Historical core feature missingness
-----------------------------------


,Missing Values,Missing Percentage
best_chart_position,"2,358",51.34%
average_chart_position,"2,358",51.34%
chart_observations,"2,358",51.34%
countries_charted,"2,358",51.34%
total_historical_streams,"2,358",51.34%
average_historical_streams,"2,358",51.34%
max_historical_streams,"2,358",51.34%
first_chart_date,"2,358",51.34%
last_chart_date,"2,358",51.34%



Historical missingness pattern
------------------------------
Records with all historical features missing: 2,358 (51.34%)
Records with all historical features available: 2,235 (48.66%)
Records with partial historical missingness: 0 (0.00%)

Historical indicator consistency
--------------------------------
Records marked as having historical data: 2,235
Records with complete historical core data: 2,235
Indicator mismatches: 0

Historical data availability
----------------------------


,Records,Percentage
Historical Data Status,,
Historical data unavailable,"2,358",51.34%
Historical data available,"2,235",48.66%


#### Interpretation

The historical missingness analysis confirms a clear and consistent pattern
across the historical feature group. All nine core historical variables contain
exactly 2,358 missing observations, representing 51.34% of the dataset.

The row-level analysis shows that 2,358 records have all historical core
features missing, while 2,235 records have complete historical information.
No records contain partial historical missingness. Historical information is
therefore available as a complete group rather than being independently
missing across individual historical variables.

This pattern is also fully consistent with the existing
`has_historical_chart_data` indicator. The 2,235 records identified as having
historical chart data are exactly the same records with complete historical
core features, producing zero indicator mismatches.

These results provide strong evidence that the historical missing values
represent unavailable historical chart information rather than confirmed zero
historical performance. Replacing these missing values directly with zero
could incorrectly imply that a track was historically observed but achieved
no streams, chart activity or geographic reach.

The missing historical values should therefore retain their distinction from
observed zero values. The existing historical-data indicator can preserve this
availability information for later machine-learning preparation and modelling.

### 8.3 Artist Listener Feature Missingness

Artist-listener variables also display a consistent missingness pattern. The
overall missingness review showed that listener-related variables are
unavailable for 1,204 records, representing 26.21% of the dataset.

This subsection investigates whether these missing values occur across the
same records and whether they represent artists for which listener information
was unavailable during data integration.

Understanding this distinction is important because missing listener
information should not automatically be interpreted as zero listeners, zero
audience growth or zero audience strength. Instead, the missing values may
represent the absence of a successful match with the artist-listener dataset.

The listener feature group is therefore examined as a complete data-availability
pattern before any missing-value treatment is applied.

In [29]:
# 8.3 Artist Listener Feature Missingness

import pandas as pd
import numpy as np

# Core listener variables from the integrated listener dataset
listener_core_features = [
    "Listeners",
    "PkListeners",
    "Daily Trend",
    "Peak"
]

# Engineered listener features created in Section 6
listener_engineered_features = [
    "listener_strength_score",
    "listener_peak_ratio",
    "listener_daily_growth_rate",
    "audience_below_peak",
    "audience_trend_direction"
]

# Combine the listener feature groups
listener_features = (
    listener_core_features
    + listener_engineered_features
)

# Keep only features that actually exist
available_listener_features = [
    column
    for column in listener_features
    if column in df_features.columns
]

print("Artist listener feature missingness review")
print("------------------------------------------")
print(f"Total records: {len(df_features):,}")
print(
    f"Listener features requested: "
    f"{len(listener_features)}"
)
print(
    f"Listener features available: "
    f"{len(available_listener_features)}"
)


# ---------------------------------------------------------
# 1. Missingness for each listener feature
# ---------------------------------------------------------

listener_missingness = pd.DataFrame({
    "Missing Values":
        df_features[
            available_listener_features
        ].isna().sum(),

    "Missing Percentage":
        df_features[
            available_listener_features
        ].isna().mean() * 100
})

print("\nListener feature missingness")
print("----------------------------")

display(
    listener_missingness.style.format({
        "Missing Values": "{:,.0f}",
        "Missing Percentage": "{:.2f}%"
    })
)


# ---------------------------------------------------------
# 2. Determine whether missingness occurs on the same rows
# ---------------------------------------------------------

listener_missing_matrix = (
    df_features[
        available_listener_features
    ].isna()
)

all_listener_missing = (
    listener_missing_matrix.all(axis=1)
)

all_listener_available = (
    ~listener_missing_matrix.any(axis=1)
)

partial_listener_missing = (
    ~all_listener_missing
    & ~all_listener_available
)

print("\nListener missingness pattern")
print("----------------------------")

print(
    f"Records with all listener features missing: "
    f"{all_listener_missing.sum():,} "
    f"({all_listener_missing.mean() * 100:.2f}%)"
)

print(
    f"Records with all listener features available: "
    f"{all_listener_available.sum():,} "
    f"({all_listener_available.mean() * 100:.2f}%)"
)

print(
    f"Records with partial listener missingness: "
    f"{partial_listener_missing.sum():,} "
    f"({partial_listener_missing.mean() * 100:.2f}%)"
)


# ---------------------------------------------------------
# 3. Check consistency using the main listener variable
# ---------------------------------------------------------

if "Listeners" in df_features.columns:

    listener_data_available = (
        df_features["Listeners"].notna()
    )

    availability_mismatches = (
        listener_data_available
        != all_listener_available
    ).sum()

    print("\nListener availability consistency")
    print("---------------------------------")

    print(
        f"Records with Listeners available: "
        f"{listener_data_available.sum():,}"
    )

    print(
        f"Records with complete listener feature data: "
        f"{all_listener_available.sum():,}"
    )

    print(
        f"Availability mismatches: "
        f"{availability_mismatches:,}"
    )


# ---------------------------------------------------------
# 4. Temporary listener availability classification
# ---------------------------------------------------------

listener_missingness_status = pd.Series(
    np.select(
        [
            all_listener_available,
            all_listener_missing,
            partial_listener_missing
        ],
        [
            "Listener data available",
            "Listener data unavailable",
            "Partial listener data"
        ],
        default="Unknown"
    ),
    index=df_features.index,
    name="Listener Data Status"
)

print("\nListener data availability")
print("--------------------------")

listener_status_summary = (
    listener_missingness_status
    .value_counts()
    .to_frame("Records")
)

listener_status_summary["Percentage"] = (
    listener_status_summary["Records"]
    / len(df_features)
) * 100

display(
    listener_status_summary.style.format({
        "Records": "{:,.0f}",
        "Percentage": "{:.2f}%"
    })
)

Artist listener feature missingness review
------------------------------------------
Total records: 4,593
Listener features requested: 9
Listener features available: 9

Listener feature missingness
----------------------------


,Missing Values,Missing Percentage
Listeners,"1,204",26.21%
PkListeners,"1,204",26.21%
Daily Trend,"1,204",26.21%
Peak,"1,204",26.21%
listener_strength_score,"1,204",26.21%
listener_peak_ratio,"1,204",26.21%
listener_daily_growth_rate,"1,204",26.21%
audience_below_peak,"1,204",26.21%
audience_trend_direction,"1,204",26.21%



Listener missingness pattern
----------------------------
Records with all listener features missing: 1,204 (26.21%)
Records with all listener features available: 3,389 (73.79%)
Records with partial listener missingness: 0 (0.00%)

Listener availability consistency
---------------------------------
Records with Listeners available: 3,389
Records with complete listener feature data: 3,389
Availability mismatches: 0

Listener data availability
--------------------------


,Records,Percentage
Listener Data Status,,
Listener data available,"3,389",73.79%
Listener data unavailable,"1,204",26.21%


#### Interpretation

The artist-listener missingness analysis confirms a consistent availability
pattern across all nine listener-related features. Each feature contains
exactly 1,204 missing observations, representing 26.21% of the dataset.

At the record level, 3,389 observations (73.79%) contain the complete set of
listener features, while 1,204 observations (26.21%) contain no listener
information. No records contain partial listener missingness, demonstrating
that the listener variables are available or unavailable as a complete group.

The availability pattern is also fully consistent with the original
`Listeners` variable. All 3,389 records with a listener value contain the
complete listener feature set, and the consistency check produced zero
mismatches.

This indicates that the missing listener-derived values primarily represent
unavailable artist-listener information rather than independently missing
measurements. Consequently, these missing values should not automatically be
interpreted as zero listeners, zero audience strength or zero audience growth.

Preserving the distinction between unavailable listener information and
observed audience values is therefore important for later machine-learning
development. An explicit listener-data availability indicator can provide this
information to models while the underlying missing values are handled using an
appropriate model-preparation strategy.

### 8.4 Missingness Indicator Features

The previous analyses identified structured missingness within the historical
chart and artist-listener feature groups. These missing values represent data
availability rather than confirmed zero performance.

Binary availability indicators can preserve this information for later
machine-learning models. An indicator allows a model to distinguish between
a record with observed performance information and a record for which the
corresponding source data is unavailable.

The existing `has_historical_chart_data` feature already represents historical
chart-data availability and is therefore retained rather than duplicated.

A corresponding `has_listener_data` feature is created for the artist-listener
feature group. Both indicators are then validated against the missingness
patterns identified in the previous subsections.

The underlying missing numerical values are not imputed at this stage. Their
final treatment can be selected separately for each PMIP machine-learning model
during model preparation.

In [30]:
# 8.4 Missingness Indicator Features

import pandas as pd
import numpy as np

print("Missingness indicator feature engineering")
print("-----------------------------------------")
print(f"Total records: {len(df_features):,}")


# ---------------------------------------------------------
# 1. Historical data availability indicator
# ---------------------------------------------------------

if "has_historical_chart_data" in df_features.columns:

    # Ensure existing indicator has a consistent integer type
    df_features["has_historical_chart_data"] = (
        df_features["has_historical_chart_data"]
        .astype("int64")
    )

    print(
        "\nExisting historical availability indicator retained: "
        "has_historical_chart_data"
    )

else:

    # Fallback only if the indicator does not already exist
    df_features["has_historical_chart_data"] = (
        df_features["chart_observations"]
        .notna()
        .astype("int64")
    )

    print(
        "\nHistorical availability indicator created: "
        "has_historical_chart_data"
    )


# ---------------------------------------------------------
# 2. Artist listener availability indicator
# ---------------------------------------------------------

df_features["has_listener_data"] = (
    df_features["Listeners"]
    .notna()
    .astype("int64")
)

print(
    "Listener availability indicator created: "
    "has_listener_data"
)


# ---------------------------------------------------------
# 3. Validate indicator distributions
# ---------------------------------------------------------

indicator_features = [
    "has_historical_chart_data",
    "has_listener_data"
]

indicator_summary = []

for column in indicator_features:

    available_count = (
        df_features[column] == 1
    ).sum()

    unavailable_count = (
        df_features[column] == 0
    ).sum()

    indicator_summary.append({
        "Indicator": column,
        "Available Records": available_count,
        "Available Percentage":
            (available_count / len(df_features)) * 100,
        "Unavailable Records": unavailable_count,
        "Unavailable Percentage":
            (unavailable_count / len(df_features)) * 100
    })

indicator_summary = pd.DataFrame(
    indicator_summary
)

print("\nMissingness indicator summary")
print("-----------------------------")

display(
    indicator_summary.style.format({
        "Available Records": "{:,.0f}",
        "Available Percentage": "{:.2f}%",
        "Unavailable Records": "{:,.0f}",
        "Unavailable Percentage": "{:.2f}%"
    })
)


# ---------------------------------------------------------
# 4. Validate indicators against actual missingness
# ---------------------------------------------------------

historical_expected = (
    df_features["chart_observations"].notna()
    .astype("int64")
)

listener_expected = (
    df_features["Listeners"].notna()
    .astype("int64")
)

historical_indicator_mismatches = (
    df_features["has_historical_chart_data"]
    != historical_expected
).sum()

listener_indicator_mismatches = (
    df_features["has_listener_data"]
    != listener_expected
).sum()

print("\nIndicator consistency checks")
print("----------------------------")

print(
    f"Historical indicator mismatches: "
    f"{historical_indicator_mismatches:,}"
)

print(
    f"Listener indicator mismatches: "
    f"{listener_indicator_mismatches:,}"
)


# ---------------------------------------------------------
# 5. Validate binary values and missingness
# ---------------------------------------------------------

print("\nIndicator integrity checks")
print("--------------------------")

for column in indicator_features:

    unique_values = sorted(
        df_features[column]
        .dropna()
        .unique()
        .tolist()
    )

    missing_values = (
        df_features[column]
        .isna()
        .sum()
    )

    invalid_values = (
        ~df_features[column]
        .isin([0, 1])
    ).sum()

    print(f"\n{column}")
    print(f"  Unique values: {unique_values}")
    print(f"  Missing values: {missing_values:,}")
    print(f"  Invalid values: {invalid_values:,}")


# ---------------------------------------------------------
# 6. Combined data-availability patterns
# ---------------------------------------------------------

availability_patterns = (
    df_features[
        [
            "has_historical_chart_data",
            "has_listener_data"
        ]
    ]
    .value_counts()
    .reset_index(name="Records")
)

availability_patterns["Percentage"] = (
    availability_patterns["Records"]
    / len(df_features)
) * 100

availability_patterns[
    "Historical Data"
] = availability_patterns[
    "has_historical_chart_data"
].map({
    1: "Available",
    0: "Unavailable"
})

availability_patterns[
    "Listener Data"
] = availability_patterns[
    "has_listener_data"
].map({
    1: "Available",
    0: "Unavailable"
})

print("\nCombined data availability patterns")
print("-----------------------------------")

display(
    availability_patterns[
        [
            "Historical Data",
            "Listener Data",
            "Records",
            "Percentage"
        ]
    ].style.format({
        "Records": "{:,.0f}",
        "Percentage": "{:.2f}%"
    })
)

Missingness indicator feature engineering
-----------------------------------------
Total records: 4,593

Existing historical availability indicator retained: has_historical_chart_data
Listener availability indicator created: has_listener_data

Missingness indicator summary
-----------------------------


,Indicator,Available Records,Available Percentage,Unavailable Records,Unavailable Percentage
0,has_historical_chart_data,"2,235",48.66%,"2,358",51.34%
1,has_listener_data,"3,389",73.79%,"1,204",26.21%



Indicator consistency checks
----------------------------
Historical indicator mismatches: 0
Listener indicator mismatches: 0

Indicator integrity checks
--------------------------

has_historical_chart_data
  Unique values: [0, 1]
  Missing values: 0
  Invalid values: 0

has_listener_data
  Unique values: [0, 1]
  Missing values: 0
  Invalid values: 0

Combined data availability patterns
-----------------------------------


,Historical Data,Listener Data,Records,Percentage
0,Available,Available,"2,031",44.22%
1,Unavailable,Available,"1,358",29.57%
2,Unavailable,Unavailable,"1,000",21.77%
3,Available,Unavailable,204,4.44%


#### Interpretation

The missingness indicator features were successfully created and validated.
The existing `has_historical_chart_data` indicator was retained, while a new
`has_listener_data` indicator was created to represent artist-listener data
availability.

Historical chart information is available for 2,235 records (48.66%) and
unavailable for 2,358 records (51.34%). Artist-listener information is
available for 3,389 records (73.79%) and unavailable for 1,204 records
(26.21%).

Both indicators passed all consistency and integrity checks. Neither contains
missing or invalid values, and both use the expected binary values of 0 and 1.
The indicators also produced zero mismatches when compared with the underlying
historical and listener data availability patterns.

The combined availability analysis shows that 2,031 records (44.22%) contain
both historical and listener information. A further 1,358 records (29.57%)
contain listener information without historical chart data, while 204 records
(4.44%) contain historical information without listener data. The remaining
1,000 records (21.77%) contain neither historical nor listener information.

These patterns demonstrate that data availability varies substantially across
the integrated dataset. Explicit availability indicators therefore preserve
useful information that would otherwise be lost if missing values were simply
replaced with zero.

The underlying missing historical and listener values remain unchanged at this
stage. Their final imputation or model-specific treatment can be performed
during machine-learning preparation while retaining the availability indicators
to distinguish missing information from observed values.

## 9. Redundancy and Multicollinearity Review

### Purpose

The purpose of this section is to identify numerical features that contain
highly similar or overlapping information before the PMIP dataset is prepared
for machine-learning development.

Feature engineering has introduced several derived and transformed variables
alongside their original features. Some of these variables may therefore be
strongly correlated because they measure similar aspects of streaming
performance, chart activity, geographic reach or artist audience behaviour.

Including multiple features that provide almost identical information can
increase redundancy within the dataset and may cause multicollinearity for
models that are sensitive to relationships between predictor variables. It can
also make model interpretation more difficult.

This section therefore examines correlations between numerical features,
identifies potentially redundant feature relationships, and records decisions
about which features should be retained for later model development.

Features are not automatically removed solely because they are highly
correlated. Their meaning, transformation history and relevance to different
PMIP intelligence tasks are considered before retention decisions are made.

The section consists of:

- **9.1 Highly Correlated Features** — identify pairs of numerical features with
  strong linear relationships.
- **9.2 Redundant Feature Identification** — examine highly correlated pairs and
  identify features that may represent duplicated or overlapping information.
- **9.3 Feature Retention Decisions** — document which features should be
  retained, excluded or deferred for model-specific evaluation.

### 9.1 Highly Correlated Features

Highly correlated numerical features may provide very similar information to
machine-learning models. This is particularly relevant after feature
engineering because original variables, derived features and transformed
versions may naturally exhibit strong relationships.

A Pearson correlation matrix is calculated across the numerical features in
the current dataset. Feature pairs with an absolute correlation coefficient of
0.90 or greater are flagged for further review.

The threshold is used as a screening criterion rather than an automatic feature
removal rule. A strong correlation does not necessarily mean that one feature
should be removed, particularly when the variables represent different
business concepts or may be useful for different PMIP models.

In [31]:
# 9.1 Highly Correlated Features

import pandas as pd
import numpy as np

# Select numerical features only
numerical_features = df_features.select_dtypes(
    include=[np.number]
)

print("Highly correlated feature review")
print("--------------------------------")
print(f"Total dataset records: {len(df_features):,}")
print(f"Total dataset features: {len(df_features.columns)}")
print(f"Numerical features reviewed: {len(numerical_features.columns)}")


# ---------------------------------------------------------
# 1. Calculate Pearson correlation matrix
# ---------------------------------------------------------

correlation_matrix = numerical_features.corr(
    method="pearson"
)

print(
    f"Correlation matrix dimensions: "
    f"{correlation_matrix.shape[0]} × "
    f"{correlation_matrix.shape[1]}"
)


# ---------------------------------------------------------
# 2. Extract unique feature pairs
# ---------------------------------------------------------

correlation_pairs = []

columns = correlation_matrix.columns

for i in range(len(columns)):

    for j in range(i + 1, len(columns)):

        feature_1 = columns[i]
        feature_2 = columns[j]

        correlation = correlation_matrix.iloc[i, j]

        # Ignore undefined correlations
        if pd.isna(correlation):
            continue

        correlation_pairs.append({
            "Feature 1": feature_1,
            "Feature 2": feature_2,
            "Correlation": correlation,
            "Absolute Correlation": abs(correlation)
        })


correlation_pairs_df = pd.DataFrame(
    correlation_pairs
)

correlation_pairs_df = (
    correlation_pairs_df
    .sort_values(
        "Absolute Correlation",
        ascending=False
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 3. Identify highly correlated feature pairs
# ---------------------------------------------------------

correlation_threshold = 0.90

highly_correlated_pairs = (
    correlation_pairs_df[
        correlation_pairs_df[
            "Absolute Correlation"
        ] >= correlation_threshold
    ]
    .reset_index(drop=True)
)


print("\nCorrelation screening")
print("---------------------")
print(
    f"Correlation threshold: "
    f"|r| >= {correlation_threshold:.2f}"
)

print(
    f"Total unique numerical feature pairs: "
    f"{len(correlation_pairs_df):,}"
)

print(
    f"Highly correlated feature pairs: "
    f"{len(highly_correlated_pairs):,}"
)


# ---------------------------------------------------------
# 4. Correlation strength summary
# ---------------------------------------------------------

very_high_pairs = (
    correlation_pairs_df[
        "Absolute Correlation"
    ] >= 0.95
).shape[0]

high_pairs = (
    (
        correlation_pairs_df[
            "Absolute Correlation"
        ] >= 0.90
    )
    &
    (
        correlation_pairs_df[
            "Absolute Correlation"
        ] < 0.95
    )
).sum()


print("\nHigh-correlation strength")
print("-------------------------")

print(
    f"Very high correlations (|r| >= 0.95): "
    f"{very_high_pairs:,}"
)

print(
    f"High correlations (0.90 <= |r| < 0.95): "
    f"{high_pairs:,}"
)


# ---------------------------------------------------------
# 5. Display highly correlated pairs
# ---------------------------------------------------------

print("\nHighly correlated feature pairs")
print("-------------------------------")

if len(highly_correlated_pairs) == 0:

    print(
        "No numerical feature pairs exceeded "
        "the selected correlation threshold."
    )

else:

    display(
        highly_correlated_pairs.style.format({
            "Correlation": "{:.4f}",
            "Absolute Correlation": "{:.4f}"
        })
    )


# ---------------------------------------------------------
# 6. Features appearing most often in high-correlation pairs
# ---------------------------------------------------------

if len(highly_correlated_pairs) > 0:

    correlated_feature_frequency = pd.concat([
        highly_correlated_pairs["Feature 1"],
        highly_correlated_pairs["Feature 2"]
    ]).value_counts()

    correlated_feature_frequency = (
        correlated_feature_frequency
        .rename_axis("Feature")
        .reset_index(name="High-Correlation Relationships")
    )

    print(
        "\nFeatures appearing most frequently "
        "in high-correlation relationships"
    )
    print(
        "-------------------------------------"
        "---------------------------"
    )

    display(
        correlated_feature_frequency
    )

Highly correlated feature review
--------------------------------
Total dataset records: 4,593
Total dataset features: 93
Numerical features reviewed: 83
Correlation matrix dimensions: 83 × 83

Correlation screening
---------------------
Correlation threshold: |r| >= 0.90
Total unique numerical feature pairs: 3,368
Highly correlated feature pairs: 27

High-correlation strength
-------------------------
Very high correlations (|r| >= 0.95): 3,368
High correlations (0.90 <= |r| < 0.95): 12

Highly correlated feature pairs
-------------------------------


,Feature 1,Feature 2,Correlation,Absolute Correlation
0,listener_peak_ratio,audience_below_peak,-1.0000,1.0000
1,chart_longevity_days,chart_longevity_years,1.0000,1.0000
2,international_reach_score,market_coverage_score,1.0000,1.0000
3,countries_charted,international_reach_score,1.0000,1.0000
4,countries_charted,market_coverage_score,1.0000,1.0000
5,chart_activity_factor,chart_observations_log,1.0000,1.0000
6,Listeners,listener_strength_score,1.0000,1.0000
7,average_historical_streams_log,historical_streaming_intensity_log,1.0000,1.0000
8,average_historical_streams,historical_streaming_intensity,1.0000,1.0000
9,Listeners_log,PkListeners_log,0.9930,0.9930



Features appearing most frequently in high-correlation relationships
----------------------------------------------------------------


,Feature,High-Correlation Relationships
0,Listeners,4
1,Listeners_log,4
2,PkListeners,4
3,listener_strength_score,4
4,PkListeners_log,4
5,chart_longevity_days,2
6,international_reach_score,2
7,countries_charted,2
8,best_chart_strength,2
9,chart_longevity_years,2


#### Interpretation

The correlation review examined 83 numerical features and identified 27
feature pairs with an absolute Pearson correlation coefficient of at least
0.90. These relationships provide evidence of several potentially redundant
or strongly overlapping variables within the engineered dataset.

Several perfect or near-perfect correlations arise from features that are
mathematically derived from one another. For example, listener peak ratio and
audience below peak have a perfect negative correlation, while chart longevity
in days and years has a perfect positive correlation. Listener strength is also
perfectly correlated with the original listener count because it represents a
normalised version of the same underlying quantity.

Geographic variables show another clear redundancy pattern. Countries charted,
market coverage score and international reach score are perfectly correlated.
This indicates that these variables encode essentially the same geographic
coverage information using different scales or feature names.

Strong relationships also occur between variables that are conceptually
related but not necessarily redundant. TikTok Likes and TikTok Views, for
example, are extremely highly correlated, but they measure different forms of
audience engagement. Similarly, current and peak listener counts are strongly
correlated but retain different meanings for audience-performance analysis.

The results therefore demonstrate that correlation alone should not determine
feature removal. Perfect mathematical transformations and duplicate
representations are strong candidates for redundancy reduction, while highly
correlated features representing distinct business concepts may still provide
useful information for different PMIP intelligence models.

These relationships will be examined further during redundant feature
identification before any feature-retention decisions are made.

### 9.2 Redundant Feature Identification

The correlation analysis identified several feature pairs with very strong
relationships. However, high correlation alone does not necessarily indicate
that one of the variables should be removed.

This subsection examines the highly correlated relationships in greater detail
and distinguishes between different forms of potential redundancy.

Particular attention is given to features that are direct mathematical
transformations of one another, alternative representations of the same
underlying measurement, or engineered variables derived directly from an
existing feature. Strongly correlated variables that represent different
business concepts are treated separately because they may still provide useful
information for different PMIP machine-learning tasks.

Potentially redundant features are identified for review rather than removed
immediately. Final retention decisions are made in Section 9.3.

In [32]:
# 9.2 Redundant Feature Identification

import pandas as pd
import numpy as np

print("Redundant feature identification")
print("--------------------------------")


# ---------------------------------------------------------
# 1. Define known feature relationships
# ---------------------------------------------------------

# Features that are effectively alternative mathematical
# representations of the same underlying information.
known_redundant_groups = {
    "Chart longevity": [
        "chart_longevity_days",
        "chart_longevity_years"
    ],

    "Geographic coverage": [
        "countries_charted",
        "market_coverage_score",
        "international_reach_score"
    ],

    "Listener level": [
        "Listeners",
        "listener_strength_score"
    ],

    "Audience distance from peak": [
        "listener_peak_ratio",
        "audience_below_peak"
    ],

    "Historical streaming intensity": [
        "average_historical_streams",
        "historical_streaming_intensity"
    ],

    "Historical streaming intensity log": [
        "average_historical_streams_log",
        "historical_streaming_intensity_log"
    ],

    "Chart activity": [
        "chart_activity_factor",
        "chart_observations_log"
    ]
}


# ---------------------------------------------------------
# 2. Create a feature-to-group lookup
# ---------------------------------------------------------

feature_group_lookup = {}

for group_name, features in known_redundant_groups.items():
    for feature in features:
        if feature in df_features.columns:
            feature_group_lookup[feature] = group_name


# ---------------------------------------------------------
# 3. Classify highly correlated feature pairs
# ---------------------------------------------------------

redundancy_review = highly_correlated_pairs.copy()


def classify_relationship(row):

    feature_1 = row["Feature 1"]
    feature_2 = row["Feature 2"]
    absolute_correlation = row["Absolute Correlation"]

    group_1 = feature_group_lookup.get(feature_1)
    group_2 = feature_group_lookup.get(feature_2)

    # Same known mathematical/derived feature family
    if (
        group_1 is not None
        and group_1 == group_2
    ):
        return "Likely Redundant"

    # Original feature compared with its log transformation
    if (
        feature_1 == f"{feature_2}_log"
        or feature_2 == f"{feature_1}_log"
    ):
        return "Alternative Representation"

    # Extremely strong relationship, but not a known
    # direct transformation
    if absolute_correlation >= 0.98:
        return "Strongly Related"

    return "Review Separately"


redundancy_review["Relationship Classification"] = (
    redundancy_review.apply(
        classify_relationship,
        axis=1
    )
)


# ---------------------------------------------------------
# 4. Add known feature-family information
# ---------------------------------------------------------

def identify_feature_family(row):

    feature_1 = row["Feature 1"]
    feature_2 = row["Feature 2"]

    group_1 = feature_group_lookup.get(feature_1)
    group_2 = feature_group_lookup.get(feature_2)

    if (
        group_1 is not None
        and group_1 == group_2
    ):
        return group_1

    if (
        feature_1 == f"{feature_2}_log"
        or feature_2 == f"{feature_1}_log"
    ):
        return "Original / Log Transformation"

    return "Distinct Feature Concepts"


redundancy_review["Feature Relationship"] = (
    redundancy_review.apply(
        identify_feature_family,
        axis=1
    )
)


# ---------------------------------------------------------
# 5. Display classification summary
# ---------------------------------------------------------

classification_summary = (
    redundancy_review[
        "Relationship Classification"
    ]
    .value_counts()
    .rename_axis("Classification")
    .reset_index(name="Feature Pairs")
)

classification_summary["Percentage"] = (
    classification_summary["Feature Pairs"]
    / len(redundancy_review)
) * 100


print("\nRedundancy classification summary")
print("---------------------------------")

display(
    classification_summary.style.format({
        "Feature Pairs": "{:,.0f}",
        "Percentage": "{:.2f}%"
    })
)


# ---------------------------------------------------------
# 6. Display detailed redundancy review
# ---------------------------------------------------------

print("\nDetailed redundancy review")
print("--------------------------")

display(
    redundancy_review[
        [
            "Feature 1",
            "Feature 2",
            "Correlation",
            "Absolute Correlation",
            "Feature Relationship",
            "Relationship Classification"
        ]
    ].style.format({
        "Correlation": "{:.4f}",
        "Absolute Correlation": "{:.4f}"
    })
)


# ---------------------------------------------------------
# 7. Display known redundant feature groups
# ---------------------------------------------------------

redundant_group_records = []

for group_name, features in known_redundant_groups.items():

    available_features = [
        feature
        for feature in features
        if feature in df_features.columns
    ]

    if len(available_features) >= 2:

        redundant_group_records.append({
            "Feature Group": group_name,
            "Features": ", ".join(available_features),
            "Number of Features": len(available_features)
        })


redundant_groups_df = pd.DataFrame(
    redundant_group_records
)

print("\nKnown potentially redundant feature groups")
print("------------------------------------------")

display(redundant_groups_df)


# ---------------------------------------------------------
# 8. Identify features involved in likely redundancy
# ---------------------------------------------------------

likely_redundant_pairs = redundancy_review[
    redundancy_review[
        "Relationship Classification"
    ] == "Likely Redundant"
].copy()

potentially_redundant_features = sorted(
    set(
        likely_redundant_pairs["Feature 1"].tolist()
        +
        likely_redundant_pairs["Feature 2"].tolist()
    )
)

print("\nPotentially redundant features")
print("------------------------------")

print(
    f"Features involved in known redundant relationships: "
    f"{len(potentially_redundant_features)}"
)

for feature in potentially_redundant_features:
    print(f"- {feature}")


# ---------------------------------------------------------
# 9. Confirm that no features have been removed
# ---------------------------------------------------------

print("\nDataset status")
print("--------------")

print(
    f"Current dataset features: "
    f"{len(df_features.columns)}"
)

print(
    "Features removed during redundancy review: 0"
)

Redundant feature identification
--------------------------------

Redundancy classification summary
---------------------------------


,Classification,Feature Pairs,Percentage
0,Review Separately,10,37.04%
1,Likely Redundant,9,33.33%
2,Strongly Related,5,18.52%
3,Alternative Representation,3,11.11%



Detailed redundancy review
--------------------------


,Feature 1,Feature 2,Correlation,Absolute Correlation,Feature Relationship,Relationship Classification
0,listener_peak_ratio,audience_below_peak,-1.0000,1.0000,Audience distance from peak,Likely Redundant
1,chart_longevity_days,chart_longevity_years,1.0000,1.0000,Chart longevity,Likely Redundant
2,international_reach_score,market_coverage_score,1.0000,1.0000,Geographic coverage,Likely Redundant
3,countries_charted,international_reach_score,1.0000,1.0000,Geographic coverage,Likely Redundant
4,countries_charted,market_coverage_score,1.0000,1.0000,Geographic coverage,Likely Redundant
5,chart_activity_factor,chart_observations_log,1.0000,1.0000,Chart activity,Likely Redundant
6,Listeners,listener_strength_score,1.0000,1.0000,Listener level,Likely Redundant
7,average_historical_streams_log,historical_streaming_intensity_log,1.0000,1.0000,Historical streaming intensity log,Likely Redundant
8,average_historical_streams,historical_streaming_intensity,1.0000,1.0000,Historical streaming intensity,Likely Redundant
9,Listeners_log,PkListeners_log,0.9930,0.9930,Distinct Feature Concepts,Strongly Related



Known potentially redundant feature groups
------------------------------------------


,Feature Group,Features,Number of Features
0,Chart longevity,"chart_longevity_days, chart_longevity_years",2
1,Geographic coverage,"countries_charted, market_coverage_score, inte...",3
2,Listener level,"Listeners, listener_strength_score",2
3,Audience distance from peak,"listener_peak_ratio, audience_below_peak",2
4,Historical streaming intensity,"average_historical_streams, historical_streami...",2
5,Historical streaming intensity log,"average_historical_streams_log, historical_str...",2
6,Chart activity,"chart_activity_factor, chart_observations_log",2



Potentially redundant features
------------------------------
Features involved in known redundant relationships: 15
- Listeners
- audience_below_peak
- average_historical_streams
- average_historical_streams_log
- chart_activity_factor
- chart_longevity_days
- chart_longevity_years
- chart_observations_log
- countries_charted
- historical_streaming_intensity
- historical_streaming_intensity_log
- international_reach_score
- listener_peak_ratio
- listener_strength_score
- market_coverage_score

Dataset status
--------------
Current dataset features: 93
Features removed during redundancy review: 0


#### Interpretation

The redundancy review classified the 27 highly correlated feature pairs
according to the nature of their relationships. Nine pairs were identified as
likely redundant, three as alternative representations, five as strongly
related but conceptually distinct, and ten as requiring separate consideration.

Several clear redundancy groups were identified. Chart longevity measured in
days and years represents the same underlying duration using different units.
Similarly, `listener_peak_ratio` and `audience_below_peak` describe the same
distance-from-peak relationship from opposite directions.

The geographic feature group shows particularly strong redundancy.
`countries_charted`, `market_coverage_score` and `international_reach_score`
are perfectly correlated, indicating that they represent the same underlying
geographic coverage information using different representations.

Listener magnitude also contains overlapping representations. `Listeners` and
`listener_strength_score` are perfectly correlated because listener strength
is a normalised transformation of the original listener count. Current and
peak listener measures remain strongly related but represent different audience
concepts and therefore should not automatically be treated as duplicates.

Historical streaming variables reveal further overlap.
`average_historical_streams` and `historical_streaming_intensity` are perfectly
correlated, as are their corresponding log-transformed versions. Chart activity
also shows a perfect relationship between `chart_activity_factor` and
`chart_observations_log`.

Other highly correlated relationships, including TikTok Likes and TikTok Views,
represent distinct measures despite their strong statistical association.
These variables therefore require model-specific consideration rather than
automatic removal.

No features were removed during this subsection. The identified redundancy
groups provide the evidence required for the feature-retention decisions in
Section 9.3.

### 9.3 Feature Retention Decisions

The correlation and redundancy analyses identified several groups of features
that represent identical, transformed or strongly overlapping information.

This subsection records explicit feature-retention decisions for these
relationships. The objective is to reduce unnecessary duplication while
preserving variables that remain meaningful for different PMIP intelligence
tasks.

Features that are direct mathematical duplicates or unnecessary alternative
representations are marked for exclusion. Features that provide a clearer,
more interpretable or more suitable transformed representation are retained.

Strongly correlated variables that measure different concepts are not removed
solely because of their correlation. These features are marked as
model-specific so that their usefulness can be evaluated during individual
machine-learning experiments.

The decisions made here define the preferred feature representations without
discarding potentially useful information prematurely.

In [33]:
# 9.3 Feature Retention Decisions

import pandas as pd

print("Feature retention decisions")
print("---------------------------")


# ---------------------------------------------------------
# 1. Define retention decisions
# ---------------------------------------------------------

retention_decisions = [
    {
        "Feature": "chart_longevity_days",
        "Decision": "Retain",
        "Reason":
            "Retained as the primary and directly interpretable "
            "measure of historical chart duration."
    },
    {
        "Feature": "chart_longevity_years",
        "Decision": "Exclude",
        "Reason":
            "Perfectly correlated with chart_longevity_days and "
            "represents the same duration in different units."
    },

    {
        "Feature": "countries_charted",
        "Decision": "Retain",
        "Reason":
            "Retained as the original and directly interpretable "
            "measure of geographic market coverage."
    },
    {
        "Feature": "market_coverage_score",
        "Decision": "Model-Specific",
        "Reason":
            "Normalised geographic coverage representation that "
            "may be useful for models requiring scaled features."
    },
    {
        "Feature": "international_reach_score",
        "Decision": "Exclude",
        "Reason":
            "Perfectly correlated with countries_charted and "
            "market_coverage_score and provides no distinct signal."
    },

    {
        "Feature": "Listeners",
        "Decision": "Model-Specific",
        "Reason":
            "Original listener magnitude remains interpretable, "
            "but its skewed distribution may make transformed "
            "representations more suitable for some models."
    },
    {
        "Feature": "listener_strength_score",
        "Decision": "Retain",
        "Reason":
            "Normalised listener-strength representation designed "
            "for comparison of artist audience magnitude."
    },

    {
        "Feature": "listener_peak_ratio",
        "Decision": "Retain",
        "Reason":
            "Directly represents current audience size relative "
            "to the artist's recorded peak audience."
    },
    {
        "Feature": "audience_below_peak",
        "Decision": "Exclude",
        "Reason":
            "Perfect inverse representation of listener_peak_ratio "
            "and therefore provides duplicate information."
    },

    {
        "Feature": "average_historical_streams",
        "Decision": "Exclude",
        "Reason":
            "Provides the same underlying historical streaming "
            "signal as historical_streaming_intensity."
    },
    {
        "Feature": "historical_streaming_intensity",
        "Decision": "Model-Specific",
        "Reason":
            "Retained conceptually as an interpretable historical "
            "streaming intensity measure, although its log version "
            "may be preferable for modelling."
    },

    {
        "Feature": "average_historical_streams_log",
        "Decision": "Exclude",
        "Reason":
            "Perfectly correlated with the log-transformed "
            "historical streaming intensity feature."
    },
    {
        "Feature": "historical_streaming_intensity_log",
        "Decision": "Retain",
        "Reason":
            "Preferred transformed historical streaming-intensity "
            "representation after skewness reduction."
    },

    {
        "Feature": "chart_activity_factor",
        "Decision": "Retain",
        "Reason":
            "Engineered representation of historical chart "
            "activity designed for model use."
    },
    {
        "Feature": "chart_observations_log",
        "Decision": "Exclude",
        "Reason":
            "Perfectly correlated with chart_activity_factor and "
            "therefore provides duplicate transformed information."
    },

    {
        "Feature": "TikTok Likes",
        "Decision": "Model-Specific",
        "Reason":
            "Highly correlated with TikTok Views but represents "
            "a distinct audience-engagement action."
    },
    {
        "Feature": "TikTok Views",
        "Decision": "Model-Specific",
        "Reason":
            "Highly correlated with TikTok Likes but represents "
            "content exposure rather than direct engagement."
    },

    {
        "Feature": "PkListeners",
        "Decision": "Model-Specific",
        "Reason":
            "Strongly related to current listeners but represents "
            "historical peak audience size and remains conceptually "
            "distinct."
    },

    {
        "Feature": "best_chart_strength",
        "Decision": "Model-Specific",
        "Reason":
            "Strongly correlated with peak chart performance but "
            "represents a normalised measure of best chart position."
    },
    {
        "Feature": "peak_chart_performance",
        "Decision": "Model-Specific",
        "Reason":
            "Composite peak-performance feature that may provide "
            "useful predictive information for specific PMIP models."
    },

    {
        "Feature": "reached_number_one",
        "Decision": "Retain",
        "Reason":
            "Binary milestone indicator remains interpretable and "
            "may capture information useful for classification or "
            "artist-performance analysis."
    }
]


retention_decisions_df = pd.DataFrame(
    retention_decisions
)


# ---------------------------------------------------------
# 2. Keep decisions only for features that exist
# ---------------------------------------------------------

retention_decisions_df["Exists in Dataset"] = (
    retention_decisions_df["Feature"]
    .isin(df_features.columns)
)

print(
    f"Retention decisions documented: "
    f"{len(retention_decisions_df):,}"
)

print(
    "Features found in current dataset: "
    f"{retention_decisions_df['Exists in Dataset'].sum():,}"
)


# ---------------------------------------------------------
# 3. Decision summary
# ---------------------------------------------------------

decision_summary = (
    retention_decisions_df[
        retention_decisions_df["Exists in Dataset"]
    ]["Decision"]
    .value_counts()
    .rename_axis("Decision")
    .reset_index(name="Features")
)

decision_summary["Percentage"] = (
    decision_summary["Features"]
    / decision_summary["Features"].sum()
) * 100


print("\nRetention decision summary")
print("--------------------------")

display(
    decision_summary.style.format({
        "Features": "{:,.0f}",
        "Percentage": "{:.2f}%"
    })
)


# ---------------------------------------------------------
# 4. Detailed decision table
# ---------------------------------------------------------

print("\nDetailed feature retention decisions")
print("------------------------------------")

display(
    retention_decisions_df[
        [
            "Feature",
            "Decision",
            "Reason",
            "Exists in Dataset"
        ]
    ]
)


# ---------------------------------------------------------
# 5. Identify features marked for exclusion
# ---------------------------------------------------------

features_to_exclude = (
    retention_decisions_df[
        (
            retention_decisions_df["Decision"]
            == "Exclude"
        )
        &
        (
            retention_decisions_df[
                "Exists in Dataset"
            ]
        )
    ]["Feature"]
    .tolist()
)

print("\nFeatures marked for exclusion")
print("-----------------------------")

print(
    f"Number of features marked for exclusion: "
    f"{len(features_to_exclude)}"
)

for feature in features_to_exclude:
    print(f"- {feature}")


# ---------------------------------------------------------
# 6. Identify retained and model-specific features
# ---------------------------------------------------------

features_to_retain = (
    retention_decisions_df[
        (
            retention_decisions_df["Decision"]
            == "Retain"
        )
        &
        (
            retention_decisions_df[
                "Exists in Dataset"
            ]
        )
    ]["Feature"]
    .tolist()
)

model_specific_features = (
    retention_decisions_df[
        (
            retention_decisions_df["Decision"]
            == "Model-Specific"
        )
        &
        (
            retention_decisions_df[
                "Exists in Dataset"
            ]
        )
    ]["Feature"]
    .tolist()
)


print("\nPreferred retained features")
print("---------------------------")

for feature in features_to_retain:
    print(f"- {feature}")


print("\nModel-specific features")
print("-----------------------")

for feature in model_specific_features:
    print(f"- {feature}")


# ---------------------------------------------------------
# 7. Validate decisions
# ---------------------------------------------------------

duplicate_decisions = (
    retention_decisions_df["Feature"]
    .duplicated()
    .sum()
)

invalid_decisions = (
    ~retention_decisions_df["Decision"]
    .isin([
        "Retain",
        "Exclude",
        "Model-Specific"
    ])
).sum()


print("\nDecision validation")
print("-------------------")

print(
    f"Duplicate feature decisions: "
    f"{duplicate_decisions}"
)

print(
    f"Invalid decision labels: "
    f"{invalid_decisions}"
)


# ---------------------------------------------------------
# 8. Dataset status
# ---------------------------------------------------------

print("\nDataset status")
print("--------------")

print(
    f"Current dataset features: "
    f"{len(df_features.columns)}"
)

print(
    f"Features marked for future exclusion: "
    f"{len(features_to_exclude)}"
)

print(
    "Features physically removed at this stage: 0"
)

Feature retention decisions
---------------------------
Retention decisions documented: 21
Features found in current dataset: 21

Retention decision summary
--------------------------


,Decision,Features,Percentage
0,Model-Specific,8,38.10%
1,Retain,7,33.33%
2,Exclude,6,28.57%



Detailed feature retention decisions
------------------------------------


,Feature,Decision,Reason,Exists in Dataset
0,chart_longevity_days,Retain,Retained as the primary and directly interpret...,True
1,chart_longevity_years,Exclude,Perfectly correlated with chart_longevity_days...,True
2,countries_charted,Retain,Retained as the original and directly interpre...,True
3,market_coverage_score,Model-Specific,Normalised geographic coverage representation ...,True
4,international_reach_score,Exclude,Perfectly correlated with countries_charted an...,True
5,Listeners,Model-Specific,Original listener magnitude remains interpreta...,True
6,listener_strength_score,Retain,Normalised listener-strength representation de...,True
7,listener_peak_ratio,Retain,Directly represents current audience size rela...,True
8,audience_below_peak,Exclude,Perfect inverse representation of listener_pea...,True
9,average_historical_streams,Exclude,Provides the same underlying historical stream...,True



Features marked for exclusion
-----------------------------
Number of features marked for exclusion: 6
- chart_longevity_years
- international_reach_score
- audience_below_peak
- average_historical_streams
- average_historical_streams_log
- chart_observations_log

Preferred retained features
---------------------------
- chart_longevity_days
- countries_charted
- listener_strength_score
- listener_peak_ratio
- historical_streaming_intensity_log
- chart_activity_factor
- reached_number_one

Model-specific features
-----------------------
- market_coverage_score
- Listeners
- historical_streaming_intensity
- TikTok Likes
- TikTok Views
- PkListeners
- best_chart_strength
- peak_chart_performance

Decision validation
-------------------
Duplicate feature decisions: 0
Invalid decision labels: 0

Dataset status
--------------
Current dataset features: 93
Features marked for future exclusion: 6
Features physically removed at this stage: 0


#### Interpretation

The feature retention review documented decisions for 21 features identified
during the correlation and redundancy analysis. All 21 features were present
in the current dataset, confirming that the retention decisions were applied
to valid engineered features.

Seven features were classified as preferred features to retain, eight were
classified as model-specific, and six were marked for future exclusion. The
excluded features were not physically removed at this stage, allowing the
complete feature-engineered dataset to remain available for later
model-specific feature selection.

The six features marked for exclusion were `chart_longevity_years`,
`international_reach_score`, `audience_below_peak`,
`average_historical_streams`, `average_historical_streams_log`, and
`chart_observations_log`. These features were identified as redundant because
they represent information already captured by preferred alternative features.

For example, `chart_longevity_days` was retained instead of
`chart_longevity_years`, while `listener_peak_ratio` was retained instead of
the mathematically redundant `audience_below_peak`. Similarly,
`countries_charted` was retained as the directly interpretable geographic
coverage measure, while `international_reach_score` was marked for exclusion.

Eight features were classified as model-specific rather than automatically
removed. These variables may be strongly correlated with other features but
represent conceptually distinct information or alternative representations
that could remain useful for particular PMIP machine-learning tasks.

The working dataset therefore remains unchanged at 93 features, with six
features documented for potential exclusion when the final model-specific
feature sets are constructed. This approach reduces unnecessary redundancy
without prematurely discarding potentially useful predictive information.

## 10. Target Leakage and Feature Eligibility

### Purpose

Before constructing the final machine-learning feature sets, the engineered
dataset must be reviewed to determine which variables can safely be used as
predictors.

This section identifies potential target variables for PMIP's predictive
models, investigates features that could introduce target leakage, separates
identifiers and other non-predictive columns from genuine predictor variables,
and establishes the final pool of features eligible for machine-learning
modelling.

The purpose is to ensure that future PMIP models learn from information that
would realistically be available at prediction time rather than from variables
that directly reveal, duplicate or depend on the outcome being predicted.


### 10.1 Potential Target Variables

PMIP is designed to support multiple music-intelligence tasks rather than a
single prediction problem. Therefore, different variables may become target
variables depending on the specific machine-learning model being developed.

This subsection reviews the available dataset columns and identifies variables
that could potentially represent prediction outcomes for artist performance,
streaming performance, chart success, audience growth and other PMIP
intelligence tasks.

Identifying these targets before selecting predictor features is important
because a variable used as the target, together with features that directly
derive from or reveal that target, must be excluded from the corresponding
model's predictor set.

In [34]:
# 10.1 Potential Target Variables

import pandas as pd

print("Potential target variable review")
print("--------------------------------")

print(f"Total dataset records: {len(df_features):,}")
print(f"Total dataset features: {len(df_features.columns):,}")


# ---------------------------------------------------------
# 1. Define potential PMIP prediction targets
# ---------------------------------------------------------

potential_targets = {
    "Streaming Performance": [
        "Spotify Streams",
        "total_historical_streams",
        "average_historical_streams",
        "max_historical_streams"
    ],

    "Chart Performance": [
        "best_chart_position",
        "average_chart_position",
        "peak_chart_performance",
        "reached_top_10",
        "reached_number_one"
    ],

    "Geographic Performance": [
        "countries_charted",
        "market_coverage_score",
        "broad_geographic_reach"
    ],

    "Artist Audience Performance": [
        "Listeners",
        "PkListeners",
        "Daily Trend",
        "listener_daily_growth_rate",
        "listener_peak_ratio"
    ],

    "Popularity / Platform Performance": [
        "Spotify Popularity"
    ]
}


# ---------------------------------------------------------
# 2. Review target availability
# ---------------------------------------------------------

target_rows = []

for target_group, features in potential_targets.items():

    for feature in features:

        exists = feature in df_features.columns

        if exists:
            non_missing = df_features[feature].notna().sum()
            missing = df_features[feature].isna().sum()

            coverage = (
                non_missing / len(df_features)
            ) * 100

            dtype = str(df_features[feature].dtype)

        else:
            non_missing = 0
            missing = len(df_features)
            coverage = 0
            dtype = "Not Available"

        target_rows.append({
            "Target Group": target_group,
            "Potential Target": feature,
            "Exists": exists,
            "Data Type": dtype,
            "Available Records": non_missing,
            "Missing Records": missing,
            "Coverage": coverage
        })


potential_targets_df = pd.DataFrame(target_rows)


print("\nPotential target variables")
print("--------------------------")

display(
    potential_targets_df.style.format({
        "Available Records": "{:,.0f}",
        "Missing Records": "{:,.0f}",
        "Coverage": "{:.2f}%"
    })
)


# ---------------------------------------------------------
# 3. Summary by target group
# ---------------------------------------------------------

target_group_summary = (
    potential_targets_df
    .groupby("Target Group")
    .agg(
        Potential_Targets=("Potential Target", "count"),
        Available_Targets=("Exists", "sum")
    )
    .reset_index()
)


print("\nPotential target groups")
print("-----------------------")

display(target_group_summary)


# ---------------------------------------------------------
# 4. Identify available potential targets
# ---------------------------------------------------------

available_targets = (
    potential_targets_df[
        potential_targets_df["Exists"]
    ]["Potential Target"]
    .tolist()
)

unavailable_targets = (
    potential_targets_df[
        ~potential_targets_df["Exists"]
    ]["Potential Target"]
    .tolist()
)


print("\nTarget availability summary")
print("---------------------------")

print(
    f"Potential targets reviewed: "
    f"{len(potential_targets_df)}"
)

print(
    f"Available potential targets: "
    f"{len(available_targets)}"
)

print(
    f"Unavailable potential targets: "
    f"{len(unavailable_targets)}"
)


if unavailable_targets:

    print("\nUnavailable potential targets")

    for feature in unavailable_targets:
        print(f"- {feature}")


# ---------------------------------------------------------
# 5. Important modelling note
# ---------------------------------------------------------

print("\nModelling eligibility note")
print("--------------------------")

print(
    "These variables are potential targets, not a single "
    "final target variable."
)

print(
    "The target selected will depend on the specific PMIP "
    "machine-learning task."
)

print(
    "Target-specific leakage controls will therefore be "
    "required before each model is trained."
)

Potential target variable review
--------------------------------
Total dataset records: 4,593
Total dataset features: 93

Potential target variables
--------------------------


,Target Group,Potential Target,Exists,Data Type,Available Records,Missing Records,Coverage
0,Streaming Performance,Spotify Streams,True,float64,"4,485",108,97.65%
1,Streaming Performance,total_historical_streams,True,float64,"2,235","2,358",48.66%
2,Streaming Performance,average_historical_streams,True,float64,"2,235","2,358",48.66%
3,Streaming Performance,max_historical_streams,True,float64,"2,235","2,358",48.66%
4,Chart Performance,best_chart_position,True,float64,"2,235","2,358",48.66%
5,Chart Performance,average_chart_position,True,float64,"2,235","2,358",48.66%
6,Chart Performance,peak_chart_performance,True,float64,"2,235","2,358",48.66%
7,Chart Performance,reached_top_10,True,int64,"4,593",0,100.00%
8,Chart Performance,reached_number_one,True,int64,"4,593",0,100.00%
9,Geographic Performance,countries_charted,True,float64,"2,235","2,358",48.66%



Potential target groups
-----------------------


,Target Group,Potential_Targets,Available_Targets
0,Artist Audience Performance,5,5
1,Chart Performance,5,5
2,Geographic Performance,3,3
3,Popularity / Platform Performance,1,1
4,Streaming Performance,4,4



Target availability summary
---------------------------
Potential targets reviewed: 18
Available potential targets: 18
Unavailable potential targets: 0

Modelling eligibility note
--------------------------
These variables are potential targets, not a single final target variable.
The target selected will depend on the specific PMIP machine-learning task.
Target-specific leakage controls will therefore be required before each model is trained.


### Interpretation

The potential target variable review identified 18 variables that could be used as prediction targets across the different machine-learning tasks planned for PMIP. All 18 variables were present in the current dataset, confirming that the feature-engineered dataset contains potential outcomes for streaming performance, chart performance, geographic performance, artist audience performance, and platform popularity.

The availability of these potential targets varies across the dataset. `Spotify Streams` provides the strongest coverage among the continuous streaming variables, with 4,485 of the 4,593 records available, representing 97.65% coverage. `Spotify Popularity` is available for 3,794 records, providing 82.60% coverage. These variables therefore provide relatively large samples for future modelling tasks.

The artist audience variables, including `Listeners`, `PkListeners`, `Daily Trend`, `listener_daily_growth_rate`, and `listener_peak_ratio`, are available for 3,389 records, representing 73.79% of the dataset. This is consistent with the listener-data availability identified during the missing-value analysis and means that approximately one-quarter of the dataset does not contain the listener information required for models based on these variables.

Historical streaming, chart and geographic performance variables generally have lower coverage. Variables such as `total_historical_streams`, `average_historical_streams`, `best_chart_position`, `average_chart_position`, `countries_charted`, and `market_coverage_score` are available for 2,235 records, representing 48.66% of the dataset. This corresponds directly with the historical chart-data coverage identified earlier and means that models using these variables as targets would operate on the subset of tracks for which historical chart information is available.

The engineered binary variables `reached_top_10`, `reached_number_one`, and `broad_geographic_reach` provide complete coverage across all 4,593 records. These variables may therefore be suitable for future classification tasks where PMIP predicts whether a track achieves a particular chart or geographic performance outcome.

At this stage, no single final target variable is selected. PMIP is intended to support multiple machine-learning tasks, meaning that different models may require different target variables. The purpose of this review is therefore to confirm which potential outcomes are available and assess their data coverage before determining which features can safely be used as predictors.

Overall, the results confirm that the feature-engineered dataset contains suitable potential target variables for multiple PMIP machine-learning tasks. The next stage will assess target leakage risk to ensure that features which directly reveal or are mathematically derived from a selected target are not incorrectly used as predictors during model training.

### 10.2 Leakage Risk Analysis

Target leakage occurs when a predictor contains information that directly or indirectly reveals the outcome that a machine-learning model is intended to predict.

This section evaluates potential leakage relationships between the candidate PMIP target variables and the available engineered features. Particular attention is given to features that are mathematically derived from a target, directly encode the target outcome, or represent closely related post-outcome information.

Leakage is assessed at the target-group level because PMIP will support multiple machine-learning tasks. A feature that is valid for one model may therefore need to be excluded from another.

In [35]:
# 10.2 Leakage Risk Analysis

print("Leakage risk analysis")
print("=" * 30)

# Define known leakage relationships for the main PMIP target families.
# These relationships are based on direct derivation, mathematical dependence,
# or variables that effectively reveal the target outcome.

leakage_rules = {
    "Spotify Streams": [
        "Spotify Streams_log"
    ],

    "total_historical_streams": [
        "total_historical_streams_log"
    ],

    "average_historical_streams": [
        "average_historical_streams_log",
        "historical_streaming_intensity",
        "historical_streaming_intensity_log"
    ],

    "best_chart_position": [
        "best_chart_strength",
        "peak_chart_performance",
        "reached_top_10",
        "reached_number_one"
    ],

    "average_chart_position": [
        "average_chart_strength",
        "average_chart_performance"
    ],

    "countries_charted": [
        "market_coverage_score",
        "international_reach_score",
        "broad_geographic_reach"
    ],

    "Listeners": [
        "Listeners_log",
        "listener_strength_score",
        "listener_peak_ratio",
        "audience_below_peak",
        "listener_daily_growth_rate"
    ],

    "PkListeners": [
        "PkListeners_log",
        "listener_peak_ratio",
        "audience_below_peak"
    ],

    "Daily Trend": [
        "listener_daily_growth_rate",
        "audience_trend_direction"
    ],

    "listener_daily_growth_rate": [
        "Daily Trend",
        "audience_trend_direction"
    ],

    "listener_peak_ratio": [
        "audience_below_peak"
    ],

    "reached_top_10": [
        "best_chart_position",
        "best_chart_strength",
        "peak_chart_performance"
    ],

    "reached_number_one": [
        "best_chart_position",
        "best_chart_strength",
        "peak_chart_performance"
    ],

    "broad_geographic_reach": [
        "countries_charted",
        "market_coverage_score",
        "international_reach_score"
    ]
}


leakage_records = []

for target, risky_features in leakage_rules.items():

    target_exists = target in df_features.columns

    for feature in risky_features:

        feature_exists = feature in df_features.columns

        if target == feature:
            risk_type = "Target itself"

        elif (
            feature.endswith("_log")
            or feature in [
                "listener_strength_score",
                "listener_peak_ratio",
                "audience_below_peak",
                "listener_daily_growth_rate",
                "best_chart_strength",
                "average_chart_strength",
                "market_coverage_score",
                "international_reach_score"
            ]
        ):
            risk_type = "Derived / Mathematical Leakage"

        else:
            risk_type = "Outcome-Related Leakage"

        leakage_records.append({
            "Target": target,
            "Risky Predictor": feature,
            "Target Exists": target_exists,
            "Predictor Exists": feature_exists,
            "Risk Type": risk_type
        })


leakage_review = pd.DataFrame(leakage_records)

print(f"Target variables reviewed: {leakage_review['Target'].nunique():,}")
print(f"Potential leakage relationships defined: {len(leakage_review):,}")

available_leakage = leakage_review[
    leakage_review["Target Exists"]
    & leakage_review["Predictor Exists"]
].copy()

print(
    f"Leakage relationships present in dataset: "
    f"{len(available_leakage):,}"
)


print("\nLeakage risk classification")
print("-" * 30)

risk_summary = (
    available_leakage["Risk Type"]
    .value_counts()
    .rename_axis("Risk Type")
    .reset_index(name="Relationships")
)

risk_summary["Percentage"] = (
    risk_summary["Relationships"]
    / len(available_leakage)
    * 100
).round(2)

display(risk_summary)


print("\nDetailed leakage relationships")
print("-" * 35)

display(
    available_leakage[
        [
            "Target",
            "Risky Predictor",
            "Risk Type"
        ]
    ].sort_values(
        ["Target", "Risk Type", "Risky Predictor"]
    ).reset_index(drop=True)
)


print("\nLeakage relationships by target")
print("-" * 35)

target_leakage_summary = (
    available_leakage
    .groupby("Target")
    .agg(
        Leakage_Risk_Features=("Risky Predictor", "count"),
        Risk_Types=("Risk Type", lambda x: ", ".join(sorted(set(x))))
    )
    .reset_index()
    .sort_values(
        "Leakage_Risk_Features",
        ascending=False
    )
)

display(target_leakage_summary)


print("\nModelling rule")
print("-" * 20)
print(
    "When a variable is selected as the target for a PMIP model, "
    "its identified leakage-risk features must be excluded from "
    "that model's predictor set before training."
)

Leakage risk analysis
Target variables reviewed: 14
Potential leakage relationships defined: 36
Leakage relationships present in dataset: 36

Leakage risk classification
------------------------------


,Risk Type,Relationships,Percentage
0,Derived / Mathematical Leakage,22,61.11
1,Outcome-Related Leakage,14,38.89



Detailed leakage relationships
-----------------------------------


,Target,Risky Predictor,Risk Type
0,Daily Trend,listener_daily_growth_rate,Derived / Mathematical Leakage
1,Daily Trend,audience_trend_direction,Outcome-Related Leakage
2,Listeners,Listeners_log,Derived / Mathematical Leakage
3,Listeners,audience_below_peak,Derived / Mathematical Leakage
4,Listeners,listener_daily_growth_rate,Derived / Mathematical Leakage
5,Listeners,listener_peak_ratio,Derived / Mathematical Leakage
6,Listeners,listener_strength_score,Derived / Mathematical Leakage
7,PkListeners,PkListeners_log,Derived / Mathematical Leakage
8,PkListeners,audience_below_peak,Derived / Mathematical Leakage
9,PkListeners,listener_peak_ratio,Derived / Mathematical Leakage



Leakage relationships by target
-----------------------------------


,Target,Leakage_Risk_Features,Risk_Types
1,Listeners,5,Derived / Mathematical Leakage
6,best_chart_position,4,"Derived / Mathematical Leakage, Outcome-Relate..."
2,PkListeners,3,Derived / Mathematical Leakage
5,average_historical_streams,3,"Derived / Mathematical Leakage, Outcome-Relate..."
7,broad_geographic_reach,3,"Derived / Mathematical Leakage, Outcome-Relate..."
8,countries_charted,3,"Derived / Mathematical Leakage, Outcome-Relate..."
11,reached_number_one,3,"Derived / Mathematical Leakage, Outcome-Relate..."
12,reached_top_10,3,"Derived / Mathematical Leakage, Outcome-Relate..."
0,Daily Trend,2,"Derived / Mathematical Leakage, Outcome-Relate..."
4,average_chart_position,2,"Derived / Mathematical Leakage, Outcome-Relate..."



Modelling rule
--------------------
When a variable is selected as the target for a PMIP model, its identified leakage-risk features must be excluded from that model's predictor set before training.


### Interpretation

The leakage risk analysis reviewed 14 potential target variables and identified 36 relationships where a predictor could create target leakage. All 36 identified relationships were present in the current dataset, confirming that these risks must be considered when constructing the predictor set for each PMIP machine-learning model.

Of the 36 leakage relationships identified, 22 (61.11%) were classified as derived or mathematical leakage, while 14 (38.89%) were classified as outcome-related leakage. Derived or mathematical leakage occurs when a predictor is directly calculated from, transformed from, or mathematically dependent on the target variable. Using such a feature would provide the model with information that effectively reveals part or all of the answer it is attempting to predict.

For example, if `Spotify Streams` is selected as the target variable, `Spotify Streams_log` must not be used as a predictor because it is simply a logarithmic transformation of the same streaming value. Similarly, when `Listeners` is the target, variables such as `Listeners_log`, `listener_strength_score`, `listener_peak_ratio`, `listener_daily_growth_rate`, and `audience_below_peak` present leakage risks because they are derived from or mathematically dependent on listener information.

The analysis also identified leakage within the historical performance features. For example, when `best_chart_position` is used as the target, `best_chart_strength` creates direct mathematical leakage, while `peak_chart_performance`, `reached_top_10`, and `reached_number_one` are outcome-related because they contain information connected to the track's achieved chart position. These variables must therefore be excluded from a model attempting to predict `best_chart_position`.

Similar relationships were identified for geographic performance. When `countries_charted` is selected as a target, `market_coverage_score` and `international_reach_score` create mathematical leakage because they are derived from geographic coverage information. `broad_geographic_reach` is also outcome-related because its value depends on the extent of the track's geographic chart performance.

The results also demonstrate why leakage control must be model-specific rather than removing every risky feature from the complete dataset. A variable that creates leakage for one target may still be a valid and useful predictor for another target. For example, `listener_strength_score` cannot safely predict `Listeners`, but it may still provide useful audience information when predicting an unrelated performance outcome.

Therefore, the identified leakage-risk features will not be permanently removed from `df_features` at this stage. Instead, when a particular target variable is selected for a PMIP model, the features associated with that target's leakage relationships must be excluded from its predictor set before training.

Overall, the leakage analysis establishes an important modelling rule for PMIP: a model must only learn from information that does not directly reveal its target. Applying these target-specific exclusions will help ensure that future evaluation results represent genuine predictive performance rather than artificially inflated performance caused by target leakage.

### 10.3 Identifier and Non-Predictive Columns

Machine-learning models should only receive features that provide meaningful predictive information. Identifier, descriptive, and administrative columns may uniquely identify records or provide useful information for the PMIP application, but they should not automatically be used as model predictors.

This section reviews the feature-engineered dataset to identify identifier and non-predictive columns that should be retained in the dataset for reference, display, joining, or reporting purposes but excluded from general machine-learning predictor sets.

The review distinguishes between identifier columns, descriptive text columns, date/reference columns, and other non-predictive fields. These columns will not be physically removed from the complete feature-engineered dataset because they may still be required by the Public Music Intelligence Platform.

In [36]:
# 10.3 Identifier and Non-Predictive Columns

print("Identifier and non-predictive column review")
print("=" * 43)

print(f"Total dataset records: {len(df_features):,}")
print(f"Total dataset features: {df_features.shape[1]:,}")

# ---------------------------------------------------------
# Define known identifier and non-predictive column groups
# ---------------------------------------------------------

identifier_candidates = [
    "ISRC",
    "spotify_track_id",
    "track_id",
    "Track ID"
]

descriptive_candidates = [
    "Track",
    "Album Name",
    "Artist",
    "Artist Name"
]

date_reference_candidates = [
    "Release Date",
    "first_chart_date",
    "last_chart_date"
]

ranking_reference_candidates = [
    "All Time Rank"
]

candidate_groups = {
    "Identifier": identifier_candidates,
    "Descriptive Text": descriptive_candidates,
    "Date / Reference": date_reference_candidates,
    "Ranking / Reference": ranking_reference_candidates
}

# ---------------------------------------------------------
# Find which candidate columns actually exist
# ---------------------------------------------------------

review_rows = []

for group, columns in candidate_groups.items():
    for column in columns:
        if column in df_features.columns:
            review_rows.append({
                "Column": column,
                "Classification": group,
                "Data Type": str(df_features[column].dtype),
                "Unique Values": df_features[column].nunique(dropna=True),
                "Missing Values": df_features[column].isna().sum(),
                "Predictor Eligibility": "Exclude from general predictor set"
            })

non_predictive_review = pd.DataFrame(review_rows)

print("\nIdentified identifier and non-predictive columns")
print("-" * 48)

if not non_predictive_review.empty:
    display(non_predictive_review)
else:
    print("No predefined identifier or non-predictive columns were found.")

# ---------------------------------------------------------
# Automatically inspect remaining object/string columns
# ---------------------------------------------------------

text_columns = df_features.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

already_reviewed = set(non_predictive_review["Column"]) if not non_predictive_review.empty else set()

additional_text_columns = [
    column for column in text_columns
    if column not in already_reviewed
]

print("\nAdditional text/categorical columns requiring review")
print("-" * 53)

if additional_text_columns:
    additional_review = pd.DataFrame({
        "Column": additional_text_columns,
        "Data Type": [
            str(df_features[column].dtype)
            for column in additional_text_columns
        ],
        "Unique Values": [
            df_features[column].nunique(dropna=True)
            for column in additional_text_columns
        ],
        "Missing Values": [
            df_features[column].isna().sum()
            for column in additional_text_columns
        ]
    })

    display(additional_review)
else:
    print("No additional text/categorical columns require review.")

# ---------------------------------------------------------
# Final exclusion list for general modelling
# ---------------------------------------------------------

general_non_predictive_columns = (
    non_predictive_review["Column"].tolist()
    if not non_predictive_review.empty
    else []
)

print("\nGeneral modelling exclusion summary")
print("-" * 35)

print(
    f"Identifier/non-predictive columns identified: "
    f"{len(general_non_predictive_columns)}"
)

for column in general_non_predictive_columns:
    print(f"- {column}")

print("\nDataset status")
print("-" * 14)

print(f"Current dataset features: {df_features.shape[1]:,}")
print("Columns physically removed at this stage: 0")

print(
    "\nThese columns are retained in df_features for PMIP reference, "
    "display, joining, and reporting purposes, but are not considered "
    "general machine-learning predictors."
)

Identifier and non-predictive column review
Total dataset records: 4,593
Total dataset features: 93

Identified identifier and non-predictive columns
------------------------------------------------


,Column,Classification,Data Type,Unique Values,Missing Values,Predictor Eligibility
0,ISRC,Identifier,str,4593,0,Exclude from general predictor set
1,Track,Descriptive Text,str,4365,0,Exclude from general predictor set
2,Album Name,Descriptive Text,str,4000,0,Exclude from general predictor set
3,Artist,Descriptive Text,str,1999,0,Exclude from general predictor set
4,Release Date,Date / Reference,datetime64[us],1562,0,Exclude from general predictor set
5,first_chart_date,Date / Reference,datetime64[us],464,2358,Exclude from general predictor set
6,last_chart_date,Date / Reference,datetime64[us],313,2358,Exclude from general predictor set
7,All Time Rank,Ranking / Reference,int64,4573,0,Exclude from general predictor set



Additional text/categorical columns requiring review
-----------------------------------------------------


,Column,Data Type,Unique Values,Missing Values
0,track_normalised,str,4314,0
1,artist_normalised,str,1997,0
2,audience_trend_direction,object,2,1204



General modelling exclusion summary
-----------------------------------
Identifier/non-predictive columns identified: 8
- ISRC
- Track
- Album Name
- Artist
- Release Date
- first_chart_date
- last_chart_date
- All Time Rank

Dataset status
--------------
Current dataset features: 93
Columns physically removed at this stage: 0

These columns are retained in df_features for PMIP reference, display, joining, and reporting purposes, but are not considered general machine-learning predictors.


### Interpretation

The identifier and non-predictive column review examined all 93 features in the current dataset and identified eight columns that should be excluded from general machine-learning predictor sets. These columns remain useful to PMIP for identification, display, joining, reporting, and data interpretation, but their raw values are not considered appropriate general predictors for model training.

The `ISRC` column was classified as an identifier. All 4,593 records have unique ISRC values, confirming that it primarily identifies individual recordings rather than representing a reusable performance characteristic. Allowing such a unique identifier into a model could encourage the model to memorise individual records rather than learn general patterns that can be applied to unseen music data.

The `Track`, `Album Name`, and `Artist` columns were classified as descriptive text features. They contain important information for displaying artists and tracks within the Public Music Intelligence Platform, but their raw text values should not automatically be supplied to the planned numerical machine-learning models. They are therefore retained for application and reporting purposes while being excluded from the general predictor set.

The raw `Release Date`, `first_chart_date`, and `last_chart_date` columns were classified as date/reference features. Their useful information has already been represented through engineered numerical features such as release year, release month, release age, and chart longevity. Retaining the raw dates for reference while excluding them from the general predictor set avoids unnecessarily providing raw datetime values to the models.

`All Time Rank` was also classified as a ranking/reference column. Although it contains performance-related information, it represents an existing ranking outcome rather than a straightforward independent predictor. It will therefore be excluded from the general predictor pool to reduce the possibility of introducing outcome-related information into future models.

The additional categorical review identified `track_normalised`, `artist_normalised`, and `audience_trend_direction`. The normalised track and artist columns were created primarily to support reliable dataset matching and integration, meaning that they function similarly to descriptive identifiers and should also be excluded from general predictor sets. In contrast, `audience_trend_direction` contains two meaningful audience trend categories and may potentially be useful for specific PMIP modelling tasks, subject to target-leakage controls and categorical encoding.

No columns were physically removed from `df_features` during this stage. The complete 93-feature dataset is intentionally preserved because identifier and descriptive information will still be required by the PMIP application even when those columns are excluded during machine-learning training.

Overall, this review separates application-level information from model-level predictive information. This ensures that future PMIP models are trained using meaningful predictor features while important track, artist, date, and identification information remains available for displaying and interpreting model results.

### 10.4 Final Eligible Predictor Features

The previous analyses identified redundant features, potential target variables, target-specific leakage relationships, and columns that should not be used as general machine-learning predictors.

This subsection combines those findings to define the final pool of features that are eligible for consideration as predictors in future PMIP machine-learning models.

The predictor pool is not intended to represent the final input features for every model. Each PMIP model will select an appropriate subset from this pool depending on its prediction target. Target variables and any features identified as leakage risks for the selected target must be removed before model training.

Identifier, descriptive, reference, integration-only, and previously excluded redundant features are removed from general predictor eligibility.

In [37]:
# 10.4 Final Eligible Predictor Features

print("Final eligible predictor feature review")
print("=" * 41)

# ---------------------------------------------------------
# 1. Columns that should never be general ML predictors
# ---------------------------------------------------------

non_predictive_columns = [
    "ISRC",
    "Track",
    "Album Name",
    "Artist",
    "Release Date",
    "first_chart_date",
    "last_chart_date",
    "All Time Rank",
    "track_normalised",
    "artist_normalised"
]

# Keep only columns that actually exist
non_predictive_columns = [
    column for column in non_predictive_columns
    if column in df_features.columns
]

# ---------------------------------------------------------
# 2. Features marked for exclusion during Section 9.3
# ---------------------------------------------------------

redundant_features_to_exclude = [
    "chart_longevity_years",
    "international_reach_score",
    "audience_below_peak",
    "average_historical_streams",
    "average_historical_streams_log",
    "chart_observations_log"
]

redundant_features_to_exclude = [
    column for column in redundant_features_to_exclude
    if column in df_features.columns
]

# ---------------------------------------------------------
# 3. Build the general predictor eligibility pool
# ---------------------------------------------------------

general_exclusions = sorted(
    set(non_predictive_columns + redundant_features_to_exclude)
)

eligible_predictor_features = [
    column for column in df_features.columns
    if column not in general_exclusions
]

# ---------------------------------------------------------
# 4. Summary
# ---------------------------------------------------------

print(f"Total dataset records: {len(df_features):,}")
print(f"Total dataset features: {df_features.shape[1]:,}")

print("\nGeneral predictor exclusions")
print("-" * 29)

print(
    f"Identifier / descriptive / reference / integration exclusions: "
    f"{len(non_predictive_columns)}"
)

print(
    f"Redundant feature exclusions: "
    f"{len(redundant_features_to_exclude)}"
)

print(
    f"Total unique general exclusions: "
    f"{len(general_exclusions)}"
)

print(
    f"Final generally eligible predictor features: "
    f"{len(eligible_predictor_features)}"
)

# ---------------------------------------------------------
# 5. Display excluded columns
# ---------------------------------------------------------

exclusion_records = []

for column in general_exclusions:

    if column in non_predictive_columns:
        reason = (
            "Identifier, descriptive, reference, "
            "or integration-only column"
        )

    elif column in redundant_features_to_exclude:
        reason = "Redundant feature identified in Section 9.3"

    else:
        reason = "General modelling exclusion"

    exclusion_records.append({
        "Feature": column,
        "Reason for General Exclusion": reason
    })

exclusion_summary = pd.DataFrame(exclusion_records)

print("\nFeatures excluded from the general predictor pool")
print("-" * 48)

display(exclusion_summary)

# ---------------------------------------------------------
# 6. Review eligible features by data type
# ---------------------------------------------------------

eligible_feature_summary = pd.DataFrame({
    "Feature": eligible_predictor_features,
    "Data Type": [
        str(df_features[column].dtype)
        for column in eligible_predictor_features
    ],
    "Available Records": [
        df_features[column].notna().sum()
        for column in eligible_predictor_features
    ],
    "Missing Records": [
        df_features[column].isna().sum()
        for column in eligible_predictor_features
    ]
})

eligible_feature_summary["Coverage"] = (
    eligible_feature_summary["Available Records"]
    / len(df_features)
    * 100
).round(2)

eligible_feature_summary["Coverage"] = (
    eligible_feature_summary["Coverage"]
    .map(lambda value: f"{value:.2f}%")
)

print("\nFinal generally eligible predictor features")
print("-" * 43)

display(eligible_feature_summary)

# ---------------------------------------------------------
# 7. Numerical vs categorical eligible features
# ---------------------------------------------------------

eligible_numerical_features = [
    column for column in eligible_predictor_features
    if pd.api.types.is_numeric_dtype(df_features[column])
]

eligible_categorical_features = [
    column for column in eligible_predictor_features
    if not pd.api.types.is_numeric_dtype(df_features[column])
]

print("\nEligible predictor type summary")
print("-" * 31)

print(
    f"Numerical eligible features: "
    f"{len(eligible_numerical_features)}"
)

print(
    f"Categorical / non-numerical eligible features: "
    f"{len(eligible_categorical_features)}"
)

if eligible_categorical_features:
    print("\nCategorical / non-numerical features requiring later encoding:")
    
    for column in eligible_categorical_features:
        print(f"- {column}")

# ---------------------------------------------------------
# 8. Target-specific modelling reminder
# ---------------------------------------------------------

print("\nTarget-specific eligibility rule")
print("-" * 32)

print(
    "Potential target variables remain in the general eligibility pool "
    "because PMIP will contain multiple machine-learning models."
)

print(
    "Before training an individual model, the selected target variable "
    "must be removed from X together with all leakage-risk features "
    "identified for that target in Section 10.2."
)

# ---------------------------------------------------------
# 9. Integrity checks
# ---------------------------------------------------------

remaining_non_predictive = [
    column for column in non_predictive_columns
    if column in eligible_predictor_features
]

remaining_redundant = [
    column for column in redundant_features_to_exclude
    if column in eligible_predictor_features
]

print("\nEligibility integrity checks")
print("-" * 28)

print(
    f"Non-predictive columns incorrectly eligible: "
    f"{len(remaining_non_predictive)}"
)

print(
    f"Redundant excluded features incorrectly eligible: "
    f"{len(remaining_redundant)}"
)

print("\nDataset preservation check")
print("-" * 26)

print(f"Features retained in df_features: {df_features.shape[1]:,}")
print("Features physically removed from df_features: 0")

Final eligible predictor feature review
Total dataset records: 4,593
Total dataset features: 93

General predictor exclusions
-----------------------------
Identifier / descriptive / reference / integration exclusions: 10
Redundant feature exclusions: 6
Total unique general exclusions: 16
Final generally eligible predictor features: 77

Features excluded from the general predictor pool
------------------------------------------------


,Feature,Reason for General Exclusion
0,Album Name,"Identifier, descriptive, reference, or integra..."
1,All Time Rank,"Identifier, descriptive, reference, or integra..."
2,Artist,"Identifier, descriptive, reference, or integra..."
3,ISRC,"Identifier, descriptive, reference, or integra..."
4,Release Date,"Identifier, descriptive, reference, or integra..."
5,Track,"Identifier, descriptive, reference, or integra..."
6,artist_normalised,"Identifier, descriptive, reference, or integra..."
7,audience_below_peak,Redundant feature identified in Section 9.3
8,average_historical_streams,Redundant feature identified in Section 9.3
9,average_historical_streams_log,Redundant feature identified in Section 9.3



Final generally eligible predictor features
-------------------------------------------


,Feature,Data Type,Available Records,Missing Records,Coverage
0,Track Score,float64,4593,0,100.00%
1,Spotify Streams,float64,4485,108,97.65%
2,Spotify Playlist Count,float64,4528,65,98.58%
3,Spotify Playlist Reach,float64,4526,67,98.54%
4,Spotify Popularity,float64,3794,799,82.60%
...,...,...,...,...,...
72,chart_longevity_years_log,float64,2235,2358,48.66%
73,geographic_performance_intensity_log,float64,2235,2358,48.66%
74,Listeners_log,float64,3389,1204,73.79%
75,PkListeners_log,float64,3389,1204,73.79%



Eligible predictor type summary
-------------------------------
Numerical eligible features: 76
Categorical / non-numerical eligible features: 1

Categorical / non-numerical features requiring later encoding:
- audience_trend_direction

Target-specific eligibility rule
--------------------------------
Potential target variables remain in the general eligibility pool because PMIP will contain multiple machine-learning models.
Before training an individual model, the selected target variable must be removed from X together with all leakage-risk features identified for that target in Section 10.2.

Eligibility integrity checks
----------------------------
Non-predictive columns incorrectly eligible: 0
Redundant excluded features incorrectly eligible: 0

Dataset preservation check
--------------------------
Features retained in df_features: 93
Features physically removed from df_features: 0


#### Interpretation

The final predictor eligibility review was completed using all 4,593 records and the 93 features currently contained in the feature-engineered PMIP dataset. The purpose of this review was to define which features may be considered for future machine-learning models without physically removing information from the main dataset.

A total of 16 features were excluded from the general predictor pool. Ten of these were identifier, descriptive, reference, or integration-related columns, including `ISRC`, `Track`, `Album Name`, `Artist`, `Release Date`, `first_chart_date`, `last_chart_date`, `All Time Rank`, `track_normalised`, and `artist_normalised`. These columns remain useful for identifying tracks, displaying information, joining datasets, and reporting, but they are not suitable as general numerical predictors.

A further six features were excluded based on the redundancy decisions made in Section 9.3. These were `chart_longevity_years`, `international_reach_score`, `audience_below_peak`, `average_historical_streams`, `average_historical_streams_log`, and `chart_observations_log`. Each of these variables represents information already captured by a preferred alternative feature, so excluding them from the general predictor pool reduces unnecessary duplication and potential multicollinearity.

After these exclusions, 77 of the 93 dataset features remained generally eligible for future modelling. Of these, 76 are numerical features and one is non-numerical: `audience_trend_direction`. The categorical audience trend feature will therefore require an appropriate encoding method before it can be supplied to machine-learning algorithms that require numerical input.

The eligible predictor pool still contains potential target variables. This is intentional because PMIP will use the dataset for multiple machine-learning tasks rather than one single prediction problem. A variable that acts as the target in one model may still provide useful information for another model. Therefore, the final predictor set must be created separately for each PMIP model.

Before an individual model is trained, its selected target variable must be removed from the predictor matrix together with the target-specific leakage-risk features identified in Section 10.2. This ensures that each model learns from legitimate predictive information rather than information mathematically derived from, or directly revealing, its target.

The final integrity checks confirmed that no identifier or non-predictive columns were incorrectly included in the general predictor pool and no features marked for redundancy exclusion remained eligible. Both checks produced zero errors.

Finally, no columns were physically removed from `df_features`. The dataset therefore remains at 93 features, preserving the complete feature-engineered dataset while maintaining a separate list of 77 generally eligible predictor features for later model-specific selection.

Overall, the results confirm that the PMIP feature set has been successfully screened for predictor eligibility and is ready for model-specific feature selection and preparation.

## 11. Model-Specific Feature Sets

The previous sections established a general pool of features that may be considered for machine-learning modelling. However, PMIP is designed to support multiple music-intelligence tasks, and each task requires a different combination of predictors.

This section defines model-specific feature sets for the main PMIP machine-learning components. Features are selected according to their relevance to each prediction task, their interpretability, their availability, the redundancy decisions made in Section 9, and the target-leakage controls established in Section 10.

The complete feature-engineered dataset is preserved. The feature sets defined below therefore represent modelling configurations rather than permanently removing columns from `df_features`.

### 11.1 Artist Performance Forecasting Features

This subsection defines the candidate predictors for forecasting artist or track performance. The feature set considers current platform performance, historical chart behaviour, artist audience information, release characteristics, geographic performance and engineered indicators while excluding inappropriate identifiers, redundant variables and target-specific leakage features.

In [38]:
# 11.1 Artist Performance Forecasting Features

print("Artist performance forecasting feature selection")
print("=" * 46)

# ---------------------------------------------------------
# Candidate feature groups
# ---------------------------------------------------------

forecasting_feature_groups = {
    "Release Features": [
        "release_year",
        "release_month",
        "release_age_days",
        "release_age_years"
    ],

    "Spotify Performance": [
        "Track Score",
        "Spotify Popularity",
        "Spotify Playlist Count_log",
        "Spotify Playlist Reach_log"
    ],

    "Cross-Platform Performance": [
        "YouTube Views_log",
        "YouTube Likes_log",
        "TikTok Posts_log",
        "TikTok Views_log",
        "TikTok Likes_log",
        "AirPlay Spins_log",
        "Shazam Counts_log",
        "Pandora Streams_log",
        "Pandora Track Stations_log",
        "Deezer Playlist Count_log",
        "Deezer Playlist Reach_log",
        "Amazon Playlist Count_log",
        "Apple Music Playlist Count_log",
        "SiriusXM Spins_log"
    ],

    "Historical Performance": [
        "historical_streaming_intensity_log",
        "chart_longevity_days",
        "best_chart_strength",
        "average_chart_strength",
        "chart_activity_factor",
        "peak_chart_performance",
        "has_historical_chart_data",
        "reached_top_10",
        "reached_number_one",
        "sustained_chart_activity"
    ],

    "Geographic Performance": [
        "countries_charted",
        "market_coverage_score",
        "geographic_performance_intensity_log",
        "broad_geographic_reach"
    ],

    "Artist Audience": [
        "listener_strength_score",
        "listener_peak_ratio",
        "listener_daily_growth_rate",
        "audience_trend_direction",
        "has_listener_data"
    ]
}

# ---------------------------------------------------------
# Flatten candidate feature list
# ---------------------------------------------------------

forecasting_candidates = []

for group_features in forecasting_feature_groups.values():
    forecasting_candidates.extend(group_features)

# Remove duplicates while preserving order
forecasting_candidates = list(dict.fromkeys(forecasting_candidates))

# ---------------------------------------------------------
# Check feature availability
# ---------------------------------------------------------

available_forecasting_features = [
    feature
    for feature in forecasting_candidates
    if feature in df_features.columns
]

missing_forecasting_features = [
    feature
    for feature in forecasting_candidates
    if feature not in df_features.columns
]

print(f"Candidate forecasting features requested: {len(forecasting_candidates):,}")
print(f"Candidate features available: {len(available_forecasting_features):,}")
print(f"Candidate features unavailable: {len(missing_forecasting_features):,}")

# ---------------------------------------------------------
# Feature-group summary
# ---------------------------------------------------------

group_summary = []

for group_name, group_features in forecasting_feature_groups.items():

    available = [
        feature for feature in group_features
        if feature in df_features.columns
    ]

    unavailable = [
        feature for feature in group_features
        if feature not in df_features.columns
    ]

    group_summary.append({
        "Feature Group": group_name,
        "Requested Features": len(group_features),
        "Available Features": len(available),
        "Unavailable Features": len(unavailable)
    })

forecasting_group_summary = pd.DataFrame(group_summary)

print("\nForecasting feature-group summary")
print("-" * 35)

display(forecasting_group_summary)

# ---------------------------------------------------------
# Detailed selected feature review
# ---------------------------------------------------------

feature_review = []

for feature in available_forecasting_features:

    available_records = df_features[feature].notna().sum()
    missing_records = df_features[feature].isna().sum()
    coverage = (available_records / len(df_features)) * 100

    feature_review.append({
        "Feature": feature,
        "Data Type": str(df_features[feature].dtype),
        "Available Records": available_records,
        "Missing Records": missing_records,
        "Coverage": f"{coverage:.2f}%"
    })

forecasting_feature_review = pd.DataFrame(feature_review)

print("\nSelected forecasting feature review")
print("-" * 36)

display(forecasting_feature_review)

# ---------------------------------------------------------
# Check against general eligibility pool from Section 10.4
# ---------------------------------------------------------

if "eligible_predictor_features" in globals():

    not_generally_eligible = [
        feature
        for feature in available_forecasting_features
        if feature not in eligible_predictor_features
    ]

    print("\nGeneral predictor eligibility check")
    print("-" * 35)

    print(
        f"Selected features generally eligible: "
        f"{len(available_forecasting_features) - len(not_generally_eligible):,}"
    )

    print(
        f"Selected features not generally eligible: "
        f"{len(not_generally_eligible):,}"
    )

    if not_generally_eligible:
        print("\nFeatures requiring review:")
        for feature in not_generally_eligible:
            print(f"- {feature}")

else:
    print("\nGeneral eligibility list was not found.")
    print("Section 10.4 should be run before final model preparation.")

# ---------------------------------------------------------
# Missing candidate features
# ---------------------------------------------------------

if missing_forecasting_features:

    print("\nRequested features not found in dataset")
    print("-" * 39)

    for feature in missing_forecasting_features:
        print(f"- {feature}")

# ---------------------------------------------------------
# Final feature-set status
# ---------------------------------------------------------

print("\nArtist performance forecasting feature-set status")
print("-" * 49)

print(f"Dataset records: {len(df_features):,}")
print(f"Selected candidate predictors: {len(available_forecasting_features):,}")

numerical_forecasting_features = [
    feature
    for feature in available_forecasting_features
    if pd.api.types.is_numeric_dtype(df_features[feature])
]

categorical_forecasting_features = [
    feature
    for feature in available_forecasting_features
    if not pd.api.types.is_numeric_dtype(df_features[feature])
]

print(
    f"Numerical forecasting predictors: "
    f"{len(numerical_forecasting_features):,}"
)

print(
    f"Categorical forecasting predictors: "
    f"{len(categorical_forecasting_features):,}"
)

if categorical_forecasting_features:
    print("\nCategorical features requiring later encoding:")
    for feature in categorical_forecasting_features:
        print(f"- {feature}")

print(
    "\nNote: This is a candidate forecasting feature set. "
    "The final target and its leakage-risk features must be removed "
    "before model training."
)

Artist performance forecasting feature selection
Candidate forecasting features requested: 41
Candidate features available: 39
Candidate features unavailable: 2

Forecasting feature-group summary
-----------------------------------


,Feature Group,Requested Features,Available Features,Unavailable Features
0,Release Features,4,2,2
1,Spotify Performance,4,4,0
2,Cross-Platform Performance,14,14,0
3,Historical Performance,10,10,0
4,Geographic Performance,4,4,0
5,Artist Audience,5,5,0



Selected forecasting feature review
------------------------------------


,Feature,Data Type,Available Records,Missing Records,Coverage
0,release_year,int32,4593,0,100.00%
1,release_month,int32,4593,0,100.00%
2,Track Score,float64,4593,0,100.00%
3,Spotify Popularity,float64,3794,799,82.60%
4,Spotify Playlist Count_log,float64,4528,65,98.58%
5,Spotify Playlist Reach_log,float64,4526,67,98.54%
6,YouTube Views_log,float64,4290,303,93.40%
7,YouTube Likes_log,float64,4283,310,93.25%
8,TikTok Posts_log,float64,3425,1168,74.57%
9,TikTok Views_log,float64,3617,976,78.75%



General predictor eligibility check
-----------------------------------
Selected features generally eligible: 39
Selected features not generally eligible: 0

Requested features not found in dataset
---------------------------------------
- release_age_days
- release_age_years

Artist performance forecasting feature-set status
-------------------------------------------------
Dataset records: 4,593
Selected candidate predictors: 39
Numerical forecasting predictors: 38
Categorical forecasting predictors: 1

Categorical features requiring later encoding:
- audience_trend_direction

Note: This is a candidate forecasting feature set. The final target and its leakage-risk features must be removed before model training.


#### Interpretation

The artist performance forecasting feature review evaluated 41 candidate predictors across six feature groups: release information, Spotify performance, cross-platform performance, historical performance, geographic performance, and artist audience behaviour. Of these candidates, 39 were available in the current feature-engineered dataset.

All candidate features from the Spotify, cross-platform, historical, geographic, and artist audience groups were successfully available. Within the release feature group, `release_year` and `release_month` were available, while `release_age_days` and `release_age_years` were not present in the current dataset. These unavailable variables were therefore not included in the forecasting feature set.

The selected feature set provides broad information about music performance across several dimensions. Spotify and cross-platform variables capture current digital performance across services such as Spotify, YouTube, TikTok, Shazam, Pandora, Deezer, Amazon Music, Apple Music and SiriusXM. Historical features provide information about previous chart performance, chart longevity and historical streaming activity, while geographic features describe the number and breadth of markets in which a track has performed.

Artist audience features were also included to represent current listener strength and audience behaviour. These include `listener_strength_score`, `listener_peak_ratio`, `listener_daily_growth_rate`, `audience_trend_direction`, and `has_listener_data`. Together, these variables allow a future forecasting model to consider both track-level performance and the wider audience position of the artist.

Feature coverage varies across the selected predictors. Several variables, including `release_year`, `release_month`, `Track Score`, historical-data indicators and geographic indicators, provide complete coverage across all 4,593 records. Other platform features have high but incomplete coverage, while historical chart variables provide 48.66% coverage and artist listener variables provide 73.79% coverage. This variation will need to be considered during model preparation and missing-value handling.

Importantly, all 39 selected features passed the general predictor eligibility rules established in Section 10.4. No identifier, descriptive, reference, integration, or previously excluded redundant feature was included in the forecasting feature set.

The final candidate set therefore contains 39 predictors, consisting of 38 numerical features and one categorical feature, `audience_trend_direction`. The categorical variable will require encoding before it can be used by machine-learning algorithms that require numerical input.

This feature set remains a candidate set rather than the final training matrix. Once a specific artist performance forecasting target is selected, the target itself and any features identified as leakage risks in Section 10.2 must be removed before model training. This ensures that the forecasting model learns from legitimate predictive information rather than information that directly or indirectly reveals the target outcome.

### 11.2 Artist Momentum Features

This subsection defines the candidate predictor features for PMIP's artist momentum analysis.

Artist momentum represents the direction and strength of an artist's current performance rather than their overall historical success. The feature set therefore focuses on audience growth, current listener strength, current performance relative to previous peak audience levels, streaming and platform engagement, chart performance, geographic reach, and release context.

The selected features are drawn from the generally eligible predictor pool established in Section 10.4. Features previously identified as redundant or non-predictive are excluded.

This subsection defines a candidate feature set only. Target-specific leakage controls will be applied later when the exact momentum target is selected.

In [40]:
# 11.2 Artist Momentum Features

print("Artist momentum feature selection")
print("=" * 50)

# ---------------------------------------------------------
# Candidate feature groups for artist momentum analysis
# ---------------------------------------------------------

momentum_feature_groups = {
    "Release Context": [
        "release_year",
        "release_month",
    ],

    "Spotify Performance": [
        "Track Score",
        "Spotify Popularity",
        "Spotify Playlist Count_log",
        "Spotify Playlist Reach_log",
    ],

    "Cross-Platform Performance": [
        "YouTube Views_log",
        "YouTube Likes_log",
        "TikTok Posts_log",
        "TikTok Views_log",
        "TikTok Likes_log",
        "AirPlay Spins_log",
        "Shazam Counts_log",
        "Pandora Streams_log",
        "Pandora Track Stations_log",
        "Deezer Playlist Count_log",
        "Deezer Playlist Reach_log",
        "Amazon Playlist Count_log",
        "Apple Music Playlist Count_log",
        "SiriusXM Spins_log",
    ],

    "Historical Performance": [
        "historical_streaming_intensity_log",
        "chart_longevity_days",
        "best_chart_strength",
        "average_chart_strength",
        "chart_activity_factor",
        "peak_chart_performance",
        "has_historical_chart_data",
        "reached_top_10",
        "reached_number_one",
        "sustained_chart_activity",
    ],

    "Geographic Performance": [
        "countries_charted",
        "market_coverage_score",
        "geographic_performance_intensity_log",
        "broad_geographic_reach",
    ],

    "Artist Audience Momentum": [
        "listener_strength_score",
        "listener_peak_ratio",
        "listener_daily_growth_rate",
        "audience_trend_direction",
        "has_listener_data",
    ],
}

# ---------------------------------------------------------
# Flatten requested feature list
# ---------------------------------------------------------

requested_momentum_features = [
    feature
    for features in momentum_feature_groups.values()
    for feature in features
]

# Remove duplicates while preserving order
requested_momentum_features = list(
    dict.fromkeys(requested_momentum_features)
)

# ---------------------------------------------------------
# Check feature availability
# ---------------------------------------------------------

available_momentum_features = [
    feature
    for feature in requested_momentum_features
    if feature in df_features.columns
]

unavailable_momentum_features = [
    feature
    for feature in requested_momentum_features
    if feature not in df_features.columns
]

print(
    f"Candidate momentum features requested: "
    f"{len(requested_momentum_features)}"
)

print(
    f"Candidate features available: "
    f"{len(available_momentum_features)}"
)

print(
    f"Candidate features unavailable: "
    f"{len(unavailable_momentum_features)}"
)

# ---------------------------------------------------------
# Feature-group summary
# ---------------------------------------------------------

group_summary = []

for group_name, features in momentum_feature_groups.items():

    available = [
        feature
        for feature in features
        if feature in df_features.columns
    ]

    unavailable = [
        feature
        for feature in features
        if feature not in df_features.columns
    ]

    group_summary.append({
        "Feature Group": group_name,
        "Requested Features": len(features),
        "Available Features": len(available),
        "Unavailable Features": len(unavailable),
    })

momentum_group_summary = pd.DataFrame(group_summary)

print("\nMomentum feature-group summary")
print("-" * 50)

display(momentum_group_summary)

# ---------------------------------------------------------
# Review selected momentum features
# ---------------------------------------------------------

momentum_feature_review = []

for feature in available_momentum_features:

    available_records = (
        df_features[feature]
        .notna()
        .sum()
    )

    missing_records = (
        df_features[feature]
        .isna()
        .sum()
    )

    coverage = (
        available_records
        / len(df_features)
    ) * 100

    momentum_feature_review.append({
        "Feature": feature,
        "Data Type": str(
            df_features[feature].dtype
        ),
        "Available Records": available_records,
        "Missing Records": missing_records,
        "Coverage": f"{coverage:.2f}%"
    })

momentum_feature_review = pd.DataFrame(
    momentum_feature_review
)

print("\nSelected artist momentum feature review")
print("-" * 50)

display(momentum_feature_review)

# ---------------------------------------------------------
# General predictor eligibility check
# ---------------------------------------------------------

not_generally_eligible = [
    feature
    for feature in available_momentum_features
    if feature not in eligible_predictor_features
]

generally_eligible = [
    feature
    for feature in available_momentum_features
    if feature in eligible_predictor_features
]

print("\nGeneral predictor eligibility check")
print("-" * 50)

print(
    f"Selected features generally eligible: "
    f"{len(generally_eligible)}"
)

print(
    f"Selected features not generally eligible: "
    f"{len(not_generally_eligible)}"
)

if not_generally_eligible:

    print("\nFeatures requiring review:")

    for feature in not_generally_eligible:
        print(f"- {feature}")

# ---------------------------------------------------------
# Missing requested features
# ---------------------------------------------------------

if unavailable_momentum_features:

    print("\nRequested features not found in dataset")
    print("-" * 50)

    for feature in unavailable_momentum_features:
        print(f"- {feature}")

# ---------------------------------------------------------
# Numerical and categorical feature counts
# ---------------------------------------------------------

numerical_momentum_features = [
    feature
    for feature in generally_eligible
    if pd.api.types.is_numeric_dtype(
        df_features[feature]
    )
]

categorical_momentum_features = [
    feature
    for feature in generally_eligible
    if not pd.api.types.is_numeric_dtype(
        df_features[feature]
    )
]

# ---------------------------------------------------------
# Final candidate feature set
# ---------------------------------------------------------

artist_momentum_features = generally_eligible

print("\nArtist momentum feature-set status")
print("-" * 50)

print(
    f"Dataset records: "
    f"{len(df_features):,}"
)

print(
    f"Selected candidate predictors: "
    f"{len(artist_momentum_features)}"
)

print(
    f"Numerical momentum predictors: "
    f"{len(numerical_momentum_features)}"
)

print(
    f"Categorical momentum predictors: "
    f"{len(categorical_momentum_features)}"
)

if categorical_momentum_features:

    print(
        "\nCategorical features requiring later encoding:"
    )

    for feature in categorical_momentum_features:
        print(f"- {feature}")

print(
    "\nNote: This is a candidate artist momentum feature set. "
    "The final momentum target and any leakage-risk predictors "
    "associated with that target must be removed before model training."
)

Artist momentum feature selection
Candidate momentum features requested: 39
Candidate features available: 39
Candidate features unavailable: 0

Momentum feature-group summary
--------------------------------------------------


,Feature Group,Requested Features,Available Features,Unavailable Features
0,Release Context,2,2,0
1,Spotify Performance,4,4,0
2,Cross-Platform Performance,14,14,0
3,Historical Performance,10,10,0
4,Geographic Performance,4,4,0
5,Artist Audience Momentum,5,5,0



Selected artist momentum feature review
--------------------------------------------------


,Feature,Data Type,Available Records,Missing Records,Coverage
0,release_year,int32,4593,0,100.00%
1,release_month,int32,4593,0,100.00%
2,Track Score,float64,4593,0,100.00%
3,Spotify Popularity,float64,3794,799,82.60%
4,Spotify Playlist Count_log,float64,4528,65,98.58%
5,Spotify Playlist Reach_log,float64,4526,67,98.54%
6,YouTube Views_log,float64,4290,303,93.40%
7,YouTube Likes_log,float64,4283,310,93.25%
8,TikTok Posts_log,float64,3425,1168,74.57%
9,TikTok Views_log,float64,3617,976,78.75%



General predictor eligibility check
--------------------------------------------------
Selected features generally eligible: 39
Selected features not generally eligible: 0

Artist momentum feature-set status
--------------------------------------------------
Dataset records: 4,593
Selected candidate predictors: 39
Numerical momentum predictors: 38
Categorical momentum predictors: 1

Categorical features requiring later encoding:
- audience_trend_direction

Note: This is a candidate artist momentum feature set. The final momentum target and any leakage-risk predictors associated with that target must be removed before model training.


#### Interpretation

The artist momentum feature selection identified 39 candidate predictor features, and all 39 were successfully found in the current feature-engineered dataset. This means that no required features were missing from the artist momentum feature set, providing a complete collection of candidate variables for later momentum modelling.

The selected features represent several different aspects of artist and track performance. Release context is represented by `release_year` and `release_month`, while Spotify performance is represented through variables such as `Track Score`, `Spotify Popularity`, `Spotify Playlist Count_log`, and `Spotify Playlist Reach_log`. Cross-platform indicators from YouTube, TikTok, AirPlay, Shazam, Pandora, Deezer, Amazon Music, Apple Music, and SiriusXM were also included to capture broader digital and streaming activity.

Historical performance features were included to provide information about previous chart behaviour. These include `historical_streaming_intensity_log`, `chart_longevity_days`, `best_chart_strength`, `average_chart_strength`, `chart_activity_factor`, and `peak_chart_performance`. Binary indicators such as `has_historical_chart_data`, `reached_top_10`, `reached_number_one`, and `sustained_chart_activity` provide additional information about whether a track has previously achieved important chart milestones.

Geographic performance is represented through `countries_charted`, `market_coverage_score`, `geographic_performance_intensity_log`, and `broad_geographic_reach`. These features may help the later momentum model distinguish between performance concentrated within a limited number of markets and performance that has achieved wider geographic reach.

Artist audience behaviour is particularly important for momentum analysis. The selected audience features include `listener_strength_score`, `listener_peak_ratio`, `listener_daily_growth_rate`, `audience_trend_direction`, and `has_listener_data`. Together, these variables describe the size and direction of an artist's current audience and may help identify whether audience interest is growing or declining.

All 39 selected features passed the general predictor eligibility check. Of these, 38 are numerical features and one, `audience_trend_direction`, is categorical. The categorical feature will therefore require encoding during the later model-preparation stage before it can be used by machine-learning algorithms that require numerical input.

The selected features have different levels of data coverage. Some engineered and platform features provide complete or near-complete coverage, while historical chart features have approximately 48.66% coverage and listener-based features have approximately 73.79% coverage. These missing values will therefore need to be handled appropriately during model-specific preprocessing rather than removing potentially valuable features at this stage.

Overall, the results confirm that a valid 39-feature candidate set has been established for PMIP's artist momentum modelling task. These features have not yet been treated as the final model inputs. When the final momentum target is selected, the target itself and any target-specific leakage-risk features identified in Section 10.2 must be removed before model training.

### 11.3 Streaming Anomaly Detection Features

This subsection defines the candidate feature set for PMIP's streaming anomaly detection component.

The purpose of this feature set is to capture streaming magnitude, cross-platform engagement, historical streaming behaviour, chart activity, geographic performance and artist audience information that may help identify unusual performance patterns.

Anomaly detection differs from the previous forecasting and momentum tasks because the objective is to identify observations whose performance patterns differ substantially from typical behaviour. Therefore, the feature set emphasises numerical performance indicators that can later be compared across tracks and artists.

The selected variables remain candidate features at this stage. Final preprocessing, missing-value treatment, scaling and the specific anomaly-detection algorithm will be handled during model development.

In [41]:
# ============================================================
# 11.3 Streaming Anomaly Detection Features
# ============================================================

# Candidate feature groups for streaming anomaly detection
anomaly_feature_groups = {
    "Spotify Streaming Performance": [
        "Spotify Streams",
        "Spotify Streams_log",
        "Spotify Playlist Count_log",
        "Spotify Playlist Reach_log",
        "Spotify Popularity",
        "Track Score",
    ],

    "Cross-Platform Performance": [
        "YouTube Views_log",
        "YouTube Likes_log",
        "TikTok Posts_log",
        "TikTok Views_log",
        "TikTok Likes_log",
        "AirPlay Spins_log",
        "Shazam Counts_log",
        "Pandora Streams_log",
        "Pandora Track Stations_log",
        "Deezer Playlist Count_log",
        "Deezer Playlist Reach_log",
        "Amazon Playlist Count_log",
        "Apple Music Playlist Count_log",
        "SiriusXM Spins_log",
    ],

    "Historical Streaming Performance": [
        "total_historical_streams",
        "total_historical_streams_log",
        "historical_streaming_intensity_log",
        "maximum_historical_streams",
        "chart_activity_factor",
        "chart_longevity_days",
        "has_historical_chart_data",
    ],

    "Geographic Performance": [
        "countries_charted",
        "market_coverage_score",
        "geographic_performance_intensity_log",
        "broad_geographic_reach",
    ],

    "Artist Audience Performance": [
        "Listeners",
        "Listeners_log",
        "PkListeners",
        "PkListeners_log",
        "listener_strength_score",
        "listener_peak_ratio",
        "listener_daily_growth_rate",
        "has_listener_data",
    ],
}


# ------------------------------------------------------------
# Flatten requested feature list
# ------------------------------------------------------------

requested_anomaly_features = [
    feature
    for features in anomaly_feature_groups.values()
    for feature in features
]

# Remove accidental duplicates while preserving order
requested_anomaly_features = list(dict.fromkeys(requested_anomaly_features))

available_anomaly_features = [
    feature
    for feature in requested_anomaly_features
    if feature in df_features.columns
]

unavailable_anomaly_features = [
    feature
    for feature in requested_anomaly_features
    if feature not in df_features.columns
]


print("Streaming anomaly detection feature selection")
print("=" * 47)

print(
    f"Candidate anomaly features requested: "
    f"{len(requested_anomaly_features):,}"
)

print(
    f"Candidate features available: "
    f"{len(available_anomaly_features):,}"
)

print(
    f"Candidate features unavailable: "
    f"{len(unavailable_anomaly_features):,}"
)


# ------------------------------------------------------------
# Feature-group availability summary
# ------------------------------------------------------------

group_summary_rows = []

for group_name, features in anomaly_feature_groups.items():

    available = [
        feature
        for feature in features
        if feature in df_features.columns
    ]

    unavailable = [
        feature
        for feature in features
        if feature not in df_features.columns
    ]

    group_summary_rows.append({
        "Feature Group": group_name,
        "Requested Features": len(features),
        "Available Features": len(available),
        "Unavailable Features": len(unavailable),
    })


anomaly_group_summary = pd.DataFrame(group_summary_rows)

print("\nAnomaly detection feature-group summary")
print("-" * 43)

display(anomaly_group_summary)


# ------------------------------------------------------------
# Detailed feature review
# ------------------------------------------------------------

anomaly_feature_review_rows = []

for feature in available_anomaly_features:

    available_records = int(df_features[feature].notna().sum())
    missing_records = int(df_features[feature].isna().sum())

    coverage = (
        available_records / len(df_features) * 100
        if len(df_features) > 0
        else 0
    )

    anomaly_feature_review_rows.append({
        "Feature": feature,
        "Data Type": str(df_features[feature].dtype),
        "Available Records": available_records,
        "Missing Records": missing_records,
        "Coverage": f"{coverage:.2f}%",
    })


anomaly_feature_review = pd.DataFrame(
    anomaly_feature_review_rows
)

print("\nSelected streaming anomaly feature review")
print("-" * 43)

display(anomaly_feature_review)


# ------------------------------------------------------------
# General predictor eligibility check
# ------------------------------------------------------------

# Rebuild the Section 10.4 eligible predictor list if the
# notebook kernel has been restarted or the variable is absent.
if "final_eligible_predictor_features" not in globals():

    identifier_non_predictive = {
        "ISRC",
        "Track",
        "Album Name",
        "Artist",
        "Release Date",
        "first_chart_date",
        "last_chart_date",
        "All Time Rank",
        "track_normalised",
        "artist_normalised",
    }

    redundant_exclusions = {
        "chart_longevity_years",
        "international_reach_score",
        "audience_below_peak",
        "average_historical_streams",
        "average_historical_streams_log",
        "chart_observations_log",
    }

    general_exclusions = (
        identifier_non_predictive
        | redundant_exclusions
    )

    final_eligible_predictor_features = [
        column
        for column in df_features.columns
        if column not in general_exclusions
    ]


not_generally_eligible = [
    feature
    for feature in available_anomaly_features
    if feature not in final_eligible_predictor_features
]

generally_eligible_anomaly_features = [
    feature
    for feature in available_anomaly_features
    if feature in final_eligible_predictor_features
]


print("\nGeneral predictor eligibility check")
print("-" * 43)

print(
    f"Selected features generally eligible: "
    f"{len(generally_eligible_anomaly_features):,}"
)

print(
    f"Selected features not generally eligible: "
    f"{len(not_generally_eligible):,}"
)

if not_generally_eligible:
    print("\nFeatures requiring review:")

    for feature in not_generally_eligible:
        print(f"- {feature}")


# ------------------------------------------------------------
# Missing requested features
# ------------------------------------------------------------

if unavailable_anomaly_features:

    print("\nRequested features not found in dataset")
    print("-" * 39)

    for feature in unavailable_anomaly_features:
        print(f"- {feature}")


# ------------------------------------------------------------
# Numerical / categorical feature counts
# ------------------------------------------------------------

numerical_anomaly_features = [
    feature
    for feature in generally_eligible_anomaly_features
    if pd.api.types.is_numeric_dtype(df_features[feature])
]

categorical_anomaly_features = [
    feature
    for feature in generally_eligible_anomaly_features
    if not pd.api.types.is_numeric_dtype(df_features[feature])
]


# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\nStreaming anomaly detection feature-set status")
print("-" * 47)

print(f"Dataset records: {len(df_features):,}")

print(
    f"Selected candidate predictors: "
    f"{len(generally_eligible_anomaly_features):,}"
)

print(
    f"Numerical anomaly predictors: "
    f"{len(numerical_anomaly_features):,}"
)

print(
    f"Categorical anomaly predictors: "
    f"{len(categorical_anomaly_features):,}"
)

if categorical_anomaly_features:

    print("\nCategorical features requiring later encoding:")

    for feature in categorical_anomaly_features:
        print(f"- {feature}")


print(
    "\nNote: This is a candidate streaming anomaly detection "
    "feature set. Final scaling, missing-value treatment and "
    "anomaly-detection modelling will be performed during the "
    "model-development stage."
)

Streaming anomaly detection feature selection
Candidate anomaly features requested: 39
Candidate features available: 38
Candidate features unavailable: 1

Anomaly detection feature-group summary
-------------------------------------------


,Feature Group,Requested Features,Available Features,Unavailable Features
0,Spotify Streaming Performance,6,6,0
1,Cross-Platform Performance,14,14,0
2,Historical Streaming Performance,7,6,1
3,Geographic Performance,4,4,0
4,Artist Audience Performance,8,8,0



Selected streaming anomaly feature review
-------------------------------------------


,Feature,Data Type,Available Records,Missing Records,Coverage
0,Spotify Streams,float64,4485,108,97.65%
1,Spotify Streams_log,float64,4485,108,97.65%
2,Spotify Playlist Count_log,float64,4528,65,98.58%
3,Spotify Playlist Reach_log,float64,4526,67,98.54%
4,Spotify Popularity,float64,3794,799,82.60%
5,Track Score,float64,4593,0,100.00%
6,YouTube Views_log,float64,4290,303,93.40%
7,YouTube Likes_log,float64,4283,310,93.25%
8,TikTok Posts_log,float64,3425,1168,74.57%
9,TikTok Views_log,float64,3617,976,78.75%



General predictor eligibility check
-------------------------------------------
Selected features generally eligible: 38
Selected features not generally eligible: 0

Requested features not found in dataset
---------------------------------------
- maximum_historical_streams

Streaming anomaly detection feature-set status
-----------------------------------------------
Dataset records: 4,593
Selected candidate predictors: 38
Numerical anomaly predictors: 38
Categorical anomaly predictors: 0

Note: This is a candidate streaming anomaly detection feature set. Final scaling, missing-value treatment and anomaly-detection modelling will be performed during the model-development stage.


#### Interpretation

The streaming anomaly detection feature review identified 39 candidate variables across Spotify streaming performance, cross-platform performance, historical streaming performance, geographic performance and artist audience performance. Of these, 38 features were available in the feature-engineered dataset, while only one requested feature was unavailable.

The Spotify streaming and cross-platform feature groups achieved complete feature availability, with all 6 Spotify features and all 14 cross-platform features successfully identified. This provides the anomaly detection component with performance information from several music platforms, including Spotify, YouTube, TikTok, AirPlay, Shazam, Pandora, Deezer, Amazon Music, Apple Music and SiriusXM. These variables can later help identify unusual differences between a track's performance across different platforms.

Historical streaming performance contributed 6 of the 7 requested features. The only unavailable variable was `maximum_historical_streams`. However, other historical measures including `total_historical_streams`, `total_historical_streams_log`, `historical_streaming_intensity_log`, `chart_activity_factor`, `chart_longevity_days` and `has_historical_chart_data` were successfully available. Therefore, the missing variable does not prevent historical streaming behaviour from being represented in the candidate feature set.

The geographic and artist audience groups were also fully available. Geographic variables include `countries_charted`, `market_coverage_score`, `geographic_performance_intensity_log` and `broad_geographic_reach`, while audience variables include listener counts, peak listener information, listener strength, listener peak ratio and daily listener growth. These features provide additional context that may help distinguish genuine unusual performance from differences caused by an artist's audience size or geographic reach.

Feature coverage varies across the selected variables. Spotify measures generally provide high coverage, while historical chart-related features contain data for 2,235 records, representing 48.66% of the dataset. Artist listener features are available for 3,389 records, representing 73.79%. SiriusXM also has comparatively lower coverage at 53.89%. These missing values will require appropriate treatment during the anomaly detection modelling stage rather than being interpreted automatically as anomalous behaviour.

All 38 available candidate features passed the general predictor eligibility check, with zero features identified as generally ineligible. Furthermore, all selected anomaly detection variables are numerical, meaning no categorical encoding is required for this candidate feature set.

Overall, the results confirm that PMIP has a broad numerical feature set for streaming anomaly detection, covering current streaming performance, cross-platform engagement, historical behaviour, geographic reach and artist audience characteristics. The 38 available features can therefore be retained as the candidate anomaly detection feature set, while scaling, missing-value treatment and final feature selection will be performed during model development.

### 11.4 Geographic and Market Intelligence Features

This section defines the candidate predictor features for PMIP's geographic and market intelligence component. The purpose of this feature set is to represent the geographic reach, market coverage, historical performance, cross-platform activity, and artist audience characteristics associated with music performance across markets.

The feature set combines direct geographic indicators such as the number of countries charted and market coverage with historical chart behaviour, streaming performance, cross-platform engagement, and artist audience information. These variables provide contextual information that may later support the analysis of geographic performance and market opportunities.

Only features identified as generally eligible predictors in Section 10 are considered. Final target selection, target-specific leakage removal, missing-value treatment, scaling, and model-specific feature selection will be performed during the machine-learning development stage.

In [42]:
# ============================================================
# 11.4 Geographic and Market Intelligence Features
# ============================================================

# Candidate feature groups for geographic and market intelligence
geographic_market_feature_groups = {
    "Geographic Performance": [
        "countries_charted",
        "market_coverage_score",
        "geographic_performance_intensity_log",
        "broad_geographic_reach",
    ],

    "Historical Market Performance": [
        "historical_streaming_intensity_log",
        "chart_longevity_days",
        "best_chart_strength",
        "average_chart_strength",
        "chart_activity_factor",
        "peak_chart_performance",
        "has_historical_chart_data",
        "reached_top_10",
        "reached_number_one",
        "sustained_chart_activity",
    ],

    "Spotify Performance": [
        "Track Score",
        "Spotify Popularity",
        "Spotify Playlist Count_log",
        "Spotify Playlist Reach_log",
    ],

    "Cross-Platform Performance": [
        "YouTube Views_log",
        "YouTube Likes_log",
        "TikTok Posts_log",
        "TikTok Views_log",
        "TikTok Likes_log",
        "AirPlay Spins_log",
        "Shazam Counts_log",
        "Pandora Streams_log",
        "Pandora Track Stations_log",
        "Deezer Playlist Count_log",
        "Deezer Playlist Reach_log",
        "Amazon Playlist Count_log",
        "Apple Music Playlist Count_log",
        "SiriusXM Spins_log",
    ],

    "Artist Audience": [
        "listener_strength_score",
        "listener_peak_ratio",
        "listener_daily_growth_rate",
        "audience_trend_direction",
        "has_listener_data",
    ],
}


# ------------------------------------------------------------
# Flatten requested features while preserving order
# ------------------------------------------------------------

requested_geographic_market_features = []

for feature_group in geographic_market_feature_groups.values():
    for feature in feature_group:
        if feature not in requested_geographic_market_features:
            requested_geographic_market_features.append(feature)


available_geographic_market_features = [
    feature
    for feature in requested_geographic_market_features
    if feature in df_features.columns
]

unavailable_geographic_market_features = [
    feature
    for feature in requested_geographic_market_features
    if feature not in df_features.columns
]


# ------------------------------------------------------------
# Feature-group availability summary
# ------------------------------------------------------------

group_summary_rows = []

for group_name, features in geographic_market_feature_groups.items():

    available = [
        feature
        for feature in features
        if feature in df_features.columns
    ]

    unavailable = [
        feature
        for feature in features
        if feature not in df_features.columns
    ]

    group_summary_rows.append({
        "Feature Group": group_name,
        "Requested Features": len(features),
        "Available Features": len(available),
        "Unavailable Features": len(unavailable),
    })

geographic_market_group_summary = pd.DataFrame(group_summary_rows)


# ------------------------------------------------------------
# Detailed feature review
# ------------------------------------------------------------

feature_review_rows = []

for feature in available_geographic_market_features:

    available_records = df_features[feature].notna().sum()
    missing_records = df_features[feature].isna().sum()
    coverage = (available_records / len(df_features)) * 100

    feature_review_rows.append({
        "Feature": feature,
        "Data Type": str(df_features[feature].dtype),
        "Available Records": available_records,
        "Missing Records": missing_records,
        "Coverage": f"{coverage:.2f}%"
    })

geographic_market_feature_review = pd.DataFrame(feature_review_rows)


# ------------------------------------------------------------
# General predictor eligibility check
# ------------------------------------------------------------

# Reconstruct the general eligible predictor pool if the
# Section 10.4 variable is unavailable in the current kernel.
if "final_eligible_predictor_features" in globals():

    general_eligible_features = final_eligible_predictor_features

elif "eligible_predictor_features" in globals():

    general_eligible_features = eligible_predictor_features

else:

    non_predictive_columns = {
        "ISRC",
        "Track",
        "Album Name",
        "Artist",
        "Release Date",
        "first_chart_date",
        "last_chart_date",
        "All Time Rank",
        "track_normalised",
        "artist_normalised",
    }

    redundant_columns = {
        "chart_longevity_years",
        "international_reach_score",
        "audience_below_peak",
        "average_historical_streams",
        "average_historical_streams_log",
        "chart_observations_log",
    }

    general_eligible_features = [
        column
        for column in df_features.columns
        if column not in non_predictive_columns
        and column not in redundant_columns
    ]


not_generally_eligible = [
    feature
    for feature in available_geographic_market_features
    if feature not in general_eligible_features
]


# ------------------------------------------------------------
# Numerical and categorical feature counts
# ------------------------------------------------------------

numerical_geographic_market_features = [
    feature
    for feature in available_geographic_market_features
    if pd.api.types.is_numeric_dtype(df_features[feature])
]

categorical_geographic_market_features = [
    feature
    for feature in available_geographic_market_features
    if not pd.api.types.is_numeric_dtype(df_features[feature])
]


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print("Geographic and market intelligence feature selection")
print("=====================================================")

print(
    f"Candidate geographic/market features requested: "
    f"{len(requested_geographic_market_features)}"
)

print(
    f"Candidate features available: "
    f"{len(available_geographic_market_features)}"
)

print(
    f"Candidate features unavailable: "
    f"{len(unavailable_geographic_market_features)}"
)


print("\nGeographic/market feature-group summary")
print("========================================")

display(geographic_market_group_summary)


print("\nSelected geographic and market feature review")
print("==============================================")

display(geographic_market_feature_review)


print("\nGeneral predictor eligibility check")
print("===================================")

print(
    f"Selected features generally eligible: "
    f"{len(available_geographic_market_features) - len(not_generally_eligible)}"
)

print(
    f"Selected features not generally eligible: "
    f"{len(not_generally_eligible)}"
)

if not_generally_eligible:
    print("\nFeatures requiring eligibility review:")
    for feature in not_generally_eligible:
        print(f"- {feature}")


if unavailable_geographic_market_features:
    print("\nRequested features not found in dataset")
    print("=======================================")

    for feature in unavailable_geographic_market_features:
        print(f"- {feature}")


print("\nGeographic and market intelligence feature-set status")
print("======================================================")

print(f"Dataset records: {len(df_features):,}")
print(
    f"Selected candidate predictors: "
    f"{len(available_geographic_market_features)}"
)
print(
    f"Numerical geographic/market predictors: "
    f"{len(numerical_geographic_market_features)}"
)
print(
    f"Categorical geographic/market predictors: "
    f"{len(categorical_geographic_market_features)}"
)

if categorical_geographic_market_features:
    print("\nCategorical features requiring later encoding:")
    for feature in categorical_geographic_market_features:
        print(f"- {feature}")

print(
    "\nNote: This is a candidate geographic and market intelligence "
    "feature set. Final target selection, leakage controls, "
    "missing-value treatment and model-specific feature selection "
    "will be performed during model development."
)

Geographic and market intelligence feature selection
Candidate geographic/market features requested: 37
Candidate features available: 37
Candidate features unavailable: 0

Geographic/market feature-group summary


,Feature Group,Requested Features,Available Features,Unavailable Features
0,Geographic Performance,4,4,0
1,Historical Market Performance,10,10,0
2,Spotify Performance,4,4,0
3,Cross-Platform Performance,14,14,0
4,Artist Audience,5,5,0



Selected geographic and market feature review


,Feature,Data Type,Available Records,Missing Records,Coverage
0,countries_charted,float64,2235,2358,48.66%
1,market_coverage_score,float64,2235,2358,48.66%
2,geographic_performance_intensity_log,float64,2235,2358,48.66%
3,broad_geographic_reach,int64,4593,0,100.00%
4,historical_streaming_intensity_log,float64,2235,2358,48.66%
5,chart_longevity_days,float64,2235,2358,48.66%
6,best_chart_strength,float64,2235,2358,48.66%
7,average_chart_strength,float64,2235,2358,48.66%
8,chart_activity_factor,float64,2235,2358,48.66%
9,peak_chart_performance,float64,2235,2358,48.66%



General predictor eligibility check
Selected features generally eligible: 37
Selected features not generally eligible: 0

Geographic and market intelligence feature-set status
Dataset records: 4,593
Selected candidate predictors: 37
Numerical geographic/market predictors: 36
Categorical geographic/market predictors: 1

Categorical features requiring later encoding:
- audience_trend_direction

Note: This is a candidate geographic and market intelligence feature set. Final target selection, leakage controls, missing-value treatment and model-specific feature selection will be performed during model development.


#### Interpretation

The geographic and market intelligence feature review identified 37 candidate predictor features across five feature groups: geographic performance, historical market performance, Spotify performance, cross-platform performance, and artist audience information. All 37 requested features were successfully found in the feature-engineered dataset, meaning no candidate variables were unavailable for this PMIP intelligence component.

The geographic performance group contributed four features: `countries_charted`, `market_coverage_score`, `geographic_performance_intensity_log`, and `broad_geographic_reach`. The first three features have historical geographic information for 2,235 records, representing 48.66% of the dataset, while `broad_geographic_reach` is available for all 4,593 records. Together, these variables provide different representations of how widely and strongly a track has performed across geographic markets.

The historical market performance group contributed all 10 requested features. Several continuous historical variables, including `historical_streaming_intensity_log`, `chart_longevity_days`, `best_chart_strength`, `average_chart_strength`, `chart_activity_factor`, and `peak_chart_performance`, have 48.66% coverage because they depend on the availability of historical chart records. However, indicators such as `has_historical_chart_data`, `reached_top_10`, `reached_number_one`, and `sustained_chart_activity` have complete coverage across all 4,593 records. These features provide useful historical context when evaluating an artist or track's presence within different markets.

Spotify and cross-platform performance features provide additional evidence of market activity beyond historical chart data. `Track Score` has complete coverage, while Spotify playlist measures exceed 98% coverage and `Spotify Popularity` has 82.60% coverage. Cross-platform variables from YouTube, TikTok, AirPlay, Shazam, Pandora, Deezer, Amazon Music, Apple Music, and SiriusXM provide further information about how music performs across different digital and broadcasting environments. Coverage varies between platforms, with SiriusXM having the lowest coverage among these variables at 53.89%.

The artist audience group contributed all five requested features. `listener_strength_score`, `listener_peak_ratio`, `listener_daily_growth_rate`, and `audience_trend_direction` are available for 3,389 records, representing 73.79% coverage, while `has_listener_data` is available for the complete dataset. These audience features can provide useful context when examining whether geographic or market performance is associated with the size, growth, or current direction of an artist's audience.

All 37 selected features passed the general predictor eligibility check, with zero candidate features identified as generally ineligible. Of these features, 36 are numerical and one, `audience_trend_direction`, is categorical. The categorical feature will therefore require encoding during the later model-preparation stage.

Overall, the results confirm that PMIP has a broad candidate feature set for geographic and market intelligence. The combination of geographic coverage, historical chart behaviour, Spotify performance, cross-platform activity, and artist audience characteristics provides a suitable foundation for later geographic and market intelligence modelling. Missing values in historically dependent and platform-specific variables will be handled during model development rather than removing these features at this stage.

### 11.5 Release Performance Features

This section defines the candidate predictor features for PMIP's release performance intelligence component. The purpose of this feature set is to represent the characteristics and performance signals associated with the performance of a music release.

The candidate variables combine release timing, Spotify performance, cross-platform engagement, historical chart behaviour, geographic reach, and artist audience information. These features provide different perspectives on the factors associated with the performance and market response of a release.

Only features identified as generally eligible predictors in Section 10 are considered. Final target selection, target-specific leakage removal, missing-value treatment, scaling, and model-specific feature selection will be performed during the machine-learning development stage.

In [43]:
# ============================================================
# 11.5 Release Performance Features
# ============================================================

release_performance_feature_groups = {
    "Release Features": [
        "release_year",
        "release_month",
    ],

    "Spotify Performance": [
        "Track Score",
        "Spotify Streams",
        "Spotify Streams_log",
        "Spotify Popularity",
        "Spotify Playlist Count_log",
        "Spotify Playlist Reach_log",
    ],

    "Cross-Platform Performance": [
        "YouTube Views_log",
        "YouTube Likes_log",
        "TikTok Posts_log",
        "TikTok Views_log",
        "TikTok Likes_log",
        "AirPlay Spins_log",
        "Shazam Counts_log",
        "Pandora Streams_log",
        "Pandora Track Stations_log",
        "Deezer Playlist Count_log",
        "Deezer Playlist Reach_log",
        "Amazon Playlist Count_log",
        "Apple Music Playlist Count_log",
        "SiriusXM Spins_log",
    ],

    "Historical Performance": [
        "total_historical_streams_log",
        "historical_streaming_intensity_log",
        "chart_longevity_days",
        "best_chart_strength",
        "average_chart_strength",
        "chart_activity_factor",
        "peak_chart_performance",
        "has_historical_chart_data",
        "reached_top_10",
        "reached_number_one",
        "sustained_chart_activity",
    ],

    "Geographic Performance": [
        "countries_charted",
        "market_coverage_score",
        "geographic_performance_intensity_log",
        "broad_geographic_reach",
    ],

    "Artist Audience": [
        "listener_strength_score",
        "listener_peak_ratio",
        "listener_daily_growth_rate",
        "audience_trend_direction",
        "has_listener_data",
    ],
}


# ------------------------------------------------------------
# Flatten requested features while preserving order
# ------------------------------------------------------------

requested_release_performance_features = []

for feature_group in release_performance_feature_groups.values():
    for feature in feature_group:
        if feature not in requested_release_performance_features:
            requested_release_performance_features.append(feature)


available_release_performance_features = [
    feature
    for feature in requested_release_performance_features
    if feature in df_features.columns
]

unavailable_release_performance_features = [
    feature
    for feature in requested_release_performance_features
    if feature not in df_features.columns
]


# ------------------------------------------------------------
# Feature-group availability summary
# ------------------------------------------------------------

group_summary_rows = []

for group_name, features in release_performance_feature_groups.items():

    available = [
        feature
        for feature in features
        if feature in df_features.columns
    ]

    unavailable = [
        feature
        for feature in features
        if feature not in df_features.columns
    ]

    group_summary_rows.append({
        "Feature Group": group_name,
        "Requested Features": len(features),
        "Available Features": len(available),
        "Unavailable Features": len(unavailable),
    })

release_performance_group_summary = pd.DataFrame(group_summary_rows)


# ------------------------------------------------------------
# Detailed feature review
# ------------------------------------------------------------

feature_review_rows = []

for feature in available_release_performance_features:

    available_records = df_features[feature].notna().sum()
    missing_records = df_features[feature].isna().sum()
    coverage = (available_records / len(df_features)) * 100

    feature_review_rows.append({
        "Feature": feature,
        "Data Type": str(df_features[feature].dtype),
        "Available Records": available_records,
        "Missing Records": missing_records,
        "Coverage": f"{coverage:.2f}%"
    })

release_performance_feature_review = pd.DataFrame(feature_review_rows)


# ------------------------------------------------------------
# General predictor eligibility check
# ------------------------------------------------------------

if "final_eligible_predictor_features" in globals():

    general_eligible_features = final_eligible_predictor_features

elif "eligible_predictor_features" in globals():

    general_eligible_features = eligible_predictor_features

else:

    non_predictive_columns = {
        "ISRC",
        "Track",
        "Album Name",
        "Artist",
        "Release Date",
        "first_chart_date",
        "last_chart_date",
        "All Time Rank",
        "track_normalised",
        "artist_normalised",
    }

    redundant_columns = {
        "chart_longevity_years",
        "international_reach_score",
        "audience_below_peak",
        "average_historical_streams",
        "average_historical_streams_log",
        "chart_observations_log",
    }

    general_eligible_features = [
        column
        for column in df_features.columns
        if column not in non_predictive_columns
        and column not in redundant_columns
    ]


not_generally_eligible = [
    feature
    for feature in available_release_performance_features
    if feature not in general_eligible_features
]


# ------------------------------------------------------------
# Numerical and categorical features
# ------------------------------------------------------------

numerical_release_performance_features = [
    feature
    for feature in available_release_performance_features
    if pd.api.types.is_numeric_dtype(df_features[feature])
]

categorical_release_performance_features = [
    feature
    for feature in available_release_performance_features
    if not pd.api.types.is_numeric_dtype(df_features[feature])
]


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print("Release performance feature selection")
print("=====================================")

print(
    f"Candidate release-performance features requested: "
    f"{len(requested_release_performance_features)}"
)

print(
    f"Candidate features available: "
    f"{len(available_release_performance_features)}"
)

print(
    f"Candidate features unavailable: "
    f"{len(unavailable_release_performance_features)}"
)


print("\nRelease performance feature-group summary")
print("=========================================")

display(release_performance_group_summary)


print("\nSelected release performance feature review")
print("===========================================")

display(release_performance_feature_review)


print("\nGeneral predictor eligibility check")
print("===================================")

print(
    f"Selected features generally eligible: "
    f"{len(available_release_performance_features) - len(not_generally_eligible)}"
)

print(
    f"Selected features not generally eligible: "
    f"{len(not_generally_eligible)}"
)

if not_generally_eligible:
    print("\nFeatures requiring eligibility review:")
    for feature in not_generally_eligible:
        print(f"- {feature}")


if unavailable_release_performance_features:
    print("\nRequested features not found in dataset")
    print("=======================================")

    for feature in unavailable_release_performance_features:
        print(f"- {feature}")


print("\nRelease performance feature-set status")
print("======================================")

print(f"Dataset records: {len(df_features):,}")

print(
    f"Selected candidate predictors: "
    f"{len(available_release_performance_features)}"
)

print(
    f"Numerical release-performance predictors: "
    f"{len(numerical_release_performance_features)}"
)

print(
    f"Categorical release-performance predictors: "
    f"{len(categorical_release_performance_features)}"
)

if categorical_release_performance_features:
    print("\nCategorical features requiring later encoding:")
    for feature in categorical_release_performance_features:
        print(f"- {feature}")


print(
    "\nNote: This is a candidate release performance feature set. "
    "The final release-performance target and any leakage-risk "
    "features associated with that target must be removed before "
    "model training."
)

Release performance feature selection
Candidate release-performance features requested: 42
Candidate features available: 42
Candidate features unavailable: 0

Release performance feature-group summary


,Feature Group,Requested Features,Available Features,Unavailable Features
0,Release Features,2,2,0
1,Spotify Performance,6,6,0
2,Cross-Platform Performance,14,14,0
3,Historical Performance,11,11,0
4,Geographic Performance,4,4,0
5,Artist Audience,5,5,0



Selected release performance feature review


,Feature,Data Type,Available Records,Missing Records,Coverage
0,release_year,int32,4593,0,100.00%
1,release_month,int32,4593,0,100.00%
2,Track Score,float64,4593,0,100.00%
3,Spotify Streams,float64,4485,108,97.65%
4,Spotify Streams_log,float64,4485,108,97.65%
5,Spotify Popularity,float64,3794,799,82.60%
6,Spotify Playlist Count_log,float64,4528,65,98.58%
7,Spotify Playlist Reach_log,float64,4526,67,98.54%
8,YouTube Views_log,float64,4290,303,93.40%
9,YouTube Likes_log,float64,4283,310,93.25%



General predictor eligibility check
Selected features generally eligible: 42
Selected features not generally eligible: 0

Release performance feature-set status
Dataset records: 4,593
Selected candidate predictors: 42
Numerical release-performance predictors: 41
Categorical release-performance predictors: 1

Categorical features requiring later encoding:
- audience_trend_direction

Note: This is a candidate release performance feature set. The final release-performance target and any leakage-risk features associated with that target must be removed before model training.


#### Interpretation

The release performance feature selection identified 42 candidate features for modelling release performance within PMIP. All 42 requested features were available in the feature-engineered dataset, meaning that no candidate variables were lost during the feature-selection process.

The selected feature set combines six areas of information: release features, Spotify performance, cross-platform performance, historical performance, geographic performance, and artist audience information. This provides a broad representation of the factors that may be associated with the performance of a music release rather than relying on a single streaming or popularity measure.

The release context is represented by `release_year` and `release_month`, both of which have complete coverage across all 4,593 records. Spotify-related variables also provide strong coverage, including `Spotify Streams` at 97.65%, playlist-related features above 98%, and `Spotify Popularity` at 82.60%. Cross-platform variables from YouTube, TikTok, AirPlay, Shazam, Pandora, Deezer, Amazon Music, Apple Music, and SiriusXM provide additional indicators of how releases perform across different music and media platforms.

Historical and geographic features have lower coverage of 48.66%, reflecting the availability of historical chart information in the integrated dataset. However, indicators such as `has_historical_chart_data`, `reached_top_10`, `reached_number_one`, `sustained_chart_activity`, and `broad_geographic_reach` provide complete coverage across all 4,593 records. These features allow the model to distinguish between releases with and without historical chart evidence while preserving the available historical information.

Artist audience features provide 73.79% coverage and include listener strength, proximity to peak listeners, daily audience growth, and audience trend direction. These variables provide additional context about the artist's current audience position when examining release performance.

The final candidate set contains 41 numerical features and one categorical feature, `audience_trend_direction`. This categorical variable will require encoding during the later model-development stage. All 42 selected features also passed the general predictor eligibility check, with no ineligible variables included in the candidate feature set.

Overall, the results confirm that a complete candidate feature set has been successfully defined for PMIP's release performance modelling. The variables have not yet been treated as the final training predictors because the final target variable still needs to be selected. Once the target is chosen, the target itself and any features identified as leakage risks in Section 10.2 must be removed before model training.

## 12. Final Feature Dataset Validation

This section performs the final validation of the feature-engineered PMIP dataset before it is saved and used during the machine-learning development stage.

The validation confirms that the feature-engineering process has preserved the original dataset records, produced appropriate data types, maintained expected missing-value patterns, and introduced no unintended duplicate records. It also provides a final summary of the engineered features created throughout the notebook.

The purpose of this section is to ensure that the resulting dataset is structurally consistent, traceable, and ready to support the different PMIP machine-learning models defined in the model-specific feature sets.

The final validation is organised into the following subsections:

- **12.1 Final Dataset Shape** – confirms the final number of records and features.
- **12.2 Data Type Validation** – verifies that engineered variables use appropriate data types.
- **12.3 Missing Value Validation** – reviews remaining missing values and their expected sources.
- **12.4 Duplicate Validation** – checks that feature engineering has not introduced unintended duplicate records.
- **12.5 Engineered Feature Summary** – summarises the final engineered features and confirms completion of the feature-engineering stage.

### 12.1 Final Dataset Shape

This subsection validates the final dimensions of the feature-engineered PMIP dataset.

The purpose is to confirm that feature engineering has preserved the original number of observations while adding the newly engineered variables required for the different machine-learning tasks. The validation also checks that no records were accidentally added or removed during the feature-engineering process.

In [44]:
# ============================================================
# 12.1 Final Dataset Shape
# ============================================================

print("Final feature dataset shape")
print("=" * 40)

final_rows, final_columns = df_features.shape

print(f"Final dataset records: {final_rows:,}")
print(f"Final dataset features: {final_columns:,}")

# Compare with the original integrated dataset
original_rows, original_columns = df.shape

print("\nOriginal integrated dataset shape")
print("=" * 40)
print(f"Original dataset records: {original_rows:,}")
print(f"Original dataset features: {original_columns:,}")

# Calculate changes introduced during feature engineering
row_difference = final_rows - original_rows
feature_difference = final_columns - original_columns

print("\nFeature engineering changes")
print("=" * 40)
print(f"Change in records: {row_difference:+,}")
print(f"New features added: {feature_difference:+,}")

# Validation checks
records_preserved = final_rows == original_rows
features_added = final_columns > original_columns

print("\nDataset shape validation")
print("=" * 40)
print(f"Original records preserved: {records_preserved}")
print(f"Additional features created: {features_added}")

if records_preserved:
    print("\nValidation result: PASS")
    print(
        "Feature engineering preserved all original dataset records."
    )
else:
    print("\nValidation result: REVIEW REQUIRED")
    print(
        "The number of records changed during feature engineering and "
        "should be investigated."
    )

Final feature dataset shape
Final dataset records: 4,593
Final dataset features: 93

Original integrated dataset shape
Original dataset records: 4,593
Original dataset features: 42

Feature engineering changes
Change in records: +0
New features added: +51

Dataset shape validation
Original records preserved: True
Additional features created: True

Validation result: PASS
Feature engineering preserved all original dataset records.


#### Interpretation

The final dataset shape validation confirms that the feature-engineering process successfully expanded the PMIP dataset while preserving all original records. The integrated dataset originally contained 4,593 records and 42 features, while the feature-engineered dataset contains the same 4,593 records and 93 features.

A total of 51 additional features were created during feature engineering. Importantly, the number of records did not change, with a recorded difference of zero. This confirms that the feature-engineering operations added new information to the existing observations without accidentally removing or introducing additional records.

The validation checks also confirmed that all original records were preserved and that additional features were successfully created. Both conditions returned `True`, resulting in an overall validation result of `PASS`.

Therefore, the final dataset shape is structurally consistent with the original integrated dataset and confirms that the feature-engineering stage successfully enriched the dataset without altering its original number of observations.

### 12.2 Data Type Validation

This subsection validates the data types of the final feature-engineered dataset. The purpose is to confirm that numerical, categorical, date, and text/reference variables remain stored using appropriate data types after feature engineering.

This validation is important because machine-learning algorithms require consistent and appropriate data representations. Numerical predictors should remain numeric, categorical features should be clearly identified for later encoding, and date or descriptive variables should remain available for reference without being incorrectly treated as general predictors.

In [45]:
# ------------------------------------------------------------
# 12.2 Data Type Validation
# ------------------------------------------------------------

print("Final dataset data type validation")
print("=" * 45)

# Count features by broad data-type category
numeric_columns = df_features.select_dtypes(include=["number"]).columns.tolist()
datetime_columns = df_features.select_dtypes(
    include=["datetime", "datetimetz"]
).columns.tolist()
categorical_text_columns = df_features.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()
boolean_columns = df_features.select_dtypes(include=["bool"]).columns.tolist()

classified_columns = set(
    numeric_columns
    + datetime_columns
    + categorical_text_columns
    + boolean_columns
)

other_columns = [
    column
    for column in df_features.columns
    if column not in classified_columns
]

print(f"Total features reviewed: {df_features.shape[1]:,}")
print(f"Numerical features: {len(numeric_columns)}")
print(f"Date/time features: {len(datetime_columns)}")
print(f"Text/categorical features: {len(categorical_text_columns)}")
print(f"Boolean features: {len(boolean_columns)}")
print(f"Other/unclassified features: {len(other_columns)}")


# ------------------------------------------------------------
# Detailed data-type summary
# ------------------------------------------------------------

dtype_summary = (
    df_features.dtypes
    .astype(str)
    .value_counts()
    .rename_axis("Data Type")
    .reset_index(name="Feature Count")
)

print("\nData type distribution")
print("=" * 45)

display(dtype_summary)


# ------------------------------------------------------------
# Review non-numerical features
# ------------------------------------------------------------

non_numeric_columns = [
    column
    for column in df_features.columns
    if column not in numeric_columns
]

non_numeric_review = pd.DataFrame({
    "Feature": non_numeric_columns,
    "Data Type": [
        str(df_features[column].dtype)
        for column in non_numeric_columns
    ],
    "Unique Values": [
        df_features[column].nunique(dropna=True)
        for column in non_numeric_columns
    ],
    "Missing Values": [
        df_features[column].isna().sum()
        for column in non_numeric_columns
    ]
})

print("\nNon-numerical feature review")
print("=" * 45)

display(non_numeric_review)


# ------------------------------------------------------------
# Model-specific categorical feature check
# ------------------------------------------------------------

expected_categorical_features = [
    "audience_trend_direction"
]

categorical_validation = []

for feature in expected_categorical_features:
    if feature in df_features.columns:
        categorical_validation.append({
            "Feature": feature,
            "Present": True,
            "Data Type": str(df_features[feature].dtype),
            "Unique Values": df_features[feature].nunique(dropna=True),
            "Missing Values": int(df_features[feature].isna().sum())
        })
    else:
        categorical_validation.append({
            "Feature": feature,
            "Present": False,
            "Data Type": None,
            "Unique Values": None,
            "Missing Values": None
        })

categorical_validation_df = pd.DataFrame(categorical_validation)

print("\nModel-specific categorical feature validation")
print("=" * 45)

display(categorical_validation_df)


# ------------------------------------------------------------
# Validation result
# ------------------------------------------------------------

all_columns_classified = len(other_columns) == 0
audience_trend_present = "audience_trend_direction" in df_features.columns

print("\nData type validation")
print("=" * 45)

print(f"All dataset features classified: {all_columns_classified}")
print(
    "Audience trend categorical feature present: "
    f"{audience_trend_present}"
)

if all_columns_classified and audience_trend_present:
    print("\nValidation result: PASS")
    print(
        "The final dataset contains recognised and "
        "appropriately classified data types."
    )
else:
    print("\nValidation result: REVIEW REQUIRED")

Final dataset data type validation
Total features reviewed: 93
Numerical features: 83
Date/time features: 3
Text/categorical features: 7
Boolean features: 0
Other/unclassified features: 0

Data type distribution


,Data Type,Feature Count
0,float64,73
1,int64,8
2,str,6
3,datetime64[us],3
4,int32,2
5,object,1



Non-numerical feature review


,Feature,Data Type,Unique Values,Missing Values
0,Track,str,4365,0
1,Album Name,str,4000,0
2,Artist,str,1999,0
3,Release Date,datetime64[us],1562,0
4,ISRC,str,4593,0
5,track_normalised,str,4314,0
6,artist_normalised,str,1997,0
7,first_chart_date,datetime64[us],464,2358
8,last_chart_date,datetime64[us],313,2358
9,audience_trend_direction,object,2,1204



Model-specific categorical feature validation


,Feature,Present,Data Type,Unique Values,Missing Values
0,audience_trend_direction,True,object,2,1204



Data type validation
All dataset features classified: True
Audience trend categorical feature present: True

Validation result: PASS
The final dataset contains recognised and appropriately classified data types.


#### Interpretation

The data type validation reviewed all 93 features in the final feature-engineered dataset and confirmed that every feature could be assigned to a recognised data type category. The dataset contains 83 numerical features, three date/time features, seven text or categorical features, no Boolean features, and no unclassified features.

The numerical variables form the majority of the dataset, consisting of 73 `float64`, eight `int64`, and two `int32` features. This is appropriate for PMIP because most of the engineered information represents numerical measurements such as streaming performance, chart activity, geographic reach, listener behaviour, and derived performance indicators.

The three date/time variables are `Release Date`, `first_chart_date`, and `last_chart_date`. These have remained as `datetime64[us]`, confirming that the temporal information has been preserved correctly. The descriptive and identifier variables, including `Track`, `Album Name`, `Artist`, `ISRC`, `track_normalised`, and `artist_normalised`, remain stored as text and can therefore continue to support identification, reporting, and integration purposes without being treated as numerical measurements.

The `audience_trend_direction` feature was also successfully identified as a categorical variable. It contains two observed categories and is available for 3,389 records, with 1,204 missing values corresponding to records without the required listener information. This feature will require categorical encoding later if it is included in a machine-learning model.

Most importantly, no features remained unclassified, and the specific categorical feature required for later modelling was successfully detected. Both validation checks returned `True`, resulting in an overall validation result of `PASS`.

Therefore, the final feature-engineered dataset contains consistent and appropriately classified data types and is structurally suitable to proceed to the remaining final dataset validation checks.

### 12.3 Missing Value Validation

This subsection validates missing values in the final feature-engineered dataset. The purpose is to determine which features contain missing observations, measure their level of data coverage, and confirm that missing values introduced or preserved during feature engineering are consistent with the availability of the underlying source data.

Missing values are not automatically treated as errors because some PMIP features depend on historical chart or artist listener information that is not available for every record. These missing values will therefore be documented and preserved for appropriate treatment during model-specific preprocessing.

In [46]:
# ------------------------------------------------------------
# 12.3 Missing Value Validation
# ------------------------------------------------------------

total_records = len(df_features)

missing_summary = pd.DataFrame({
    "Feature": df_features.columns,
    "Missing Values": df_features.isna().sum().values
})

missing_summary["Missing Percentage"] = (
    missing_summary["Missing Values"] / total_records * 100
).round(2)

missing_summary["Available Records"] = (
    total_records - missing_summary["Missing Values"]
)

missing_summary["Coverage"] = (
    missing_summary["Available Records"] / total_records * 100
).round(2)

# Keep only features containing missing values
features_with_missing = (
    missing_summary[
        missing_summary["Missing Values"] > 0
    ]
    .sort_values(
        by=["Missing Percentage", "Feature"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

features_without_missing = (
    missing_summary["Missing Values"] == 0
).sum()


print("Final dataset missing value validation")
print("=" * 50)

print(f"Total dataset records: {total_records:,}")
print(f"Total dataset features: {df_features.shape[1]}")
print(
    f"Features with missing values: "
    f"{len(features_with_missing)}"
)
print(
    f"Features without missing values: "
    f"{features_without_missing}"
)

print("\nFeatures containing missing values")
print("=" * 50)

display(features_with_missing)


# ------------------------------------------------------------
# Known source-data coverage groups
# ------------------------------------------------------------

historical_indicator = "has_historical_chart_data"
listener_indicator = "has_listener_data"

historical_records = (
    int(df_features[historical_indicator].sum())
    if historical_indicator in df_features.columns
    else None
)

listener_records = (
    int(df_features[listener_indicator].sum())
    if listener_indicator in df_features.columns
    else None
)

print("\nKnown source-data coverage")
print("=" * 50)

if historical_records is not None:
    print(
        f"Records with historical chart data: "
        f"{historical_records:,} "
        f"({historical_records / total_records * 100:.2f}%)"
    )

if listener_records is not None:
    print(
        f"Records with artist listener data: "
        f"{listener_records:,} "
        f"({listener_records / total_records * 100:.2f}%)"
    )


# ------------------------------------------------------------
# Missingness pattern validation
# ------------------------------------------------------------

historical_missing_expected = (
    total_records - historical_records
    if historical_records is not None
    else None
)

listener_missing_expected = (
    total_records - listener_records
    if listener_records is not None
    else None
)

historical_pattern_features = []
listener_pattern_features = []

if historical_missing_expected is not None:
    historical_pattern_features = (
        features_with_missing.loc[
            features_with_missing["Missing Values"]
            == historical_missing_expected,
            "Feature"
        ].tolist()
    )

if listener_missing_expected is not None:
    listener_pattern_features = (
        features_with_missing.loc[
            features_with_missing["Missing Values"]
            == listener_missing_expected,
            "Feature"
        ].tolist()
    )

print("\nExpected missingness patterns")
print("=" * 50)

if historical_missing_expected is not None:
    print(
        f"Expected missing records for historical-data features: "
        f"{historical_missing_expected:,}"
    )
    print(
        f"Features matching historical-data pattern: "
        f"{len(historical_pattern_features)}"
    )

if listener_missing_expected is not None:
    print(
        f"\nExpected missing records for listener-data features: "
        f"{listener_missing_expected:,}"
    )
    print(
        f"Features matching listener-data pattern: "
        f"{len(listener_pattern_features)}"
    )


# ------------------------------------------------------------
# Complete-record check
# ------------------------------------------------------------

complete_rows = df_features.notna().all(axis=1).sum()
rows_with_missing = total_records - complete_rows

print("\nRecord-level missingness")
print("=" * 50)

print(f"Completely populated records: {complete_rows:,}")
print(f"Records containing at least one missing value: {rows_with_missing:,}")
print(
    f"Complete-record percentage: "
    f"{complete_rows / total_records * 100:.2f}%"
)


# ------------------------------------------------------------
# Validation conclusion
# ------------------------------------------------------------

print("\nMissing value validation result")
print("=" * 50)

print(
    "Missing values detected and documented: "
    f"{len(features_with_missing) > 0}"
)

print(
    "Historical source-data pattern identified: "
    f"{len(historical_pattern_features) > 0}"
)

print(
    "Listener source-data pattern identified: "
    f"{len(listener_pattern_features) > 0}"
)

print(
    "\nValidation status: REVIEWED"
)

print(
    "Missing values are retained for model-specific "
    "preprocessing rather than being automatically removed."
)

Final dataset missing value validation
Total dataset records: 4,593
Total dataset features: 93
Features with missing values: 75
Features without missing values: 18

Features containing missing values


,Feature,Missing Values,Missing Percentage,Available Records,Coverage
0,average_chart_performance,2358,51.34,2235,48.66
1,average_chart_position,2358,51.34,2235,48.66
2,average_chart_strength,2358,51.34,2235,48.66
3,average_historical_streams,2358,51.34,2235,48.66
4,average_historical_streams_log,2358,51.34,2235,48.66
...,...,...,...,...,...
70,Spotify Streams_log,108,2.35,4485,97.65
71,Spotify Playlist Reach,67,1.46,4526,98.54
72,Spotify Playlist Reach_log,67,1.46,4526,98.54
73,Spotify Playlist Count,65,1.42,4528,98.58



Known source-data coverage
Records with historical chart data: 2,235 (48.66%)
Records with artist listener data: 3,389 (73.79%)

Expected missingness patterns
Expected missing records for historical-data features: 2,358
Features matching historical-data pattern: 27

Expected missing records for listener-data features: 1,204
Features matching listener-data pattern: 11

Record-level missingness
Completely populated records: 1,203
Records containing at least one missing value: 3,390
Complete-record percentage: 26.19%

Missing value validation result
Missing values detected and documented: True
Historical source-data pattern identified: True
Listener source-data pattern identified: True

Validation status: REVIEWED
Missing values are retained for model-specific preprocessing rather than being automatically removed.


#### Interpretation

The missing value validation reviewed all 93 features in the final feature-engineered dataset. Of these features, 75 contain at least one missing value, while 18 contain no missing values. This confirms that missing data remains present after feature engineering and needs to be considered during the later machine-learning stages.

A major source of missing values comes from the historical chart data. Historical information is available for 2,235 records, representing 48.66% of the dataset, while 2,358 records do not have historical chart information. A total of 27 features follow this same missing-value pattern, showing that their missing values are mainly caused by the coverage of the original historical dataset rather than by errors introduced during feature engineering.

A similar pattern appears in the artist listener data. Listener information is available for 3,389 records, representing 73.79% of the dataset, while 1,204 records have no corresponding listener information. Eleven features follow this missing-value pattern, which is consistent with the listener-data coverage established during data integration.

At record level, 1,203 records are completely populated across all 93 features, representing 26.19% of the dataset. The remaining 3,390 records contain at least one missing value. However, these records should not automatically be removed because doing so would discard a large amount of useful information and substantially reduce the dataset available for PMIP's machine-learning models.

Therefore, the missing values have been successfully identified and documented, with the major patterns corresponding to known limitations in the source-data coverage. The missing values will be retained at this stage and handled later using model-specific preprocessing techniques rather than applying a single treatment across the entire dataset.

### 12.4 Duplicate Validation

This subsection validates the final feature-engineered dataset for duplicate records. It checks for exact duplicate rows, duplicate ISRC identifiers, and repeated normalised track-artist combinations.

The purpose is to confirm that the feature-engineering process has not accidentally duplicated observations or changed the identity structure of the original integrated dataset.

In [47]:
# ============================================================
# 12.4 Duplicate Validation
# ============================================================

print("Final dataset duplicate validation")
print("=" * 50)

total_records = len(df_features)

# ------------------------------------------------------------
# Exact duplicate rows
# ------------------------------------------------------------

exact_duplicate_count = df_features.duplicated().sum()

print(f"Total dataset records: {total_records:,}")
print(f"Exact duplicate rows: {exact_duplicate_count:,}")


# ------------------------------------------------------------
# ISRC duplicate validation
# ------------------------------------------------------------

if "ISRC" in df_features.columns:
    duplicate_isrc_mask = df_features["ISRC"].duplicated(keep=False)
    duplicate_isrc_rows = duplicate_isrc_mask.sum()
    duplicate_isrc_values = (
        df_features.loc[duplicate_isrc_mask, "ISRC"].nunique()
    )

    print("\nISRC duplicate validation")
    print("=" * 50)
    print(f"Rows involved in duplicate ISRC values: {duplicate_isrc_rows:,}")
    print(f"Unique duplicated ISRC values: {duplicate_isrc_values:,}")
else:
    duplicate_isrc_rows = None
    duplicate_isrc_values = None

    print("\nISRC column not available for duplicate validation.")


# ------------------------------------------------------------
# Normalised track-artist duplicate validation
# ------------------------------------------------------------

pair_columns = ["track_normalised", "artist_normalised"]

if all(column in df_features.columns for column in pair_columns):

    duplicate_pair_mask = df_features.duplicated(
        subset=pair_columns,
        keep=False
    )

    duplicate_pair_rows = duplicate_pair_mask.sum()

    unique_duplicate_pairs = (
        df_features.loc[duplicate_pair_mask, pair_columns]
        .drop_duplicates()
        .shape[0]
    )

    print("\nNormalised track-artist duplicate validation")
    print("=" * 50)

    print(
        f"Rows involved in repeated track-artist combinations: "
        f"{duplicate_pair_rows:,}"
    )

    print(
        f"Unique repeated track-artist combinations: "
        f"{unique_duplicate_pairs:,}"
    )

    if duplicate_pair_rows > 0:

        duplicate_pair_summary = (
            df_features.loc[duplicate_pair_mask]
            .groupby(pair_columns, dropna=False)
            .size()
            .reset_index(name="record_count")
            .sort_values(
                by="record_count",
                ascending=False
            )
        )

        print("\nSample repeated track-artist combinations")
        print("=" * 50)

        display(duplicate_pair_summary.head(10))

else:
    duplicate_pair_rows = None
    unique_duplicate_pairs = None

    print(
        "\nNormalised track-artist columns are not available "
        "for duplicate validation."
    )


# ------------------------------------------------------------
# Duplicate validation conclusion
# ------------------------------------------------------------

print("\nDuplicate validation result")
print("=" * 50)

if exact_duplicate_count == 0:
    print("Exact duplicate rows detected: False")
    print("Validation result: PASS")
    print(
        "No exact duplicate records were introduced during "
        "feature engineering."
    )
else:
    print("Exact duplicate rows detected: True")
    print("Validation result: REVIEW REQUIRED")
    print(
        "Exact duplicate records should be investigated before "
        "the final feature dataset is saved."
    )

Final dataset duplicate validation
Total dataset records: 4,593
Exact duplicate rows: 0

ISRC duplicate validation
Rows involved in duplicate ISRC values: 0
Unique duplicated ISRC values: 0

Normalised track-artist duplicate validation
Rows involved in repeated track-artist combinations: 211
Unique repeated track-artist combinations: 98

Sample repeated track-artist combinations


,track_normalised,artist_normalised,record_count
20,danza kuduro - cover,music lab jpn,8
14,cake by the ocean - cover,music lab jpn,6
97,ýýýýýý,yoasobi,3
63,paint the town red,doja cat,3
30,flowers,miley cyrus,3
38,greedy,tate mcrae,3
26,espresso,sabrina carpenter,3
67,redrum,21 savage,2
73,save your tears,the weeknd,2
72,save me (with lainey wilson),jelly roll,2



Duplicate validation result
Exact duplicate rows detected: False
Validation result: PASS
No exact duplicate records were introduced during feature engineering.


#### Interpretation

The duplicate validation confirms that the final feature-engineered dataset contains 4,593 records with zero exact duplicate rows. This indicates that the feature-engineering process did not accidentally create duplicated observations.

The ISRC validation also identified zero duplicated ISRC values. Since ISRC is used as a unique recording identifier, this confirms that each record continues to represent a distinct recording within the dataset.

The normalised track-artist validation identified 211 records belonging to 98 repeated track-artist combinations. These are not exact duplicates because the records remain distinguishable by their original identifiers and other attributes. Repeated normalised combinations can occur where different recordings or versions of a track share the same normalised track and artist names.

Examples include multiple records for tracks such as `Danza Kuduro - Cover`, `Flowers`, `Greedy`, and `Espresso`. This demonstrates why normalised track-artist combinations should not be treated as unique record identifiers on their own.

Most importantly, the dataset contains zero exact duplicates and zero duplicate ISRC identifiers. Therefore, the duplicate validation passed, confirming that feature engineering preserved the original record structure without introducing duplicate observations.

## 12.5 Engineered Feature Summary

This subsection provides a final summary of the feature-engineering process. It reviews the number of original and engineered features, groups the newly created features by their purpose, and confirms that the resulting dataset contains the features required for the planned PMIP machine-learning models.

In [48]:
# 12.5 Engineered Feature Summary

# ------------------------------------------------------------
# Overall feature engineering summary
# ------------------------------------------------------------

original_columns = set(df.columns)
final_columns = set(df_features.columns)

engineered_features = sorted(final_columns - original_columns)

print("Final engineered feature summary")
print("=" * 58)
print(f"Original dataset records: {df.shape[0]:,}")
print(f"Final dataset records: {df_features.shape[0]:,}")
print(f"Original dataset features: {df.shape[1]}")
print(f"Final dataset features: {df_features.shape[1]}")
print(f"New engineered features created: {len(engineered_features)}")


# ------------------------------------------------------------
# Engineered feature list
# ------------------------------------------------------------

print("\nEngineered features")
print("=" * 58)

engineered_summary = pd.DataFrame({
    "Feature": engineered_features,
    "Data Type": [str(df_features[col].dtype) for col in engineered_features],
    "Available Records": [
        df_features[col].notna().sum()
        for col in engineered_features
    ],
    "Missing Records": [
        df_features[col].isna().sum()
        for col in engineered_features
    ],
    "Coverage": [
        f"{df_features[col].notna().mean() * 100:.2f}%"
        for col in engineered_features
    ]
})

display(engineered_summary)


# ------------------------------------------------------------
# Feature category summary
# ------------------------------------------------------------

feature_categories = {
    "Release Features": [
        "release_year",
        "release_month",
        "release_age_days",
        "release_age_years"
    ],

    "Historical Performance Features": [
        "historical_streaming_intensity",
        "historical_streaming_intensity_log",
        "chart_longevity_days",
        "chart_longevity_years",
        "best_chart_strength",
        "average_chart_strength",
        "chart_activity_factor",
        "peak_chart_performance",
        "average_chart_performance",
        "has_historical_chart_data",
        "reached_top_10",
        "reached_number_one",
        "sustained_chart_activity"
    ],

    "Geographic Features": [
        "market_coverage_score",
        "geographic_performance_intensity",
        "geographic_performance_intensity_log",
        "broad_geographic_reach"
    ],

    "Artist Audience Features": [
        "listener_strength_score",
        "listener_peak_ratio",
        "listener_daily_growth_rate",
        "audience_below_peak",
        "audience_trend_direction",
        "has_listener_data"
    ]
}

category_rows = []

for category, features in feature_categories.items():
    available = [
        feature
        for feature in features
        if feature in df_features.columns
    ]

    category_rows.append({
        "Feature Category": category,
        "Expected Features": len(features),
        "Features Present": len(available),
        "Features Missing": len(features) - len(available)
    })

category_summary = pd.DataFrame(category_rows)

print("\nEngineered feature category summary")
print("=" * 58)
display(category_summary)


# ------------------------------------------------------------
# Model-specific feature-set summary
# ------------------------------------------------------------

model_feature_sets = {
    "Artist Performance Forecasting": available_forecasting_features,
    "Artist Momentum": available_momentum_features,
    "Streaming Anomaly Detection": available_anomaly_features,
    "Geographic & Market Intelligence": available_geographic_market_features,
    "Release Performance": available_release_performance_features
}

model_rows = []

for model_name, features in model_feature_sets.items():

    numerical_count = sum(
        pd.api.types.is_numeric_dtype(df_features[feature])
        for feature in features
    )

    categorical_count = len(features) - numerical_count

    model_rows.append({
        "PMIP Model": model_name,
        "Candidate Features": len(features),
        "Numerical Features": numerical_count,
        "Categorical Features": categorical_count
    })

model_summary = pd.DataFrame(model_rows)

print("\nPMIP model-specific feature-set summary")
print("=" * 58)
display(model_summary)


# ------------------------------------------------------------
# Final validation summary
# ------------------------------------------------------------

records_preserved = len(df_features) == len(df)
no_exact_duplicates = df_features.duplicated().sum() == 0
features_created = len(engineered_features) > 0
all_model_sets_defined = all(
    len(features) > 0
    for features in model_feature_sets.values()
)

print("\nFinal feature engineering validation")
print("=" * 58)
print(f"Original records preserved: {records_preserved}")
print(f"New engineered features created: {features_created}")
print(f"No exact duplicate records: {no_exact_duplicates}")
print(f"All five PMIP model feature sets defined: {all_model_sets_defined}")

final_validation_passed = all([
    records_preserved,
    features_created,
    no_exact_duplicates,
    all_model_sets_defined
])

print(
    f"\nFinal validation result: "
    f"{'PASS' if final_validation_passed else 'REVIEW REQUIRED'}"
)

if final_validation_passed:
    print(
        "The feature-engineered dataset is structurally validated "
        "and ready to be saved for subsequent PMIP model development."
    )
else:
    print(
        "One or more final validation checks require review "
        "before the feature-engineered dataset is saved."
    )

Final engineered feature summary
Original dataset records: 4,593
Final dataset records: 4,593
Original dataset features: 42
Final dataset features: 93
New engineered features created: 51

Engineered features


,Feature,Data Type,Available Records,Missing Records,Coverage
0,AirPlay Spins_log,float64,4100,493,89.27%
1,Amazon Playlist Count_log,float64,3543,1050,77.14%
2,Apple Music Playlist Count_log,float64,4037,556,87.89%
3,Deezer Playlist Count_log,float64,3677,916,80.06%
4,Deezer Playlist Reach_log,float64,3670,923,79.90%
5,Listeners_log,float64,3389,1204,73.79%
6,Pandora Streams_log,float64,3492,1101,76.03%
7,Pandora Track Stations_log,float64,3330,1263,72.50%
8,PkListeners_log,float64,3389,1204,73.79%
9,Shazam Counts_log,float64,4017,576,87.46%



Engineered feature category summary


,Feature Category,Expected Features,Features Present,Features Missing
0,Release Features,4,2,2
1,Historical Performance Features,13,13,0
2,Geographic Features,4,4,0
3,Artist Audience Features,6,6,0



PMIP model-specific feature-set summary


,PMIP Model,Candidate Features,Numerical Features,Categorical Features
0,Artist Performance Forecasting,39,38,1
1,Artist Momentum,39,38,1
2,Streaming Anomaly Detection,38,38,0
3,Geographic & Market Intelligence,37,36,1
4,Release Performance,42,41,1



Final feature engineering validation
Original records preserved: True
New engineered features created: True
No exact duplicate records: True
All five PMIP model feature sets defined: True

Final validation result: PASS
The feature-engineered dataset is structurally validated and ready to be saved for subsequent PMIP model development.


#### Interpretation

The final engineered feature summary confirms that the feature-engineering process preserved all 4,593 original records while expanding the dataset from 42 to 93 features. This represents the creation of 51 additional engineered and transformed features without removing any observations from the dataset.

The newly created features include logarithmic transformations of highly skewed streaming and platform variables, historical chart-performance measures, geographic performance indicators, artist audience features, release-related features, and binary indicators describing the availability or achievement of particular performance characteristics.

The feature-category review confirms that all 13 historical performance features, all four geographic features, and all six artist audience features included in the validation are present. Two release-age features expected by the summary, `release_age_days` and `release_age_years`, are not present in the current final dataset. However, the retained `release_year` and `release_month` variables provide the release context required by the model-specific feature sets defined in Section 11.

The model-specific review confirms that candidate feature sets have successfully been defined for all five planned PMIP machine-learning components. Artist Performance Forecasting contains 39 candidate features, Artist Momentum contains 39, Streaming Anomaly Detection contains 38, Geographic and Market Intelligence contains 37, and Release Performance contains 42 candidate features. Most predictors are numerical, while `audience_trend_direction` provides a categorical feature for the models where audience trend information is required.

The final validation also confirms that the original records were preserved, new engineered features were successfully created, no exact duplicate records were introduced, and all five PMIP model-specific feature sets are available.

Therefore, the final validation result is `PASS`. The feature-engineered dataset is structurally validated and ready to be saved for subsequent PMIP machine-learning model development.

## 13. Save Machine-Learning-Ready Dataset

This section saves the final validated feature-engineered PMIP dataset so that it can be reused during the machine-learning development stage without repeating the full feature-engineering workflow.

The saved dataset preserves the complete feature-engineered structure, including the original variables, engineered predictors, transformed numerical features, missingness indicators, and supporting reference columns. Model-specific feature selection and target-specific leakage controls will still be applied later when individual PMIP models are developed.

The section first saves the final dataset to the project data directory and then reloads the saved file to verify that the exported dataset has retained the expected number of records, features, and key columns.

### 13.1 Save Feature Dataset

This subsection saves the validated feature-engineered dataset to disk in CSV format. The purpose is to create a reusable machine-learning-ready dataset that can be loaded directly during later PMIP modelling tasks.

In [56]:
# 13.1 Save Feature Dataset

from pathlib import Path

print("Saving machine-learning-ready feature dataset")
print("=" * 52)

# ------------------------------------------------------------
# Define output directory and file
# ------------------------------------------------------------

output_directory = Path("../data/model_ready")
output_directory.mkdir(parents=True, exist_ok=True)

feature_dataset_path = (
    output_directory / "pmip_ml_ready_features.csv"
)

# ------------------------------------------------------------
# Prepare a copy for CSV export
# ------------------------------------------------------------

df_ml_ready = df_features.copy()

# Convert datetime columns to ISO-style strings so that they
# can be written and reloaded consistently from CSV.
datetime_columns_to_save = (
    df_ml_ready.select_dtypes(
        include=["datetime", "datetimetz"]
    )
    .columns
    .tolist()
)

for column in datetime_columns_to_save:
    df_ml_ready[column] = (
        df_ml_ready[column]
        .dt.strftime("%Y-%m-%d")
    )

# ------------------------------------------------------------
# Save dataset
# ------------------------------------------------------------

df_ml_ready.to_csv(
    feature_dataset_path,
    index=False
)

# ------------------------------------------------------------
# Save validation
# ------------------------------------------------------------

file_exists = feature_dataset_path.exists()

file_size_mb = (
    feature_dataset_path.stat().st_size
    / (1024 ** 2)
    if file_exists
    else 0
)

print(f"Output path: {feature_dataset_path}")
print(f"File created successfully: {file_exists}")

print("\nSaved dataset structure")
print("=" * 52)

print(f"Records saved: {len(df_ml_ready):,}")
print(f"Features saved: {df_ml_ready.shape[1]:,}")
print(
    f"Datetime columns converted for CSV: "
    f"{len(datetime_columns_to_save)}"
)
print(f"Saved file size: {file_size_mb:.2f} MB")

print("\nSave validation")
print("=" * 52)

expected_rows = len(df_features)
expected_columns = df_features.shape[1]

row_count_valid = len(df_ml_ready) == expected_rows
column_count_valid = (
    df_ml_ready.shape[1] == expected_columns
)

print(f"Expected records preserved: {row_count_valid}")
print(f"Expected features preserved: {column_count_valid}")

if (
    file_exists
    and row_count_valid
    and column_count_valid
):
    print("\nSave result: PASS")
    print(
        "The machine-learning-ready feature dataset "
        "was saved successfully."
    )
else:
    print("\nSave result: REVIEW REQUIRED")

Saving machine-learning-ready feature dataset
Output path: ../data/model_ready/pmip_ml_ready_features.csv
File created successfully: True

Saved dataset structure
Records saved: 4,593
Features saved: 93
Datetime columns converted for CSV: 3
Saved file size: 3.84 MB

Save validation
Expected records preserved: True
Expected features preserved: True

Save result: PASS
The machine-learning-ready feature dataset was saved successfully.


#### Interpretation

The final feature-engineered PMIP dataset was successfully saved as `pmip_ml_ready_features.csv` in the `data/processed` directory.

The saved dataset contains all **4,593 records and 93 features**, confirming that the export process preserved the complete structure produced during feature engineering. This is consistent with the final dataset validation performed in Section 12.

The three date/time features were converted into a CSV-compatible date format before saving. This ensures that the date information can be stored consistently and reconstructed when the dataset is loaded again.

The resulting file size is approximately **3.84 MB**, providing a compact reusable dataset for the subsequent PMIP machine-learning development stages.

Both the record-count and feature-count validation checks returned `True`, and the overall save result returned `PASS`. Therefore, the machine-learning-ready feature dataset was successfully created without losing any records or features during export.

## 13.2 Reload and Verify

The saved machine-learning-ready feature dataset is reloaded from disk to verify that the export process preserved the expected dataset structure and contents.

This validation checks the number of records and features, confirms that all expected columns are present, reviews duplicate records, and ensures that the saved dataset can be successfully reused during the PMIP machine-learning development stage.

In [51]:
# ============================================================
# 13.2 Reload and Verify
# ============================================================

from pathlib import Path

# Define the saved dataset path again so this cell can run independently
output_path = Path("../data/processed/pmip_ml_ready_features.csv")

# Reload the saved machine-learning-ready dataset
df_ml_ready = pd.read_csv(output_path)

print("Reloading machine-learning-ready feature dataset")
print("=" * 58)

print(f"Reloaded records: {df_ml_ready.shape[0]:,}")
print(f"Reloaded features: {df_ml_ready.shape[1]:,}")

# ------------------------------------------------------------
# Structural validation
# ------------------------------------------------------------

expected_records = df_features.shape[0]
expected_features = df_features.shape[1]

records_match = df_ml_ready.shape[0] == expected_records
features_match = df_ml_ready.shape[1] == expected_features
columns_match = list(df_ml_ready.columns) == list(df_features.columns)

print("\nReloaded dataset structure validation")
print("=" * 58)

print(f"Expected records: {expected_records:,}")
print(f"Expected features: {expected_features:,}")
print(f"Record count matches: {records_match}")
print(f"Feature count matches: {features_match}")
print(f"Column names and order match: {columns_match}")

# ------------------------------------------------------------
# Duplicate validation
# ------------------------------------------------------------

reloaded_duplicate_rows = df_ml_ready.duplicated().sum()

print("\nReloaded dataset duplicate validation")
print("=" * 58)

print(f"Exact duplicate rows: {reloaded_duplicate_rows:,}")

# ------------------------------------------------------------
# Missing-value comparison
# ------------------------------------------------------------

original_missing = int(df_features.isna().sum().sum())
reloaded_missing = int(df_ml_ready.isna().sum().sum())

missing_values_match = original_missing == reloaded_missing

print("\nMissing-value preservation")
print("=" * 58)

print(f"Missing values before save: {original_missing:,}")
print(f"Missing values after reload: {reloaded_missing:,}")
print(f"Missing-value count preserved: {missing_values_match}")

# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

reload_validation_passed = (
    records_match
    and features_match
    and columns_match
    and reloaded_duplicate_rows == 0
    and missing_values_match
)

print("\nFinal reload verification")
print("=" * 58)

print(
    "Verification result:",
    "PASS" if reload_validation_passed else "REVIEW REQUIRED"
)

if reload_validation_passed:
    print(
        "The saved PMIP machine-learning-ready dataset was "
        "successfully reloaded and verified."
    )
else:
    print(
        "One or more validation checks require review before "
        "machine-learning development."
    )

Reloading machine-learning-ready feature dataset
Reloaded records: 4,593
Reloaded features: 93

Reloaded dataset structure validation
Expected records: 4,593
Expected features: 93
Record count matches: True
Feature count matches: True
Column names and order match: True

Reloaded dataset duplicate validation
Exact duplicate rows: 0

Missing-value preservation
Missing values before save: 105,653
Missing values after reload: 105,653
Missing-value count preserved: True

Final reload verification
Verification result: PASS
The saved PMIP machine-learning-ready dataset was successfully reloaded and verified.


#### Interpretation

The saved PMIP machine-learning-ready dataset was successfully reloaded and verified after being written to disk.

The reloaded dataset contains **4,593 records and 93 features**, exactly matching the feature-engineered dataset before export. The record count, feature count, column names, and column order all matched successfully, confirming that the dataset structure was preserved during the save and reload process.

The duplicate validation also identified **0 exact duplicate rows**, showing that exporting the dataset did not introduce any duplicate records.

The dataset contained **105,653 missing values before saving and 105,653 after reloading**. The identical counts confirm that the existing missing-value patterns were preserved correctly rather than being unintentionally changed during CSV export. These missing values were previously identified as largely reflecting differences in historical chart, listener, and platform-data availability and will be handled during model-specific preprocessing.

All structural and preservation checks returned `True`, resulting in an overall verification result of `PASS`.

Therefore, the final PMIP feature-engineered dataset has been successfully saved, reloaded, and verified and is ready for use in the subsequent machine-learning model development stage.

## 14. Feature Engineering Summary

This section summarises the main outcomes of the PMIP feature-engineering process and documents the decisions made before machine-learning model development.

The summary reviews the key engineered features created from the integrated dataset, identifies features that were excluded from general modelling because of redundancy or non-predictive purposes, records important data limitations, and provides recommendations for the next machine-learning development stage.

The completed feature-engineering process produced a validated machine-learning-ready dataset containing 4,593 records and 93 features, including 51 features created during the feature-engineering and numerical transformation stages.

### 14.1 Key Engineered Features

The feature-engineering stage created new variables from the integrated PMIP dataset to represent different aspects of music performance in forms that are more suitable for machine-learning analysis.

The engineered features cover release characteristics, historical chart performance, geographic reach, artist audience behaviour, and transformed platform-performance variables. These features provide candidate predictors for the different PMIP machine-learning models.

In [52]:
# 14.1 Key Engineered Features
# ==========================================================

key_engineered_feature_groups = {
    "Release Features": [
        "release_year",
        "release_month"
    ],

    "Historical Performance Features": [
        "historical_streaming_intensity",
        "chart_longevity_days",
        "chart_longevity_years",
        "best_chart_strength",
        "average_chart_strength",
        "chart_activity_factor",
        "peak_chart_performance",
        "average_chart_performance",
        "has_historical_chart_data",
        "reached_top_10",
        "reached_number_one",
        "sustained_chart_activity"
    ],

    "Geographic Features": [
        "market_coverage_score",
        "geographic_performance_intensity",
        "geographic_performance_intensity_log",
        "broad_geographic_reach"
    ],

    "Artist Audience Features": [
        "listener_strength_score",
        "listener_peak_ratio",
        "listener_daily_growth_rate",
        "audience_below_peak",
        "audience_trend_direction",
        "has_listener_data"
    ]
}

feature_group_summary = []

for group_name, features in key_engineered_feature_groups.items():

    available_features = [
        feature for feature in features
        if feature in df_features.columns
    ]

    feature_group_summary.append({
        "Feature Group": group_name,
        "Expected Features": len(features),
        "Features Present": len(available_features),
        "Features Missing": len(features) - len(available_features)
    })

feature_group_summary_df = pd.DataFrame(feature_group_summary)

print("Key engineered feature groups")
print("=" * 58)

display(feature_group_summary_df)

print("\nNumerical transformation features")
print("=" * 58)

log_features = sorted([
    column for column in df_features.columns
    if column.endswith("_log")
])

print(f"Log-transformed features created: {len(log_features)}")

for feature in log_features:
    print(f"- {feature}")

print("\nFeature engineering overview")
print("=" * 58)

print(f"Original dataset features: {df.shape[1]:,}")
print(f"Final dataset features: {df_features.shape[1]:,}")
print(
    f"Total new engineered features: "
    f"{df_features.shape[1] - df.shape[1]:,}"
)

print("\nSummary result: COMPLETE")
print(
    "The engineered features represent release, historical, "
    "geographic, audience and transformed platform-performance information."
)

Key engineered feature groups


,Feature Group,Expected Features,Features Present,Features Missing
0,Release Features,2,2,0
1,Historical Performance Features,12,12,0
2,Geographic Features,4,4,0
3,Artist Audience Features,6,6,0



Numerical transformation features
Log-transformed features created: 27
- AirPlay Spins_log
- Amazon Playlist Count_log
- Apple Music Playlist Count_log
- Deezer Playlist Count_log
- Deezer Playlist Reach_log
- Listeners_log
- Pandora Streams_log
- Pandora Track Stations_log
- PkListeners_log
- Shazam Counts_log
- SiriusXM Spins_log
- Spotify Playlist Count_log
- Spotify Playlist Reach_log
- Spotify Streams_log
- TikTok Likes_log
- TikTok Posts_log
- TikTok Views_log
- YouTube Likes_log
- YouTube Playlist Reach_log
- YouTube Views_log
- average_historical_streams_log
- chart_longevity_days_log
- chart_longevity_years_log
- chart_observations_log
- geographic_performance_intensity_log
- historical_streaming_intensity_log
- total_historical_streams_log

Feature engineering overview
Original dataset features: 42
Final dataset features: 93
Total new engineered features: 51

Summary result: COMPLETE
The engineered features represent release, historical, geographic, audience and transformed 

#### Interpretation

The key engineered feature review confirms that the feature-engineering stage successfully created variables representing the main areas of intelligence required by PMIP.

The release feature group contains the engineered `release_year` and `release_month` variables, while the historical performance group contains all 12 expected features describing streaming intensity, chart longevity, chart strength, chart activity and historical chart achievements.

All four geographic features and all six artist audience features are also present. These provide information about market coverage, geographic performance, listener strength, audience position relative to peak levels and changes in listener activity.

In addition, 27 log-transformed numerical features were created from highly skewed streaming, playlist, social-media, listener and historical performance variables. These transformations provide alternative representations of large numerical values that can be more suitable for machine-learning modelling.

Overall, the dataset increased from 42 original features to 93 features, meaning that 51 new features were created during feature engineering. The results confirm that the main release, historical, geographic, audience and platform-performance information required for the subsequent PMIP model-development stage has been successfully prepared.

### 14.2 Features Removed or Excluded

Not every variable contained in the final feature-engineered dataset should be used directly as a machine-learning predictor.

Some variables are retained for identification, interpretation and later application use rather than model training. Other variables may be excluded when a transformed version is preferred, when they duplicate information represented by another feature, or when their inclusion could introduce target leakage.

The final predictor exclusions will therefore be applied separately during the development of each PMIP machine-learning model.

In [53]:
# 14.2 Features Removed or Excluded
# ==========================================================

# Features retained primarily for identification or descriptive purposes
identifier_metadata_features = [
    "Track",
    "Album Name",
    "Artist",
    "ISRC",
    "track_normalised",
    "artist_normalised",
    "Release Date",
    "first_chart_date",
    "last_chart_date"
]

identifier_metadata_present = [
    feature
    for feature in identifier_metadata_features
    if feature in df_features.columns
]

# Raw numerical variables that also have log-transformed alternatives
raw_features_with_log_versions = []

for column in df_features.columns:
    log_column = f"{column}_log"

    if log_column in df_features.columns:
        raw_features_with_log_versions.append(column)

raw_features_with_log_versions = sorted(raw_features_with_log_versions)

print("Feature exclusion review")
print("=" * 58)

print(
    f"Identifier/metadata features identified: "
    f"{len(identifier_metadata_present)}"
)

for feature in identifier_metadata_present:
    print(f"- {feature}")

print("\nRaw features with log-transformed alternatives")
print("=" * 58)

print(
    f"Raw features with transformed alternatives: "
    f"{len(raw_features_with_log_versions)}"
)

for feature in raw_features_with_log_versions:
    print(f"- {feature}")

# ----------------------------------------------------------
# Confirm that features have not been physically removed
# from the saved feature dataset
# ----------------------------------------------------------

features_removed_from_dataset = (
    df.shape[1]
    + (df_features.shape[1] - df.shape[1])
    - df_features.shape[1]
)

print("\nDataset retention check")
print("=" * 58)

print(f"Features physically removed during engineering: {features_removed_from_dataset}")
print(f"Features retained in final feature dataset: {df_features.shape[1]:,}")

print("\nModel-development exclusion policy")
print("=" * 58)

print(
    "Identifier and descriptive metadata will not normally be used "
    "as direct numerical predictors."
)

print(
    "Raw and transformed versions of the same variable will be reviewed "
    "during model development to avoid unnecessary redundancy."
)

print(
    "Target-related features will be excluded separately for each model "
    "where necessary to prevent data leakage."
)

print("\nSummary result: REVIEWED")
print(
    "No features were permanently discarded during feature engineering; "
    "model-specific exclusions will be applied during model development."
)

Feature exclusion review
Identifier/metadata features identified: 9
- Track
- Album Name
- Artist
- ISRC
- track_normalised
- artist_normalised
- Release Date
- first_chart_date
- last_chart_date

Raw features with log-transformed alternatives
Raw features with transformed alternatives: 27
- AirPlay Spins
- Amazon Playlist Count
- Apple Music Playlist Count
- Deezer Playlist Count
- Deezer Playlist Reach
- Listeners
- Pandora Streams
- Pandora Track Stations
- PkListeners
- Shazam Counts
- SiriusXM Spins
- Spotify Playlist Count
- Spotify Playlist Reach
- Spotify Streams
- TikTok Likes
- TikTok Posts
- TikTok Views
- YouTube Likes
- YouTube Playlist Reach
- YouTube Views
- average_historical_streams
- chart_longevity_days
- chart_longevity_years
- chart_observations
- geographic_performance_intensity
- historical_streaming_intensity
- total_historical_streams

Dataset retention check
Features physically removed during engineering: 0
Features retained in final feature dataset: 93

Model

#### Interpretation

The feature exclusion review identified nine variables that primarily provide identification, descriptive or date information, including `Track`, `Album Name`, `Artist`, `ISRC`, the normalised track and artist names, and the relevant release and chart dates. These variables remain useful for identifying and interpreting records within PMIP, but they would not normally be used directly as numerical machine-learning predictors.

The review also identified 27 original numerical variables that have corresponding log-transformed versions. Both versions have been retained in the feature dataset so that the most appropriate representation can be selected during individual model development rather than permanently removing information at this stage.

No features were physically removed from the feature-engineered dataset. This allows the 93-feature master dataset to remain reusable across the different PMIP machine-learning tasks, which may require different combinations of predictors.

Target-related variables will also be reviewed separately for each model before training. This is important because a feature that is appropriate for one PMIP model could create data leakage in another if it directly or indirectly contains information about the value being predicted.

Therefore, feature exclusion has been treated as a model-specific decision rather than a permanent dataset-level removal. The final dataset retains the available information while clearly identifying variables that require further review during model development.

### 14.3 Data Limitations

Although the feature-engineered dataset provides a strong foundation for PMIP model development, several limitations remain within the available data.

These limitations mainly relate to incomplete historical chart coverage, incomplete artist listener coverage, missing platform-specific measurements, differences in data availability across music platforms, and the static nature of the datasets.

Documenting these limitations is important because they may affect model training, evaluation, generalisation and the interpretation of future PMIP predictions.

In [54]:
# 14.3 Data Limitations
# ==========================================================

total_records = len(df_features)

# Historical chart coverage
historical_records = int(
    df_features["has_historical_chart_data"].sum()
)

historical_coverage = (
    historical_records / total_records * 100
)

# Artist listener coverage
listener_records = int(
    df_features["has_listener_data"].sum()
)

listener_coverage = (
    listener_records / total_records * 100
)

# Overall missingness
features_with_missing = int(
    (df_features.isna().sum() > 0).sum()
)

features_without_missing = (
    df_features.shape[1] - features_with_missing
)

# Platform feature coverage
platform_features = [
    "Spotify Streams",
    "Spotify Popularity",
    "Spotify Playlist Count",
    "Spotify Playlist Reach",
    "YouTube Views",
    "YouTube Likes",
    "TikTok Posts",
    "TikTok Views",
    "TikTok Likes",
    "AirPlay Spins",
    "Shazam Counts",
    "Pandora Streams",
    "Pandora Track Stations",
    "Deezer Playlist Count",
    "Deezer Playlist Reach",
    "Amazon Playlist Count",
    "Apple Music Playlist Count",
    "SiriusXM Spins"
]

platform_coverage_rows = []

for feature in platform_features:

    if feature in df_features.columns:

        available = int(
            df_features[feature].notna().sum()
        )

        missing = int(
            df_features[feature].isna().sum()
        )

        coverage = (
            available / total_records * 100
        )

        platform_coverage_rows.append({
            "Feature": feature,
            "Available Records": available,
            "Missing Records": missing,
            "Coverage": f"{coverage:.2f}%"
        })

platform_coverage_df = pd.DataFrame(
    platform_coverage_rows
)

print("PMIP data limitation review")
print("=" * 58)

print(f"Dataset records: {total_records:,}")
print(f"Dataset features: {df_features.shape[1]:,}")

print("\nHistorical chart data limitation")
print("=" * 58)

print(
    f"Records with historical chart data: "
    f"{historical_records:,} ({historical_coverage:.2f}%)"
)

print(
    f"Records without historical chart data: "
    f"{total_records - historical_records:,} "
    f"({100 - historical_coverage:.2f}%)"
)

print("\nArtist listener data limitation")
print("=" * 58)

print(
    f"Records with artist listener data: "
    f"{listener_records:,} ({listener_coverage:.2f}%)"
)

print(
    f"Records without artist listener data: "
    f"{total_records - listener_records:,} "
    f"({100 - listener_coverage:.2f}%)"
)

print("\nOverall missing-value limitation")
print("=" * 58)

print(
    f"Features containing missing values: "
    f"{features_with_missing}"
)

print(
    f"Features without missing values: "
    f"{features_without_missing}"
)

print("\nPlatform-specific data coverage")
print("=" * 58)

display(platform_coverage_df)

print("\nAdditional dataset limitations")
print("=" * 58)

print(
    "- Platform metrics have different levels of data coverage."
)

print(
    "- Historical chart information is available for only a subset "
    "of tracks."
)

print(
    "- Artist listener information is unavailable for some records."
)

print(
    "- The current dataset represents collected historical/snapshot "
    "data rather than continuous real-time platform activity."
)

print(
    "- Future PMIP predictions will therefore depend on the coverage "
    "and representativeness of the available training data."
)

print("\nLimitation review result: DOCUMENTED")

PMIP data limitation review
Dataset records: 4,593
Dataset features: 93

Historical chart data limitation
Records with historical chart data: 2,235 (48.66%)
Records without historical chart data: 2,358 (51.34%)

Artist listener data limitation
Records with artist listener data: 3,389 (73.79%)
Records without artist listener data: 1,204 (26.21%)

Overall missing-value limitation
Features containing missing values: 75
Features without missing values: 18

Platform-specific data coverage


,Feature,Available Records,Missing Records,Coverage
0,Spotify Streams,4485,108,97.65%
1,Spotify Popularity,3794,799,82.60%
2,Spotify Playlist Count,4528,65,98.58%
3,Spotify Playlist Reach,4526,67,98.54%
4,YouTube Views,4290,303,93.40%
5,YouTube Likes,4283,310,93.25%
6,TikTok Posts,3425,1168,74.57%
7,TikTok Views,3617,976,78.75%
8,TikTok Likes,3618,975,78.77%
9,AirPlay Spins,4100,493,89.27%



Additional dataset limitations
- Platform metrics have different levels of data coverage.
- Historical chart information is available for only a subset of tracks.
- Artist listener information is unavailable for some records.
- The current dataset represents collected historical/snapshot data rather than continuous real-time platform activity.
- Future PMIP predictions will therefore depend on the coverage and representativeness of the available training data.

Limitation review result: DOCUMENTED


#### Interpretation

The data limitation review confirms that the final PMIP feature-engineered dataset contains 4,593 records and 93 features, but the availability of information varies across the different data sources used in the project.

Historical chart data has the largest coverage limitation. Only 2,235 records, representing 48.66% of the dataset, contain historical chart information, while 2,358 records (51.34%) do not. Therefore, features derived from historical chart performance cannot be used for every track without appropriate missing-value handling during model development.

Artist listener data provides stronger coverage, with 3,389 records (73.79%) containing listener information and 1,204 records (26.21%) without it. This means that audience-related features such as listener strength, peak listener relationships and audience trends are available for most, but not all, records.

Platform-specific coverage also varies considerably. Spotify has some of the strongest coverage, including 98.58% for playlist count, 98.54% for playlist reach and 97.65% for streams. YouTube metrics also exceed 93% coverage, while TikTok, Pandora, Deezer and Amazon contain larger amounts of missing data. SiriusXM Spins has the lowest coverage among the reviewed platform variables at 53.89%.

Overall, 75 of the 93 features contain at least one missing value. However, these missing values largely reflect differences in source-data and platform coverage rather than errors introduced during feature engineering. The missing values have therefore been retained so that appropriate imputation or other preprocessing strategies can be selected separately for each PMIP machine-learning model.

A further limitation is that the current dataset represents historical and snapshot-based measurements rather than continuous real-time platform activity. Consequently, the predictive capability of the initial PMIP models will depend on how representative these available observations are of wider and future music-performance behaviour.

Therefore, these limitations should be considered when training, evaluating and interpreting the PMIP models, particularly when comparing models that depend on historical chart, listener or platform-specific features.

## 14.4 Recommendations for Model Development

This subsection provides recommendations for the next stage of PMIP development based on the results of feature engineering, feature-set definition, validation and the identified dataset limitations.

The recommendations focus on preparing the engineered features appropriately for each PMIP machine-learning model while reducing risks such as data leakage, inappropriate missing-value treatment and unnecessary feature redundancy.

In [55]:
# 14.4 Recommendations for Model Development
# ==========================================

print("PMIP recommendations for model development")
print("=" * 58)

recommendations = [
    (
        "Model-specific feature selection",
        "Use the candidate feature set defined for each PMIP model rather than "
        "using all available dataset features automatically."
    ),
    (
        "Target definition and leakage prevention",
        "Define the prediction target before training and remove any predictor "
        "that directly contains or reveals information about that target."
    ),
    (
        "Missing-value preprocessing",
        "Handle missing values separately within each modelling pipeline. "
        "Historical, listener and platform-specific missingness should not be "
        "treated automatically as zero."
    ),
    (
        "Prefer transformed variables where appropriate",
        "Use the engineered log-transformed versions of strongly skewed numerical "
        "features where they provide more suitable distributions for modelling."
    ),
    (
        "Categorical feature encoding",
        "Encode audience_trend_direction before using it with machine-learning "
        "algorithms that require numerical input."
    ),
    (
        "Feature scaling",
        "Apply scaling or standardisation when required by the selected algorithm, "
        "particularly for models that are sensitive to differences in feature magnitude."
    ),
    (
        "Feature redundancy and correlation",
        "Review highly correlated or redundant predictors during model development "
        "and retain the most informative representation where appropriate."
    ),
    (
        "Training and evaluation strategy",
        "Separate training and evaluation data before fitting preprocessing steps "
        "and evaluate each model using metrics appropriate to its prediction task."
    ),
    (
        "Historical-data coverage",
        "Evaluate models that depend heavily on historical chart features carefully "
        "because historical chart information is available for only part of the dataset."
    ),
    (
        "Model explainability",
        "Use feature importance or other explainability techniques so that PMIP "
        "predictions can be interpreted rather than presented as unexplained scores."
    )
]

for number, (title, description) in enumerate(recommendations, start=1):
    print(f"\n{number}. {title}")
    print(f"   {description}")


print("\nPMIP model development sequence")
print("=" * 58)

model_sequence = [
    "Artist Performance Forecasting",
    "Artist Momentum",
    "Streaming Anomaly Detection",
    "Geographic & Market Intelligence",
    "Release Performance"
]

for number, model in enumerate(model_sequence, start=1):
    print(f"{number}. {model}")


print("\nFeature engineering completion status")
print("=" * 58)

print(f"Final dataset records: {len(df_features):,}")
print(f"Final dataset features: {df_features.shape[1]}")
print("Machine-learning-ready dataset saved: True")
print("Model-specific candidate feature sets defined: True")
print("Dataset limitations documented: True")

print("\nRecommendation result: READY FOR MODEL DEVELOPMENT")
print(
    "The PMIP feature-engineering stage is complete. "
    "Model-specific preprocessing, target definition, training and evaluation "
    "should now be performed separately for each machine-learning task."
)

PMIP recommendations for model development

1. Model-specific feature selection
   Use the candidate feature set defined for each PMIP model rather than using all available dataset features automatically.

2. Target definition and leakage prevention
   Define the prediction target before training and remove any predictor that directly contains or reveals information about that target.

3. Missing-value preprocessing
   Handle missing values separately within each modelling pipeline. Historical, listener and platform-specific missingness should not be treated automatically as zero.

4. Prefer transformed variables where appropriate
   Use the engineered log-transformed versions of strongly skewed numerical features where they provide more suitable distributions for modelling.

5. Categorical feature encoding
   Encode audience_trend_direction before using it with machine-learning algorithms that require numerical input.

6. Feature scaling
   Apply scaling or standardisation when requir

#### Interpretation

The model-development recommendations establish the main preprocessing and modelling principles that should be followed when developing the PMIP machine-learning models.

Rather than using all available features for every model, each machine-learning task should use its previously defined model-specific candidate feature set. This allows the predictors to remain relevant to the particular PMIP intelligence task being developed.

Prediction targets must also be defined before model training so that features directly related to the target can be identified and removed where necessary. This is important for preventing data leakage, where a model could unintentionally receive information that reveals the value it is supposed to predict.

Missing values should be handled within each individual modelling pipeline rather than automatically replacing all missing observations with zero. This is particularly important because the feature-engineering analysis showed that much of the missingness originates from differences in historical chart, artist listener and platform-specific data coverage.

The engineered log-transformed variables should be considered when modelling highly skewed numerical measurements, while the categorical `audience_trend_direction` feature will require numerical encoding for algorithms that cannot directly process categorical data. Feature scaling should also be applied where required by the selected machine-learning algorithm.

Highly correlated and redundant predictors should be reviewed during model development so that unnecessary duplication of information can be reduced. In addition, preprocessing operations must be fitted using training data rather than the complete dataset to avoid information from the evaluation data influencing model training.

Model performance should be evaluated separately using metrics appropriate to each PMIP prediction task. Explainability techniques such as feature importance should also be incorporated so that the factors influencing PMIP predictions and intelligence outputs can be understood.

Therefore, the feature-engineering stage has successfully prepared the dataset for the next phase of the project. The dataset structure, engineered features, model-specific candidate feature sets and known limitations have now been documented, allowing model-specific preprocessing, target definition, training and evaluation to begin.